# EXT 평가셋 — 사람이 표시한 정직한 잣대

**왜 필요한가.** 지금 라벨로는 무엇을 만들어도 채점이 안 된다 —
`0731 §4.13` conf≥0.10 검출의 **46.4%가 라벨 미대응** · `0731 §9` 육안으로 라벨 누락 확정 ·
`0803 §11` GT가 결함 전체가 아니라 **조각**(면적비 p50 112배) · `0803 §12` 모델 레버 7개가 전부 0.12~0.22 수렴.
= **깨진 자로 8번 측정했다.** 새 방법이 나아졌는지 말하려면 자부터 새로 만들어야 한다.

---

## 잣대 (실행 전에 못 박는다)

| | |
|---|---|
| 표시 방식 | **점 클릭** — 결함 중심에 점 하나. 박스 아님 |
| 적중 정의 | **점이 예측 박스 안에 들어가면 적중** |
| 주지표 | **점 recall** (사람이 본 결함 중 몇 %를 덮었나) |
| 부지표 | 장당 박스 수 · 유형별 recall · 무결함 프레임 오탐 |
| 금지 | **IoU 안 쓴다** |

**왜 점인가.** IoU는 "예측이 GT와 크기까지 비슷한가"를 묻는다. 결함 전체를 정확히 잡을수록
면적이 커져 IoU가 **떨어진다**(`0803 §11`). 점은 조각이 될 수 없고 크기 논쟁도 없다.
표시 속도도 박스의 3~5배라 150장이 한 시간 안에 끝난다.

⚠️ 점 recall은 **`0.217`과 같은 잣대가 아니다.** 그 숫자와 나란히 놓지 말 것.
새 자에서는 기존 YOLO 챔피언도 같이 채점해서(§C) **같은 시험지**로 비교한다.

---

```
§A 표본 추출 — zip에서 150장만 뽑아 Drive에 둔다
§B 클릭 라벨링 UI (중단·재개 가능, 팀 분담 가능)
§C 채점 — 예측 dict를 넣으면 점 recall
```

---

## 🔴 Colab 세션 격리 — 이 노트북은 **자립**한다

Colab은 노트북마다 VM이 따로라 다른 노트북이 `/content`에 푼 것은 **여기서 안 보인다.**
그래서 §A가 크롭본 zip에서 **150장만 뽑아 Drive에 복사**한다(~10MB). 8GB를 두 번 풀지 않는다.

```
Drive: kt_out/ext_evalset/
    images/*.jpg          ← 150장. 어느 세션에서도 같은 경로
    sample_seed42_n150.json
    labels_{WORKER}.json  ← 팀원별. §C가 전부 합친다
```

**모든 산출물이 Drive에 있으므로 §B·§C·unsup 노트북이 각자 다른 세션에서 돌아도 된다.**
§C는 Drive만 읽으므로 **unsup 세션에 그대로 붙여넣어** 돌리는 게 편하다(거기 `detect()`가 이미 있다).


In [ ]:
# == §A 표본 추출 — zip에서 150장만 뽑아 Drive에 ==
# 🔴 Colab 세션 격리 대응: 다른 노트북이 /content에 푼 크롭본은 여기서 안 보인다.
#    그래서 zip **namelist만** 읽어 표본을 정하고, 그 150장만 꺼내 Drive에 둔다(~10MB).
#
# 셀당 최대 2장인 이유: 한 셀의 270프레임은 서로 닮아 **상관 표본**이 된다(유효 표본 수가 준다).
import json, random, re, os, glob, zipfile
from pathlib import Path
from collections import defaultdict

N_SAMPLE = 150               # 목표 장수. 150장이면 결함 수백 개
PER_CELL = 2
SPLIT    = 'val'             # ★val 고정. test는 최종 1회용으로 남긴다(선택 편향 차단)
SEED     = 42
WORKER   = 'me'              # ★팀 분담: 각자 다른 이름
SHARD    = (0, 1)            # ★(내 몫, 전체 인원). 3명이면 (0,3) (1,3) (2,3)

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass

# 🔴 `kt_out`이 공유 폴더로 옮겨져 **읽기 전용**일 수 있다.
#    mkdir이 `OSError: [Errno 30] Read-only file system`으로 죽는다.
#    → 쓰기 가능한 곳을 실제로 **써 보고** 고른다. 추측하지 않는다.
CANDS = [Path('/content/drive/MyDrive/kt_out/ext_evalset'),   # 기존 규약(쓰기 되면 최선)
         Path('/content/drive/MyDrive/ext_evalset'),          # 내 드라이브 — 항상 쓰기 가능
         Path('/content/ext_evalset')]                        # 최후: 세션과 함께 사라짐

def evalset_dir():
    for c in CANDS:                                  # ① 이미 쓰던 곳이 있으면 그대로
        if (c / 'images').is_dir() and any((c / 'images').glob('*.jpg')):
            print(f'기존 위치 사용: {c}'); return c
    for c in CANDS:                                  # ② 없으면 쓰기 되는 첫 곳
        try:
            (c / 'images').mkdir(parents=True, exist_ok=True)
            probe = c / '.probe'; probe.write_text('ok'); probe.unlink()
            if c == CANDS[-1]:
                print(f'⚠️ Drive에 못 써서 로컬 사용: {c} — **세션이 죽으면 라벨이 사라진다.**\n'
                      f'   §B 끝나면 반드시 다운로드하거나 Drive로 복사할 것')
            else: print(f'저장 위치: {c}')
            return c
        except OSError as e:
            print(f'  건너뜀 {c} — {e.strerror}')
    raise RuntimeError('★쓰기 가능한 위치가 없다')

OUT      = evalset_dir()
IMGDIR   = OUT / 'images'
MANIFEST = OUT / f'sample_seed{SEED}_n{N_SAMPLE}.json'
_ID = re.compile(r'cylindrical_(\d+)_')

if MANIFEST.exists() and len(list(IMGDIR.glob('*.jpg'))) >= N_SAMPLE:
    # ★한 번 정해지면 안 바꾼다 — 시험지가 흔들리면 모델 비교가 통째로 죽는다
    names = json.loads(MANIFEST.read_text())
    print(f'기존 표본 재사용: {MANIFEST}')
else:
    # v42 태그가 붙은 것만 — 태그 없는 crop zip은 v4.1 시절이다
    # MyDrive를 먼저 훑는다 — Drive 전체 재귀 글롭은 FUSE에서 매우 느리다
    cand = []
    for pat in ('/content/drive/MyDrive/**/*crop*v42*.zip', '/content/drive/**/*crop*v42*.zip'):
        cand = glob.glob(pat, recursive=True)
        if cand: break
    assert cand, '★ext_crop_v42.zip을 못 찾았다. retrain 노트북 §2로 크롭본을 빌드하거나 Drive 바로가기 확인'
    ZP = max(cand, key=os.path.getsize)
    print(f'zip: {ZP} ({os.path.getsize(ZP)/1e9:.1f} GB) — namelist만 읽는다')

    with zipfile.ZipFile(ZP) as z:
        members = [m for m in z.namelist()
                   if m.startswith(f'images/{SPLIT}/') and m.endswith('.jpg')]
        assert members, f'★zip에 images/{SPLIT}/ 이미지가 없다'
        by = defaultdict(list)
        for m in members:
            g = _ID.search(m)
            by[int(g.group(1)) if g else -1].append(m)
        rng = random.Random(SEED)
        pool = []
        for cid in sorted(by):
            v = sorted(by[cid]); rng.shuffle(v); pool += v[:PER_CELL]
        rng.shuffle(pool)
        pick = pool[:N_SAMPLE]

        # ⚠️ Drive FUSE 무작위 접근은 장당 ~1초다. 150장이면 2~3분.
        names = []
        for i, m in enumerate(pick, 1):
            nm = Path(m).name
            (IMGDIR / nm).write_bytes(z.read(m))
            names.append(nm)
            if i % 30 == 0: print(f'  {i}/{len(pick)}', flush=True)
    MANIFEST.write_text(json.dumps(names, ensure_ascii=False))
    print(f'표본 확정 · Drive 복사 완료: {MANIFEST}')

SAMPLE = [IMGDIR / n for n in names]
MINE   = [p for i, p in enumerate(SAMPLE) if i % SHARD[1] == SHARD[0]]
_ncell = len({(_ID.search(n).group(1) if _ID.search(n) else n) for n in names})
print(f'\n전체 {len(SAMPLE)}장 (셀 {_ncell}개) · 내 몫 {len(MINE)}장 [{WORKER} {SHARD[0]+1}/{SHARD[1]}]')
print(f'이미지 = {IMGDIR}  ← 어느 세션에서든 이 경로')


In [ ]:
# == §B 클릭 라벨링 — 결함 중심에 점 하나 ==
# 조작:  좌클릭=점 추가 · 우클릭/취소=마지막 점 삭제 · [결함 없음]=점 0개로 확정 · [다음]=저장하고 진행
#        유형 버튼을 누르면 그 뒤에 찍는 점의 유형이 바뀐다(기본 '기타').
# 저장:  한 장 끝날 때마다 즉시 기록 → 세션이 죽어도 이어서 하면 된다.
import json, base64, io
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML
from google.colab import output as _co

# ★unsup §1의 QUERY_MAP 키와 **같은 이름·같은 순서**여야 §C가 유형별 recall을 낸다.
#   '기타' = 아래 유형에 안 들어가는 결함. 여기 자주 찍히면 QUERY_MAP에 유형을 추가할 신호다.
TYPES  = ['녹·부식', '벗겨짐·박리', '파손·찢김', '긁힘·스크래치', '들뜸', '오염·이물질', '기타']
LABELS = OUT / f'labels_{WORKER}.json'
done   = json.loads(LABELS.read_text()) if LABELS.exists() else {}
todo   = [p for p in MINE if p.name not in done]
print(f'완료 {len(done)} · 남은 {len(todo)}장\n')

_JS = r"""
window.labelOne = function(src, types, cap) {
  return new Promise((resolve) => {
    const wrap = document.createElement('div');
    wrap.innerHTML = '<div style="font:14px sans-serif;margin:4px 0">' + cap + '</div>';
    const bar = document.createElement('div'); bar.style.margin = '6px 0';
    let cur = types[types.length - 1];
    const btns = types.map(t => {
      const b = document.createElement('button');
      b.textContent = t; b.style.cssText = 'margin:2px;padding:4px 8px;font:13px sans-serif';
      b.onclick = () => { cur = t; btns.forEach(x => x.style.fontWeight = 'normal');
                          b.style.fontWeight = 'bold'; };
      bar.appendChild(b); return b;
    });
    btns[btns.length - 1].style.fontWeight = 'bold';
    const cv = document.createElement('canvas'), ctx = cv.getContext('2d');
    const act = document.createElement('div'); act.style.margin = '6px 0';
    const mk = (t) => { const b = document.createElement('button');
      b.textContent = t; b.style.cssText = 'margin:4px;padding:6px 14px;font:14px sans-serif';
      act.appendChild(b); return b; };
    const bUndo = mk('되돌리기'), bNone = mk('결함 없음'), bNext = mk('다음 ▶');
    wrap.appendChild(bar); wrap.appendChild(cv); wrap.appendChild(act);
    document.body.appendChild(wrap);

    const img = new Image(); const pts = [];
    img.onload = () => {
      const H = Math.min(900, img.height), s = H / img.height;
      cv.width = img.width * s; cv.height = H; draw();
    };
    img.src = src;
    function draw() {
      ctx.drawImage(img, 0, 0, cv.width, cv.height);
      pts.forEach((p, i) => {
        ctx.beginPath(); ctx.arc(p.x * cv.width, p.y * cv.height, 7, 0, 7);
        ctx.fillStyle = 'rgba(255,40,40,.85)'; ctx.fill();
        ctx.strokeStyle = '#fff'; ctx.lineWidth = 2; ctx.stroke();
        ctx.fillStyle = '#ff0'; ctx.font = 'bold 13px sans-serif';
        ctx.fillText(String(i + 1), p.x * cv.width + 10, p.y * cv.height - 8);
      });
    }
    cv.oncontextmenu = (e) => { e.preventDefault(); pts.pop(); draw(); };
    cv.onclick = (e) => {
      const r = cv.getBoundingClientRect();
      pts.push({ x: (e.clientX - r.left) / r.width, y: (e.clientY - r.top) / r.height, t: cur });
      draw();
    };
    bUndo.onclick = () => { pts.pop(); draw(); };
    const fin = () => { wrap.remove(); resolve(JSON.stringify(pts)); };
    bNone.onclick = () => { pts.length = 0; fin(); };
    bNext.onclick = fin;
  });
};
"""

def _uri(p, maxh=1100):
    im = Image.open(p).convert('RGB')
    if im.height > maxh: im = im.resize((round(im.width * maxh / im.height), maxh))
    b = io.BytesIO(); im.save(b, 'JPEG', quality=88)
    return 'data:image/jpeg;base64,' + base64.b64encode(b.getvalue()).decode()

for i, p in enumerate(todo, 1):
    cap = f'[{i}/{len(todo)}] {p.name} — 보이는 결함 중심을 클릭. 없으면 [결함 없음]'
    display(HTML(f'<script>{_JS}</script>'))
    raw = _co.eval_js(f'labelOne("{_uri(p)}", {json.dumps(TYPES, ensure_ascii=False)}, '
                      f'{json.dumps(cap, ensure_ascii=False)})')
    done[p.name] = {'points': json.loads(raw)}       # ★경로는 저장 안 한다 — 세션마다 다르다
    LABELS.write_text(json.dumps(done, ensure_ascii=False, indent=1))   # ★매 장 저장

n_pt = sum(len(v['points']) for v in done.values())
n_cl = sum(1 for v in done.values() if not v['points'])
print(f'\n완료 {len(done)}장 · 점 {n_pt}개 · 무결함 {n_cl}장 → {LABELS}')


In [ ]:
# == §C 채점 — 점 recall (예측 dict만 있으면 어떤 모델이든) ==
# 🔴 이 셀은 **자립**한다. Drive만 읽으므로 §A·§B를 안 돌린 세션에도 붙여넣을 수 있다.
#    → 실제로는 **unsup 노트북 세션에 붙여넣어** 돌리는 게 편하다(거기 detect()가 이미 있다).
#
# 입력: PREDS = {파일명: [(x1, y1, x2, y2, score, tag), ...]}  ← 크롭 좌표계(px)
#       unsup 노트북의 detect()가 그대로 이 모양이다. YOLO 예측도 같은 모양으로 만들면 된다.
# 적중: 사람이 찍은 점이 예측 박스 안 → 적중. **IoU 안 쓴다.**
import json, glob
from pathlib import Path
from collections import defaultdict
from PIL import Image

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass
# §A와 같은 후보 목록 — 라벨이 있는 첫 곳을 쓴다(공유 kt_out이 읽기전용이면 §A가 다른 데 뒀다)
OUT = next((c for c in (Path('/content/drive/MyDrive/kt_out/ext_evalset'),
                        Path('/content/drive/MyDrive/ext_evalset'),
                        Path('/content/ext_evalset'))
            if list(c.glob('labels_*.json'))), None)
assert OUT, '★labels_*.json을 못 찾았다 — §B를 먼저 돌리거나 OUT을 직접 지정할 것'
IMGDIR = OUT / 'images'
print(f'평가셋: {OUT}')

GT = {}
for f in sorted(OUT.glob('labels_*.json')):   # ★팀원 몫을 전부 합친다
    for k, v in json.loads(f.read_text()).items(): GT[k] = v
assert GT, f'★라벨 없음: {OUT}/labels_*.json — §B를 먼저'
EVAL_PATHS = [IMGDIR / k for k in GT]                     # ← 예측을 돌릴 대상
_miss = [p.name for p in EVAL_PATHS if not p.exists()]
assert not _miss, f'★이미지 {len(_miss)}장 없음 (예: {_miss[:2]}) — §A를 돌려 Drive에 채울 것'
print(f'GT {len(GT)}장 · 점 {sum(len(v["points"]) for v in GT.values())}개 '
      f'· 무결함 {sum(1 for v in GT.values() if not v["points"])}장')

def score_preds(PREDS, name='model', budget=None, quiet=False):
    """budget=장당 박스 상한(점수 상위만). None이면 전량. quiet=True면 요약 한 줄만
       🔴 budget은 진단용이다 — 정상 프레임에도 N개를 강제로 꽂아 오탐이 정확히 N×무결함장수가
          된다(0804 §20.1). **운영점 비교에는 쓰지 말고 점수 임계로 스윕할 것.**"""
    hit = defaultdict(int); tot = defaultdict(int)
    nbox = fp_clean = n_clean = 0
    for fn, g in GT.items():
        bx = sorted(PREDS.get(fn, []), key=lambda b: -b[4])
        if budget: bx = bx[:budget]
        nbox += len(bx)
        if not g['points']:
            n_clean += 1; fp_clean += len(bx); continue
        W, H = Image.open(IMGDIR / fn).size
        for pt in g['points']:
            x, y = pt['x'] * W, pt['y'] * H
            tot[pt['t']] += 1
            if any(b[0] <= x <= b[2] and b[1] <= y <= b[3] for b in bx): hit[pt['t']] += 1
    T, Hh = sum(tot.values()), sum(hit.values())
    if quiet:
        print(f'  ({name}) 점 recall {Hh/max(1,T):.3f} · 장당 {nbox/len(GT):.1f}박스')
        return Hh / max(1, T)
    print(f'\n■ {name}' + (f' (예산 {budget}박스/장)' if budget else ''))
    print(f'  점 recall {Hh}/{T} = {Hh/max(1,T):.3f}   |   장당 박스 {nbox/len(GT):.1f}')
    print(f'  무결함 {n_clean}장에 찍힌 박스 {fp_clean}개 ({fp_clean/max(1,n_clean):.1f}/장)')
    for t in sorted(tot, key=lambda k: -tot[k]):
        print(f'    {t:<12} {hit[t]:>3}/{tot[t]:<3} = {hit[t]/tot[t]:.3f}')
    return Hh / max(1, T)

# ⚠️ 유형별 recall이 의미를 가지려면 §B 라벨 유형과 예측 tag 이름이 같아야 한다 —
#    unsup §1의 QUERY_MAP 키가 §B의 TYPES와 같은 이름으로 맞춰져 있다.
#
print(f'\nEVAL_PATHS {len(EVAL_PATHS)}장 준비됨 — 사용 예는 위 주석 참조')


---

## 여기부터: 어제 채팅으로만 받았던 셀들 (0805 정식 편입)

```
§C-run  채점 실행 — OWLv2 vs YOLO · 순열 대조군 · 프레임 게이트   (0804 §20 재현)
§D-1    캡 라벨링 ① 컨택트 시트                                   (선택 — 이미 끝냈으면 건너뜀)
§D-2    캡 라벨링 ② 번호 붙여넣고 저장 → labels_cap.json
§E      전량 VLM 덤프 150장 → vlm_dump.json                        ★미실행
§F      덤프에서 규칙 고르기 (재추론 없음)                          ★미실행
```

**§C·§C-run·§E·§F는 `detect`(unsup §1)와 `_gen`(unsup §4)을 쓴다** →
unsup 노트북 세션에 이 셀들을 붙여넣어 돌리는 것이 정석이다(§C 주석 참조).

🔴 §E·§F를 나눈 이유는 §20.1에서 배운 것과 같다 — 추론과 채점을 붙이면
규칙 하나 바꿀 때마다 전체를 다시 돌린다. 넓게 한 번 모으고 사후에 고른다.

In [ ]:
# == §C-run 채점 실행 — OWLv2 vs YOLO, 순열 대조군 ==
# 선행: unsup §0(FILES) → unsup §1(detect) → 이 노트북 §C(GT · EVAL_PATHS · score_preds)
#       ★unsup 세션에 §C와 이 셀을 붙여넣어 돌리는 게 편하다.
#
# 🔴 사전 확정 규칙 (결과를 보고 바꾸지 않는다): **순열 배율 < 2 이면 그 운영점은 폐기.**
#    남의 이미지 박스로 채점한 recall = 우연 적중이다. 배율이 낮으면 그냥 많이 뿌린 것.
# 🔴 예산 모드(장당 N박스)는 쓰지 않는다 — 정상 프레임에도 N개를 강제로 꽂아 넣어
#    오탐이 정확히 N×51이 된다 = 신호가 가려진다. **점수 임계로만 스윕한다.**
import random
from collections import Counter

def control(preds, name, thr):
    """실측 vs 순열 대조군. 배율 = 실측/대조"""
    f = {k: [t for t in v if t[4] >= thr] for k, v in preds.items()}
    ks = list(f); pm = ks[:]; random.Random(0).shuffle(pm)
    real = score_preds(f, f'{name} thr={thr}')
    ctrl = score_preds({a: f[b] for a, b in zip(ks, pm)}, f'{name} thr={thr} 순열대조', quiet=True)
    r = real / max(ctrl, 1e-9)
    print(f'  ▶ 순열 {ctrl:.3f} → 배율 {r:.2f}× ' + ('✅ 채택' if r >= 2 else '❌ 폐기(<2)'))
    return real, ctrl, r

def gate(preds, name, thrs=(0.05, 0.08, 0.12, 0.18), Ns=(1, 2, 3, 5, 8, 12)):
    """프레임 판정 = 임계 이상 박스가 N개 이상이면 결함. 위치 정확도가 필요 없는 값싼 게이트"""
    rows = []
    for thr in thrs:
        for N in Ns:
            tp = fp = fn = 0
            for k, g in GT.items():
                pred = sum(1 for b in preds.get(k, []) if b[4] >= thr) >= N
                truth = bool(g['points'])
                tp += pred and truth; fp += pred and not truth; fn += (not pred) and truth
            P, R = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
            rows.append((2 * P * R / max(P + R, 1e-9), thr, N, P, R))
    rows.sort(reverse=True)
    print(f'\n■ {name} 프레임 게이트 (상위 8)')
    for F, thr, N, P, R in rows[:8]:
        print(f'   thr {thr:.2f} · N≥{N:<3d} P {P:.3f}  R {R:.3f}  F1 {F:.3f}')
    return rows

# ── ① OWLv2 — 넓게 한 번 뽑고 사후 임계 필터 (재추론 없음) ──────────────
OWL = {p.name: v for p, v in detect(EVAL_PATHS, thr=0.02, topk=50).items()}
print(f'OWLv2 후보 {sum(len(v) for v in OWL.values())}개 · 장당 {sum(len(v) for v in OWL.values())/len(OWL):.1f}\n')

for thr in (0.03, 0.05, 0.08, 0.12, 0.18, 0.25):
    control(OWL, 'OWLv2', thr)

# 정상/결함 박스 밀도 비대칭 — 우연이 아니라 실제로 구분하는지 본다
print('\n■ 박스 밀도 (결함 프레임 vs 무결함 프레임)')
DEF = [k for k, g in GT.items() if g['points']]; CLN = [k for k, g in GT.items() if not g['points']]
for thr in (0.05, 0.08, 0.12, 0.18, 0.25):
    d = sum(sum(1 for b in OWL[k] if b[4] >= thr) for k in DEF) / len(DEF)
    c = sum(sum(1 for b in OWL[k] if b[4] >= thr) for k in CLN) / max(len(CLN), 1)
    print(f'   thr {thr:.2f}   결함 {d:5.1f}/장   무결함 {c:5.1f}/장   비대칭 {d/max(c,1e-9):.1f}×')

gate(OWL, 'OWLv2')

# ── ② YOLO 챔피언 — **같은 시험지**에. 이게 없으면 "나아졌다"를 말할 수 없다 ──
import glob, os
CAND = sorted(glob.glob('/content/drive/MyDrive/**/train_ext_*/weights/*.pt', recursive=True))
print('\n가중치 후보:', Counter(os.path.basename(os.path.dirname(os.path.dirname(c))) for c in CAND))
WPICK = next((c for c in CAND if 'v5_idcap' in c and c.endswith('best.pt')),
             next((c for c in CAND if c.endswith('best.pt')), None))
if not WPICK:
    print('⚠️ YOLO 가중치를 못 찾음 — 위 후보에서 직접 골라 WPICK에 넣을 것')
else:
    print(f'YOLO: {WPICK}')
    from ultralytics import YOLO
    _m = YOLO(WPICK)
    YOL = {}
    for p in EVAL_PATHS:
        r = _m.predict(str(p), conf=0.005, verbose=False)[0].boxes
        YOL[p.name] = [(*b.tolist(), float(c), 'yolo', 'yolo')
                       for b, c in zip(r.xyxy.cpu(), r.conf.cpu())]
    for thr in (0.01, 0.05, 0.10, 0.25):
        control(YOL, 'YOLO', thr)
    gate(YOL, 'YOLO')

print('\n' + '=' * 64)
print('판정: 배율 ≥2를 통과하는 운영점끼리만 비교한다. 통과 못 한 숫자는 쓰지 않는다.')

In [ ]:
# == §D-1 캡 라벨링 ① — 상단 캡만 잘라 컨택트 시트 ==
# 왜 캡만 따로 다는가: VLM이 셀 상단 금속캡의 **반사·검은 홈을 긁힘으로 오탐**한다.
#   원인은 대조 기준이다 — 기준 패치를 늘 몸통에서 떠 오니 캡을 외피와 비교하게 된다.
#   캡은 셀이 달라도 같은 금속이라 **셀 간 비교가 성립한다**(외피는 색이 달라 안 된다).
#   → 사람이 확인한 정상 캡 갤러리를 만들어 §E가 왼쪽 기준으로 쓴다.
# 🔴 Colab에서 input()은 긴 출력 아래에 숨어 안 뜬다 → stdin 안 쓴다. 번호를 §D-2에 붙여넣는다.
import json
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

CAP_ZONE, GRID, TW = 0.22, 5, 150     # 셀 높이 상단 22% · 5×5 시트 · 칸 폭

def cell_box(im, tol=55, frac=0.12):
    """배경(테두리 색)과 다른 픽셀이 몰린 구간 = 셀 몸통"""
    a = np.asarray(im, np.float32); H, W = a.shape[:2]; e = max(1, W // 20)
    bg = np.concatenate([a[:, :e].reshape(-1, 3), a[:, -e:].reshape(-1, 3)])
    m = np.abs(a - np.median(bg, 0)).sum(2) > tol
    xs = np.where(m.mean(0) > frac)[0]; ys = np.where(m.mean(1) > frac)[0]
    return (0, 0, W, H) if len(xs) < 5 or len(ys) < 5 else \
           (int(xs[0]), int(ys[0]), int(xs[-1]) + 1, int(ys[-1]) + 1)

def cap_crop(p):
    im = Image.open(p).convert('RGB'); cb = cell_box(im)
    return im.crop((cb[0], cb[1], cb[2], int(cb[1] + (cb[3] - cb[1]) * CAP_ZONE)))

CAPFILES = sorted(GT)                      # ★번호 1..150 고정. 이 순서를 §D-2가 그대로 쓴다
print(f'캡 {len(CAPFILES)}장 · 5×5 시트 {-(-len(CAPFILES)//(GRID*GRID))}장\n'
      f'보는 법: 캡에 **부식·눌림·변형·이물질**이 있으면 결함. 반사·검은 홈·각인은 정상.\n')

for s in range(0, len(CAPFILES), GRID * GRID):
    chunk = CAPFILES[s:s + GRID * GRID]
    ims = [cap_crop(IMGDIR / n) for n in chunk]
    TH = max(1, round(TW * max(i.height / i.width for i in ims)))
    sh = Image.new('RGB', (GRID * TW, -(-len(ims) // GRID) * TH), 'white')
    dr = ImageDraw.Draw(sh)
    for i, im in enumerate(ims):
        ox, oy = i % GRID * TW, i // GRID * TH
        sh.paste(im.resize((TW, TH), Image.LANCZOS), (ox, oy))
        dr.rectangle([ox, oy, ox + 30, oy + 18], fill=(0, 0, 0))
        dr.text((ox + 4, oy + 4), str(s + i + 1), fill=(255, 255, 0))
    print(f'── {s+1} ~ {s+len(ims)} ──'); display(sh)

print('\n▶ 결함 캡의 번호를 적어 두고 §D-2에 붙여넣을 것 (예: BAD = "1 3 7 12-15 20")')

In [ ]:
# == §D-2 캡 라벨링 ② — 번호 붙여넣고 저장 ==
# BAD = 결함 캡 번호 · UNK = 애매해서 판단 보류(= -1, 채점에서 제외한다. 억지로 0/1을 주면
#       그 오라벨이 그대로 §F 판정에 들어간다. 모르는 건 모른다고 기록하는 게 맞다)
# 공백·쉼표 아무거나 · 범위는 하이픈(12-15)
BAD = ""
UNK = ""

import json, re
from PIL import Image
from IPython.display import display

def _nums(s):
    out = set()
    for tok in re.split(r'[\s,]+', s.strip()):
        if not tok: continue
        if '-' in tok:
            a, b = tok.split('-'); out |= set(range(int(a), int(b) + 1))
        else: out.add(int(tok))
    return out

bad, unk = _nums(BAD), _nums(UNK)
assert not (bad & unk), f'★BAD와 UNK에 겹치는 번호: {sorted(bad & unk)}'
assert all(1 <= n <= len(CAPFILES) for n in bad | unk), f'★1~{len(CAPFILES)} 범위를 벗어난 번호'

CAPL = {n: (1 if i + 1 in bad else (-1 if i + 1 in unk else 0))
        for i, n in enumerate(CAPFILES)}
(OUT / 'labels_cap.json').write_text(json.dumps(CAPL, ensure_ascii=False, indent=1), encoding='utf-8')

_ok = [n for n, v in CAPL.items() if v == 0]
_ncell = len({n.split('cylindrical_')[-1][:4] for n in _ok})
print(f'저장 {OUT/"labels_cap.json"}\n'
      f'정상 캡 {len(_ok)}장 (셀 {_ncell}개) · 결함 {sum(v==1 for v in CAPL.values())} '
      f'· 불확실 {sum(v==-1 for v in CAPL.values())}')

# 정상 캡 갤러리 미리보기 — §E가 왼쪽 기준으로 쓸 바로 그 그림이다. 여기 결함이 섞이면 §E가 통째로 망가진다
sel = []
for n in _ok:                                    # 셀 하나당 1장씩만 (같은 캡 16장은 정보가 1장이다)
    cid = n.split('cylindrical_')[-1][:4]
    if cid not in [s[0] for s in sel]: sel.append((cid, n))
    if len(sel) >= 16: break
ims = [cap_crop(IMGDIR / n) for _, n in sel]
T = 110; TH = max(1, round(T * max(i.height / i.width for i in ims)))
g = Image.new('RGB', (4 * T, -(-len(ims) // 4) * TH), 'white')
for i, im in enumerate(ims): g.paste(im.resize((T, TH), Image.LANCZOS), (i % 4 * T, i // 4 * TH))
print(f'\n정상 캡 갤러리 {len(ims)}장 (서로 다른 셀) — 여기 결함이 보이면 §D-1로 돌아갈 것')
display(g)

In [ ]:
# == §E 전량 VLM 덤프 — 150장 추론 → vlm_dump.json (채점은 §F) ==
# 🔴 추론과 채점을 붙이면 규칙 하나 바꿀 때마다 40분을 다시 쓴다.
#    여기서는 넓게 한 번 모으고(모든 박스 점수 · 등급 · 종류 · 관찰 · 면적 · ΔE) 규칙은 §F에서 고른다.
# 🔴 프롬프트에 **JSON 예시도 정상 특징 나열도 넣지 않는다** — 넣으면 그 문장을 그대로 읊는다.
#    → 답을 세 줄 평문으로 받고 정규식으로 읽는다. 베낄 틀이 없다.
#
# 선행: unsup §0 → §1(detect) → §4(_gen · GRADES · KIND_TXT) · 이 노트북 §C(GT · IMGDIR · OUT)
#       §D는 선택 — labels_cap.json이 있으면 캡 갤러리를 기준으로 쓰고, 없으면 몸통 기준만 쓴다.
import json, re, time
import numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display

assert callable(detect), '★unsup §1을 먼저 (detect가 없다)'
assert callable(_gen),   '★unsup §4를 먼저 (_gen이 없다)'

PROMPT_VER = 'v8'          # ★프롬프트를 바꾸면 여기를 올린다 → 덤프 파일이 갈려 옛 결과가 안 섞인다
DUMP = OUT / f'vlm_dump_{PROMPT_VER}.json'
THR_V, CAP_N   = 0.12, 6      # §20.2 확정 위치 임계 · 프레임당 VLM에 보낼 박스 상한
MARGIN, PANEL  = 0.35, 336    # 크롭 여백(박스 최대변의 35%) · 확대 크기
TOP_SKIP, BOT_SKIP = 0.02, 0.93   # 셀 위/아래 밴드 제외 (바닥 그림자·테두리를 결함으로 찍는다)
CAP_ZONE, GAL_N    = 0.22, 16     # 셀 높이 상단 22% = 캡 구역 · 갤러리 장수

def fit(im, px):
    """🔴 thumbnail()은 축소만 하고 확대는 안 한다(0804 §11) → resize로 양방향"""
    k = px / max(im.size)
    return im.resize((max(1, round(im.width * k)), max(1, round(im.height * k))), Image.LANCZOS)

def cell_box(im, tol=55, frac=0.12):
    """배경(테두리 색)과 다른 픽셀이 몰린 구간 = 셀 몸통. 바닥·여백을 좌표계에서 빼는 데 쓴다"""
    a = np.asarray(im, np.float32); H, W = a.shape[:2]; e = max(1, W // 20)
    bg = np.concatenate([a[:, :e].reshape(-1, 3), a[:, -e:].reshape(-1, 3)])
    m = np.abs(a - np.median(bg, 0)).sum(2) > tol
    xs = np.where(m.mean(0) > frac)[0]; ys = np.where(m.mean(1) > frac)[0]
    return (0, 0, W, H) if len(xs) < 5 or len(ys) < 5 else \
           (int(xs[0]), int(ys[0]), int(xs[-1]) + 1, int(ys[-1]) + 1)

def _lab(rgb):
    m = np.array([[0.412, 0.358, 0.180], [0.213, 0.715, 0.072], [0.019, 0.119, 0.950]])
    xyz = rgb @ m.T / np.array([0.9505, 1.0, 1.089])
    f = np.where(xyz > 0.008856, np.cbrt(xyz), 7.787 * xyz + 16 / 116)
    return np.stack([116 * f[..., 1] - 16, 500 * (f[..., 0] - f[..., 1]),
                     200 * (f[..., 1] - f[..., 2])], -1)

def tight(im, b):
    w, h = b[2] - b[0], b[3] - b[1]; m = max(w, h) * MARGIN
    return fit(im.crop((max(0, b[0] - m), max(0, b[1] - m),
                        min(im.width, b[2] + m), min(im.height, b[3] + m))), PANEL)

def ref_patch(im, cb, boxes, b):
    """같은 셀에서 **박스가 가장 먼 곳**을 같은 크기로 떠 온다 = 이 셀의 정상 표면.
       셀마다 외피 색이 달라 셀 간 비교가 성립하지 않는다 → 몸통 기준은 반드시 같은 셀에서."""
    w, h = b[2] - b[0], b[3] - b[1]; cx = (cb[0] + cb[2]) / 2
    best, bd = None, -1.0
    for f in np.linspace(0.30, 0.88, 9):
        cy = cb[1] + (cb[3] - cb[1]) * f
        d = min((abs(cy - (q[1] + q[3]) / 2) for q in boxes), default=1e9)
        if d > bd: best, bd = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2), d
    return fit(im.crop(tuple(map(int, best))), PANEL)

def pair(ref, tgt):
    """왼쪽 기준 · 오른쪽 검사 대상. 한 장으로 붙여야 '차이가 있나'라는 질문이 성립한다"""
    W, H = ref.width + tgt.width + 12, max(ref.height, tgt.height)
    p = Image.new('RGB', (W, H), (255, 255, 255))
    p.paste(ref, (0, (H - ref.height) // 2)); p.paste(tgt, (ref.width + 12, (H - tgt.height) // 2))
    return p

# ── 정상 캡 갤러리 (§D 산출물이 있으면) ────────────────────────────────
# 왜 캡만 따로: 기준 패치를 늘 몸통에서 뜨니 캡을 외피와 비교하게 되고, 반사·검은 홈이
#   "왼쪽에 없는 것"이 되어 긁힘으로 나온다. 캡은 셀이 달라도 같은 금속이라
#   **셀 간 비교가 성립하는 유일한 부위**다.
GAL = None
_cf = OUT / 'labels_cap.json'
_CAPL = json.loads(_cf.read_text(encoding='utf-8')) if _cf.exists() else {}
if _CAPL:
    _ok, _seen = [], set()
    for n, v in _CAPL.items():
        cid = n.split('cylindrical_')[-1][:4]
        if v == 0 and cid not in _seen: _seen.add(cid); _ok.append(n)
        if len(_ok) >= GAL_N: break
    _ims = []
    for n in _ok:
        im = Image.open(IMGDIR / n).convert('RGB'); cb = cell_box(im)
        _ims.append(im.crop((cb[0], cb[1], cb[2], int(cb[1] + (cb[3] - cb[1]) * CAP_ZONE))))
    T = 110; TH = max(1, round(T * max(i.height / i.width for i in _ims)))
    GAL = Image.new('RGB', (4 * T, -(-len(_ims) // 4) * TH), 'white')
    for i, im in enumerate(_ims):
        GAL.paste(im.resize((T, TH), Image.LANCZOS), (i % 4 * T, i // 4 * TH))
    GAL = fit(GAL, PANEL + 120)
    print(f'정상 캡 갤러리 {len(_ims)}장 (셀 {len(_seen)}개)'); display(GAL)
else:
    print('⚠️ labels_cap.json 없음 — 캡도 몸통 기준으로 본다(캡 오탐이 남는다). §D를 돌리면 개선된다')

# ── 프롬프트 v8 — 실측으로 두 가지를 뜯어고친다 ────────────────
# 🔴 ① 채점 기준표(GRADES)를 프롬프트에서 뺀다.
# 🔴 ② 한글을 생성시키지 않는다. **번호·알파벳으로만 답하게** 한다.
# 🔑 크기를 따로 묻는 이유 = **검증 장치**. 우리가 픽셀로 잰 면적과 맞아떨어지면 모델이 실제로
#    사진을 보고 있다는 증거고, 어긋나면 또 베끼는 중이다. §F가 이 상관을 찍는다.
# 🔑 등급은 VLM에게 묻지 않는다 — (종류, 크기)에서 **코드가 결정론적으로 계산**한다.
#    모델 카드에 적을 수 있는 규칙이 되고, 같은 입력이면 같은 등급이 나온다.
KIND_ID = {0: '정상', 1: '녹·부식', 2: '벗겨짐·박리', 3: '파손·찢김',
           4: '긁힘·스크래치', 5: '들뜸', 6: '오염·이물질'}
SIZE_G  = {'A': 4, 'B': 3, 'C': 2, 'D': 1}      # 크기 → 기본 등급
KIND_UP = {2, 3}                                # 벗겨짐·파손은 금속 노출로 이어진다 → +1

_ANS = ('\n\n오른쪽에서 보이는 것을 고르라. 설명하지 말고 아래 세 줄만 답하라.\n\n'
        '종류  0 없음  1 녹·부식  2 벗겨짐  3 찢김·파손  4 긁힘  5 들뜸  6 오염·이물질\n'
        '크기  A 사진 절반 이상   B 사진의 1/4쯤   C 손톱만 함   D 점 하나\n\n'
        '종류: (숫자)\n크기: (알파벳)\n본것: (한 문장)')
P_BODY = ('왼쪽은 같은 셀의 다른 부위다. 오른쪽이 검사 대상이다.\n'
          '왼쪽에는 없고 오른쪽에만 있는 것을 찾아라.' + _ANS)
P_CAP  = ('왼쪽은 정상으로 확인된 배터리 상단 금속 캡 사진들이다. 오른쪽이 검사 대상이다.\n'
          '왼쪽 어느 사진에도 없는 것을 찾아라.' + _ANS)

def parse_v8(r):
    """→ (종류번호, 크기, 등급, 한 문장). 못 읽으면 종류 0(=정상)으로 둔다 — 모르는 걸 결함으로 치지 않는다"""
    t = str(r.get('셀요약') or json.dumps(r, ensure_ascii=False)) if isinstance(r, dict) else str(r)
    k = re.search(r'종류\D{0,4}([0-6])', t)
    z = re.search(r'크기\W{0,4}([A-Da-d])', t)
    o = re.search(r'본것\s*[:：]\s*([^\n]+)', t)
    kid = int(k.group(1)) if k else 0
    sz  = (z.group(1).upper() if z else 'D')
    g   = 0 if kid == 0 else min(5, SIZE_G[sz] + (1 if kid in KIND_UP else 0))
    return kid, sz, g, (o.group(1).strip() if o else t.strip())[:140]

# ── 검출은 한 번만. 모든 점수를 저장해 §F가 어떤 임계로도 게이트를 재구성한다 ──
NAMES = sorted(GT)
DETS  = {p.name: v for p, v in detect([IMGDIR / n for n in NAMES], thr=0.02, topk=50).items()}
dump  = json.loads(DUMP.read_text(encoding='utf-8')) if DUMP.exists() else {}
# 🔴 이름만 보고 건너뛰면 **스키마가 바뀐 걸 못 본다** — 옛 덤프가 그대로 남아
#    §F가 KeyError로 죽었다. 필요한 키가 다 있는 것만 완료로 친다.
_NEED = {'frame_defect', 'scores', 'scores_band', 'items'}   # ★파일명이 버전별로 갈린다
todo  = [n for n in NAMES if not _NEED <= set(dump.get(n, {}))]
print(f'\n덤프 {len(dump)}장 완료 · 남은 {len(todo)}장')

t0 = time.time()
for i, name in enumerate(todo, 1):
    im = Image.open(IMGDIR / name).convert('RGB'); cb = cell_box(im)
    CH = max(cb[3] - cb[1], 1); CA = max((cb[2] - cb[0]) * CH, 1)
    med = np.median(_lab(np.asarray(im.crop(cb).resize((48, 160)), np.float32) / 255.)
                    .reshape(-1, 3), 0)

    raw  = DETS.get(name, [])
    band = [t for t in raw if TOP_SKIP < ((t[1] + t[3]) / 2 - cb[1]) / CH < BOT_SKIP]
    keep = [t for t in band if t[4] >= THR_V][:CAP_N]

    items = []
    for t in keep:
        b = list(t[:4])
        zone = 'cap' if ((b[1] + b[3]) / 2 - cb[1]) / CH < CAP_ZONE else 'body'
        use_gal = (zone == 'cap' and GAL is not None)
        ref = GAL if use_gal else ref_patch(im, cb, [list(x[:4]) for x in keep], b)
        kid, sz, g, obs = parse_v8(_gen(pair(ref, tight(im, b)),
                                        P_CAP if use_gal else P_BODY, 120))
        cl = _lab(np.asarray(fit(im.crop(tuple(map(int, b))), 32), np.float32) / 255.)
        items.append({'bbox': [round(v, 1) for v in b], 'score': round(t[4], 4),
                      'tag': t[5], 'query': t[6], 'zone': zone,
                      'area': round((b[2] - b[0]) * (b[3] - b[1]) / CA * 100, 3),
                      'dE': round(float(np.linalg.norm(cl.reshape(-1, 3).mean(0) - med)), 2),
                      'grade': g, 'kind_id': kid, 'kind': KIND_ID[kid],
                      'size': sz, 'obs': obs})

    dump[name] = {'frame_defect': bool(GT[name]['points']), 'n_points': len(GT[name]['points']),
                  'cap_label': _CAPL.get(name),
                  'scores':      [round(t[4], 4) for t in raw],   # ★§F가 임계를 자유롭게 정한다
                  'scores_band': [round(t[4], 4) for t in band],
                  'items': items}
    if i % 5 == 0 or i == len(todo):
        DUMP.write_text(json.dumps(dump, ensure_ascii=False), encoding='utf-8')
        el = time.time() - t0
        print(f'  {i}/{len(todo)} · {el/i:.1f}s/장 · 남은 {(len(todo)-i)*el/i/60:.0f}분', flush=True)

print(f'\n덤프 완료 {len(dump)}장 → {DUMP}\n다음: §F — 재추론 없이 규칙을 고른다')

In [ ]:
# == §E-fix 덤프 스키마 보정 — VLM 재호출 없음 (1~2분) ==
# 원인: 이전 버전 §E가 만든 항목에는 박스 점수 목록(scores/scores_band)이 없다.
#       §E의 재개 로직이 **이름만 보고** 건너뛰어서 옛 스키마가 그대로 남았다 → §F가 KeyError.
# 여기서 채우는 건 전부 **검출기와 기하에서 나오는 값**이다(등급·종류·관찰은 VLM 것이라 손대지 않는다).
#   scores/scores_band → detect() 재실행 · zone/area/dE → bbox와 셀 좌표계에서 계산
# 선행: unsup §1(detect) · 이 노트북 §C(GT·IMGDIR·OUT)
import json
import numpy as np
from PIL import Image

TOP_SKIP, BOT_SKIP, CAP_ZONE = 0.02, 0.93, 0.22
DUMP = OUT / 'vlm_dump.json'
dump = json.loads(DUMP.read_text(encoding='utf-8'))

def cell_box(im, tol=55, frac=0.12):
    a = np.asarray(im, np.float32); H, W = a.shape[:2]; e = max(1, W // 20)
    bg = np.concatenate([a[:, :e].reshape(-1, 3), a[:, -e:].reshape(-1, 3)])
    m = np.abs(a - np.median(bg, 0)).sum(2) > tol
    xs = np.where(m.mean(0) > frac)[0]; ys = np.where(m.mean(1) > frac)[0]
    return (0, 0, W, H) if len(xs) < 5 or len(ys) < 5 else \
           (int(xs[0]), int(ys[0]), int(xs[-1]) + 1, int(ys[-1]) + 1)

def _lab(rgb):
    m = np.array([[0.412, 0.358, 0.180], [0.213, 0.715, 0.072], [0.019, 0.119, 0.950]])
    xyz = rgb @ m.T / np.array([0.9505, 1.0, 1.089])
    f = np.where(xyz > 0.008856, np.cbrt(xyz), 7.787 * xyz + 16 / 116)
    return np.stack([116 * f[..., 1] - 16, 500 * (f[..., 0] - f[..., 1]),
                     200 * (f[..., 1] - f[..., 2])], -1)

# ── 무엇이 없는지부터 찍는다. 추측으로 채우지 않는다 ──────────────────
FRAME_NEED = {'frame_defect', 'scores', 'scores_band', 'items'}
ITEM_NEED  = {'bbox', 'score', 'zone', 'area', 'dE', 'grade', 'kind', 'obs'}
_fk = set().union(*(set(v) for v in dump.values()))
_ik = set().union(*(set(i) for v in dump.values() for i in v.get('items', [])), set())
print(f'덤프 {len(dump)}장')
print(f'  프레임 키 있음 : {sorted(_fk)}')
print(f'  프레임 키 없음 : {sorted(FRAME_NEED - _fk)}')
print(f'  박스   키 있음 : {sorted(_ik)}')
print(f'  박스   키 없음 : {sorted(ITEM_NEED - _ik)}')
_lost = {'grade', 'kind', 'obs'} - _ik
assert not _lost, (f'★VLM 산출물 {sorted(_lost)}이 없다 = 이 덤프는 못 살린다.\n'
                   f'   {DUMP} 를 지우고 §E를 처음부터 돌릴 것')

need = [n for n, v in dump.items() if not FRAME_NEED <= set(v)]
print(f'\n보정 대상 {len(need)}장')

if need:
    DETS = {p.name: v for p, v in detect([IMGDIR / n for n in need], thr=0.02, topk=50).items()}
    for n in need:
        v = dump[n]
        im = Image.open(IMGDIR / n).convert('RGB'); cb = cell_box(im)
        CH = max(cb[3] - cb[1], 1); CA = max((cb[2] - cb[0]) * CH, 1)
        raw = DETS.get(n, [])
        v['scores'] = [round(t[4], 4) for t in raw]
        v['scores_band'] = [round(t[4], 4) for t in raw
                            if TOP_SKIP < ((t[1] + t[3]) / 2 - cb[1]) / CH < BOT_SKIP]
        v.setdefault('frame_defect', bool(GT[n]['points']))
        v.setdefault('n_points', len(GT[n]['points']))
        med = None
        for it in v.get('items', []):
            b = it['bbox']
            it.setdefault('zone', 'cap' if ((b[1] + b[3]) / 2 - cb[1]) / CH < CAP_ZONE else 'body')
            it.setdefault('area', round((b[2] - b[0]) * (b[3] - b[1]) / CA * 100, 3))
            if 'dE' not in it:
                if med is None:
                    med = np.median(_lab(np.asarray(im.crop(cb).resize((48, 160)), np.float32)
                                         / 255.).reshape(-1, 3), 0)
                c = im.crop(tuple(map(int, b)))
                k = 32 / max(c.size)
                c = c.resize((max(1, round(c.width * k)), max(1, round(c.height * k))))
                it['dE'] = round(float(np.linalg.norm(
                    _lab(np.asarray(c, np.float32) / 255.).reshape(-1, 3).mean(0) - med)), 2)
    DUMP.write_text(json.dumps(dump, ensure_ascii=False), encoding='utf-8')
    print(f'보정 완료 → {DUMP}')

_bad = [n for n, v in dump.items() if not FRAME_NEED <= set(v)
        or any(not ITEM_NEED <= set(i) for i in v.get('items', []))]
assert not _bad, f'★아직 부족한 프레임 {len(_bad)}장: {_bad[:3]}'
print(f'검증 통과 — {len(dump)}장 전부 §F 스키마. 이제 §F를 돌리면 된다')

In [ ]:
# == §F 채점 — vlm_dump.json에서 규칙을 고른다 (재추론 없음) ==
# 여기서 답할 것 하나: **VLM 등급이 OWLv2 박스 개수보다 나은 판정을 주는가?**
#   못 주면 게이트는 OWLv2로 두고 VLM은 리포트(유형·근거 문장)에만 쓴다 — 그것도 결론이다.
# 🔴 규칙을 늘려 최고 F1을 뽑으면 150장에 과적합한다. 그래서 (a) 규칙군을 미리 정해 두고
#    (b) **동점이면 단순한 쪽**을 고르고 (c) 1등과 2등의 차이가 프레임 2장 이내면 "차이 없음"이라 쓴다.
# 선행: §E (vlm_dump.json)
import json
from pathlib import Path
from collections import Counter

_c = sorted(OUT.glob('vlm_dump*.json'), key=lambda p: p.stat().st_mtime)
assert _c, '★§E를 먼저 (vlm_dump*.json 없음)'
DF = _c[-1]                       # 가장 최근 것. 옛 버전을 보려면 DF를 직접 지정
D = json.loads(DF.read_text(encoding='utf-8'))
NDEF = sum(v['frame_defect'] for v in D.values())
print(f'덤프 {DF.name} · {len(D)}장 (결함 {NDEF} · 무결함 {len(D)-NDEF})')
print(f'다른 덤프: {[p.name for p in _c[:-1]] or "없음"}\n')

def ev(pred, label):
    tp = fp = fnn = 0
    for v in D.values():
        p, t = pred(v), v['frame_defect']
        tp += p and t; fp += p and not t; fnn += (not p) and t
    P, R = tp / max(tp + fp, 1), tp / max(tp + fnn, 1)
    return (2 * P * R / max(P + R, 1e-9), P, R, fp, fnn, label)

def gmax(v, zone=None):
    g = [i['grade'] for i in v['items'] if zone is None or i['zone'] == zone]
    return max(g, default=0)

def gcnt(v, g, zone=None):
    return sum(1 for i in v['items'] if i['grade'] >= g and (zone is None or i['zone'] == zone))

ROWS = []
for thr in (0.05, 0.08, 0.12, 0.18):
    for N in (1, 2, 3, 5, 8, 12):
        ROWS.append(ev(lambda v, t=thr, n=N: sum(1 for s in v['scores_band'] if s >= t) >= n,
                       f'검출VLM(OWLv2) thr{thr:.2f} N≥{N}'))
# ①' 밴드 필터 A/B — §E가 넣은 위 2%·아래 7% 제외가 실제로 이득인가
for thr in (0.08, 0.12):
    for N in (2, 5, 8, 12):
        ROWS.append(ev(lambda v, t=thr, n=N: sum(1 for s in v['scores'] if s >= t) >= n,
                       f'검출VLM 밴드OFF thr{thr:.2f} N≥{N}'))
# ② VLM 최고등급
for g in (2, 3, 4, 5):
    ROWS.append(ev(lambda v, g=g: gmax(v) >= g, f'VLM  최고등급 ≥{g}'))
    ROWS.append(ev(lambda v, g=g: gmax(v, 'body') >= g, f'VLM  몸통만 최고등급 ≥{g}'))
# ③ VLM 등급별 박스 개수
for g in (2, 3, 4):
    for k in (1, 2, 3):
        ROWS.append(ev(lambda v, g=g, k=k: gcnt(v, g) >= k, f'VLM  {g}등급↑ 박스 ≥{k}'))
        ROWS.append(ev(lambda v, g=g, k=k: gcnt(v, g, 'body') >= k, f'VLM  몸통 {g}등급↑ ≥{k}'))
# ③' v8 종류 번호 — 0(없음)이 아니면 결함. 등급을 안 거치는 가장 직접적인 신호
if any('kind_id' in i for v in D.values() for i in v['items']):
    for k in (1, 2, 3):
        ROWS.append(ev(lambda v, k=k: sum(1 for i in v['items'] if i.get('kind_id')) >= k,
                       f'VLM  결함판정 박스 ≥{k}'))
        ROWS.append(ev(lambda v, k=k: sum(1 for i in v['items']
                                          if i.get('kind_id') and i['zone'] == 'body') >= k,
                       f'VLM  몸통 결함판정 ≥{k}'))
# ③'' VLM 크기 — 등급을 안 거치고 크기만으로. 크기가 면적과 맞는 걸로 나왔다
if any('size' in i for v in D.values() for i in v['items']):
    for z in ('A', 'AB'):
        for k in (1, 2, 3):
            ROWS.append(ev(lambda v, z=z, k=k: sum(1 for i in v['items']
                                                   if i.get('size') in z) >= k,
                           f'VLM  크기{z} 박스 ≥{k}'))
# ④ 측정치 (VLM 말고 픽셀) — 등급이 못 가르면 이게 가르는지 본다
for a in (0.2, 0.5, 1.0):
    ROWS.append(ev(lambda v, a=a: any(i['area'] >= a for i in v['items']), f'면적  ≥{a}%'))
for d in (8, 14, 20):
    ROWS.append(ev(lambda v, d=d: any(i['dE'] >= d for i in v['items']), f'ΔE    ≥{d}'))
# ⑤ 조합 — 값싼 게이트(OWLv2)로 넓게 잡고 VLM으로 좁힌다
for g in (3, 4):
    ROWS.append(ev(lambda v, g=g: sum(1 for s in v['scores_band'] if s >= 0.08) >= 8 and gmax(v) >= g,
                   f'OWLv2 N≥8  AND  VLM ≥{g}등급'))
    ROWS.append(ev(lambda v, g=g: sum(1 for s in v['scores_band'] if s >= 0.08) >= 8 or gmax(v) >= g,
                   f'OWLv2 N≥8  OR   VLM ≥{g}등급'))

ROWS.sort(reverse=True)
print(f'{"규칙":32s} {"F1":>6s} {"P":>6s} {"R":>6s} {"오탐":>5s} {"놓침":>5s}')
for F, P, R, fp, fnn, lab in ROWS[:18]:
    print(f'{lab:32s} {F:6.3f} {P:6.3f} {R:6.3f} {fp:5d} {fnn:5d}')

best = ROWS[0]
simple = max((r for r in ROWS if r[5].startswith('검출VLM(OWLv2)')), key=lambda r: r[0])
d_fp = abs(best[3] - simple[3]) + abs(best[4] - simple[4])
print(f'\n■ 1등          {best[5]}  F1 {best[0]:.3f}')
print(f'■ 1단만        {simple[5]}  F1 {simple[0]:.3f}   (OWLv2도 VLM이다 — 텍스트 질의 검출)')
print('■ 판정: ' + (f'{best[5].strip()} 가 1단 단독보다 프레임 {d_fp}장 낫다 → 2단을 게이트에도 쓴다'
                    if best[0] > simple[0] and d_fp > 2 else
                    f'1단(OWLv2)만으로 충분하다 (프레임 차 {d_fp}장 ≤ 2).\n'
                    '        → 게이트 = 1단 박스 개수 · 2단(Qwen) = 유형·크기·문장 리포트.\n'
                    '        두 단 다 VLM이다. 어느 쪽도 빠지지 않는다.'))

# ── 진단: VLM이 실제로 사진을 보고 있는가 ──────────────────────────────
ITEMS = [i for v in D.values() for i in v['items']]
obs = [i['obs'] for i in ITEMS]
print(f'\n박스 {len(ITEMS)}개 · 관찰 문장 서로 다른 것 {len(set(obs))}개 '
      f'({len(set(obs))/max(len(obs),1):.0%})' +
      ('  (문장 다양성은 요구사항이 아니다 — 리포트 VLM이 따로 있다. '
       '핵심 단어가 맞는지는 §G에서 잰다)' if len(set(obs)) < len(obs) * 0.5 else ''))
print('등급 분포  결함프레임:',
      Counter(i['grade'] for v in D.values() if v['frame_defect'] for i in v['items']).most_common())
print('           무결함  :',
      Counter(i['grade'] for v in D.values() if not v['frame_defect'] for i in v['items']).most_common())
_g = lambda t: [i['grade'] for v in D.values() if v['frame_defect'] == t for i in v['items']]
_d, _c = _g(True), _g(False)
print('★4등급↑ 비율   결함 %.1f%% (%d/%d)  vs  무결함 %.1f%% (%d/%d)  → %s' % (
    100 * sum(x >= 4 for x in _d) / max(len(_d), 1), sum(x >= 4 for x in _d), len(_d),
    100 * sum(x >= 4 for x in _c) / max(len(_c), 1), sum(x >= 4 for x in _c), len(_c),
    '분리력 없음(등급을 판정에 쓰지 말 것)'
    if sum(x >= 4 for x in _d) / max(len(_d), 1) <= sum(x >= 4 for x in _c) / max(len(_c), 1) + .1
    else '분리됨'))
print('구역별 평균등급  캡 %.2f  몸통 %.2f' % (
    sum(i['grade'] for i in ITEMS if i['zone'] == 'cap') / max(sum(i['zone'] == 'cap' for i in ITEMS), 1),
    sum(i['grade'] for i in ITEMS if i['zone'] == 'body') / max(sum(i['zone'] == 'body' for i in ITEMS), 1)))
print('종류:', Counter(i['kind'] for i in ITEMS).most_common(8))
if any('kind_id' in i for i in ITEMS):
    _ok = sum(1 for i in ITEMS if i.get('kind_id') == 0)
    print(f"★2단이 '정상'이라 한 박스 {_ok}/{len(ITEMS)} = {100*_ok/max(len(ITEMS),1):.1f}%"
          '  → 거의 안 거르면 게이트로 못 쓴다(1단이 이미 고른 것이라 당연하기도 하다)')
if any('size' in i for i in ITEMS):
    import statistics as _st, random as _rd
    print('\n★크기 검증 — VLM이 말한 크기 vs **우리가 픽셀로 잰** 면적')
    for z in 'ABCD':
        a = sorted(i['area'] for i in ITEMS if i.get('size') == z)
        if a: print(f'   {z}  n={len(a):4d}  면적 중앙값 {_st.median(a):6.3f}%')

    # 🔴 칸별 중앙값 눈대중은 표본이 한쪽으로 쏠리면 못 읽는다.
    #    순위상관은 칸 개수·불균형에 안 휘둘린다. 그리고 늘 하던 대로 **순열 대조군**을 붙인다.
    _SR = {'A': 4, 'B': 3, 'C': 2, 'D': 1}
    _p = [(_SR[i['size']], i['area']) for i in ITEMS if i.get('size') in _SR]
    def _rank(xs):
        o = sorted(range(len(xs)), key=lambda k: xs[k]); r = [0.0] * len(xs); i = 0
        while i < len(o):                       # 동점은 평균 순위(크기는 동점이 많다)
            j = i
            while j + 1 < len(o) and xs[o[j + 1]] == xs[o[i]]: j += 1
            for k in range(i, j + 1): r[o[k]] = (i + j) / 2 + 1
            i = j + 1
        return r
    def _rho(a, b):
        ra, rb = _rank(a), _rank(b); n = len(a)
        ma, mb = sum(ra) / n, sum(rb) / n
        num = sum((x - ma) * (y - mb) for x, y in zip(ra, rb))
        den = (sum((x - ma) ** 2 for x in ra) * sum((y - mb) ** 2 for y in rb)) ** .5
        return num / den if den else 0.0
    _sz = [x for x, _ in _p]; _ar = [y for _, y in _p]
    _obs = _rho(_sz, _ar)
    _rg = _rd.Random(0)
    _null = []
    for _ in range(1000):
        _sh = _sz[:]; _rg.shuffle(_sh); _null.append(_rho(_sh, _ar))
    _pv = (sum(1 for x in _null if abs(x) >= abs(_obs)) + 1) / 1001
    print(f'   순위상관 rho = {_obs:+.3f}  (순열 1000회 p = {_pv:.3f}, 대조 |rho| p95 '
          f'{sorted(map(abs, _null))[949]:.3f})')
    print(f'   → {"✅ 모델이 실제로 크기를 보고 있다" if _obs > 0.2 and _pv < 0.01 else "🔴 우연과 구분 안 됨 — 베끼는 중"}'
          f'   (rho>0.2 & p<0.01 을 통과 기준으로 사전 고정)')
    # 실제로 쓰이는 칸이 몇 개인가 — 4단계를 준다고 4단계로 답하지 않는다
    _use = Counter(i['size'] for i in ITEMS)
    _dom = ' · '.join(f'{z} {100*_use[z]/max(len(ITEMS),1):.0f}%' for z in 'ABCD')
    print(f'   실제 사용 분포: {_dom}  → 유효 단계 '
          f'{sum(1 for z in "ABCD" if _use[z] >= len(ITEMS) * .1)}개')

In [ ]:
# == §G 2단 유형 정확도 — 핵심 단어가 이미지와 맞는가 (재추론 없음) ==
# 리포트 VLM이 따로 있으므로 2단이 내놓아야 할 것은 **문장이 아니라 맞는 핵심 단어**다.
# 문장 다양성은 안 본다. 여기서 답할 것 하나: **2단이 부른 종류가 사람이 찍은 유형과 맞는가.**
#
# 🔑 잴 수 있는 이유: §B 라벨의 점에는 유형(`t`)이 붙어 있다. 박스 안에 든 점의 유형 = 정답.
# 🔑 같은 잣대로 1단(OWLv2 태그)도 채점한다 — 2단이 1단보다 나은지 알아야 2단이 정당화된다.
# 🔴 순열 대조군: 종류를 박스끼리 섞어 채점. 배율 <2면 그 정확도는 우연이다(집 규칙).
import json, random
from collections import Counter, defaultdict
from PIL import Image

_c = sorted(OUT.glob('vlm_dump*.json'), key=lambda p: p.stat().st_mtime)
assert _c, '★§E를 먼저'
D = json.loads(_c[-1].read_text(encoding='utf-8'))
print(f'덤프 {_c[-1].name}\n')

# ── 박스마다 "안에 든 사람 점의 유형" = 정답 ──────────────────────────
PAIRS = []                       # (사람유형, 2단종류, 1단태그, 면적, 점수)
n_box = n_hit = 0
covered, total_pt = 0, 0
for name, v in D.items():
    pts = GT[name]['points']; total_pt += len(pts)
    if not pts:
        n_box += len(v['items']); continue
    W, H = Image.open(IMGDIR / name).size
    P = [(p['x'] * W, p['y'] * H, p['t']) for p in pts]
    used = set()
    for it in v['items']:
        n_box += 1
        b = it['bbox']
        ins = [(i, t) for i, (x, y, t) in enumerate(P) if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        n_hit += 1; used |= {i for i, _ in ins}
        PAIRS.append((Counter(t for _, t in ins).most_common(1)[0][0],
                      it.get('kind', '?'), it.get('tag', '?'), it['area'], it['score']))
    covered += len(used)

print(f'덤프 박스 {n_box}개 중 사람 점을 담은 것 {n_hit}개 ({n_hit/max(n_box,1):.1%})')
print(f'사람 점 {total_pt}개 중 덤프 박스가 덮은 것 {covered}개 ({covered/max(total_pt,1):.1%})')
print('  ※ 덤프는 thr0.12·장당 상위 6박스만 담는다 — 커버리지가 낮은 건 설계상 당연하다\n')
assert PAIRS, '★사람 점을 담은 박스가 0개 — 덤프의 임계/상한을 낮춰야 한다'

def acc(pred_i, label):
    ok = sum(1 for p in PAIRS if p[pred_i] == p[0])
    return ok / len(PAIRS), ok, label

# ── 대조군: 예측 유형을 박스끼리 섞는다 ────────────────────────────────
def ctrl(pred_i):
    v = [p[pred_i] for p in PAIRS]; r = random.Random(0)
    s = []
    for _ in range(200):
        sh = v[:]; r.shuffle(sh)
        s.append(sum(1 for a, b in zip(PAIRS, sh) if a[0] == b) / len(PAIRS))
    return sum(s) / len(s)

print(f'{"":22s} {"정확도":>7s} {"순열대조":>8s} {"배율":>6s}')
for i, lab in ((1, '2단 Qwen 종류'), (2, '1단 OWLv2 태그')):
    a, ok, _ = acc(i, lab); c = ctrl(i); r = a / max(c, 1e-9)
    print(f'{lab:22s} {a:7.3f} {c:8.3f} {r:6.2f}× '
          f'{"✅" if r >= 2 else "❌ 폐기(<2)"}   ({ok}/{len(PAIRS)})')

# ── 유형별 — 어느 단어가 맞고 어느 단어가 틀리나 ───────────────────────
print(f'\n{"사람 유형":14s} {"n":>4s} {"2단":>6s} {"1단":>6s}   2단이 대신 부른 이름')
for t, n in Counter(p[0] for p in PAIRS).most_common():
    sub = [p for p in PAIRS if p[0] == t]
    a2 = sum(1 for p in sub if p[1] == t) / n
    a1 = sum(1 for p in sub if p[2] == t) / n
    wrong = Counter(p[1] for p in sub if p[1] != t).most_common(3)
    print(f'{t:14s} {n:4d} {a2:6.3f} {a1:6.3f}   ' +
          ' · '.join(f'{k}×{c}' for k, c in wrong))

print('\n읽는 법')
print('  · 2단 배율 <2  → 종류는 우연 수준. 리포트 VLM에 넘길 핵심 단어로 못 쓴다')
print('  · 2단 < 1단    → 2단이 1단보다 못하다 = 종류는 1단 태그를 그대로 쓰는 게 낫다')
print('  · 특정 유형만 0 → 그 유형의 단어가 안 나온다. 프롬프트 선택지·질의어 문제')

In [ ]:
# == §H 유형이 픽셀에 있는가 — VLM 없이 (2~3분) ==
# ⚠️ 이 셀의 기각 판정은 무효다 — "순열 배율 ≥2"는 F1 스윕에 안 맞는 기준이다.
#    순열 F1의 바닥이 기저율 F1(~0.5)이라 배율 상한이 사실상 2.0 = 완벽해야 통과한다.
# 🔴 늘 하던 대로 순열 대조군을 붙인다 — 규칙을 여러 개 스윕하면 우연히 하나는 맞는다.
import json, random
import numpy as np
from collections import Counter
from PIL import Image

_c = sorted(OUT.glob('vlm_dump*.json'), key=lambda p: p.stat().st_mtime)
D = json.loads(_c[-1].read_text(encoding='utf-8'))
print(f'덤프 {_c[-1].name}\n')

# ── 박스마다 픽셀 통계 + 사람 유형 ─────────────────────────────────────
ROWS = []      # (사람유형, H, S, V, 대비, 가로세로비, 면적, 이긴질의, 2단종류)
for name, v in D.items():
    pts = GT[name]['points']
    if not pts or not v['items']: continue
    im = Image.open(IMGDIR / name).convert('RGB'); W, H_ = im.size
    hsv = np.asarray(im.convert('HSV'), np.float32)
    P = [(p['x'] * W, p['y'] * H_, p['t']) for p in pts]
    for it in v['items']:
        b = it['bbox']
        ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
        x1, y1 = min(W, int(b[2]) + 1), min(H_, int(b[3]) + 1)
        if x1 <= x0 or y1 <= y0: continue
        p = hsv[y0:y1, x0:x1].reshape(-1, 3)
        ROWS.append((Counter(ins).most_common(1)[0][0],
                     float(np.median(p[:, 0])), float(np.median(p[:, 1])),
                     float(np.median(p[:, 2])), float(p[:, 2].std()),
                     (x1 - x0) / max(y1 - y0, 1), it['area'],
                     it.get('query', '?'), it.get('kind', '?')))
assert ROWS, '★사람 점을 담은 박스가 0개'
print(f'사람 점을 담은 박스 {len(ROWS)}개\n')

# ── 유형별 픽셀 프로필 — 눈으로 먼저 본다 ──────────────────────────────
print(f'{"사람 유형":14s} {"n":>4s} {"색상H":>6s} {"채도S":>6s} {"밝기V":>6s} {"대비":>6s} {"가로/세로":>8s} {"면적%":>7s}')
for t, n in Counter(r[0] for r in ROWS).most_common():
    s = [r for r in ROWS if r[0] == t]
    med = lambda i: float(np.median([x[i] for x in s]))
    print(f'{t:14s} {n:4d} {med(1):6.1f} {med(2):6.1f} {med(3):6.1f} '
          f'{med(4):6.1f} {med(5):8.2f} {med(6):7.3f}')
print('  ※ H는 0~255 눈금. 녹·부식이면 주황~갈색(대략 5~35)에 채도가 높아야 한다')

# ── 한 유형 vs 나머지를 픽셀 규칙으로 가를 수 있나 (순열 대조 포함) ────
def sweep(target, feat, lo_grid, hi_grid, fname):
    y = [r[0] == target for r in ROWS]; x = [r[feat] for r in ROWS]
    npos = sum(y)
    if npos < 5: return None
    best = None
    for lo in lo_grid:
        for hi in hi_grid:
            if hi <= lo: continue
            pr = [lo <= v <= hi for v in x]
            tp = sum(1 for a, b in zip(pr, y) if a and b)
            fp = sum(1 for a, b in zip(pr, y) if a and not b)
            fn = sum(1 for a, b in zip(pr, y) if not a and b)
            P_, R_ = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
            f1 = 2 * P_ * R_ / max(P_ + R_, 1e-9)
            if best is None or f1 > best[0]: best = (f1, lo, hi, P_, R_)
    # 순열 대조: 정답을 섞고 같은 스윕을 돌린다 = "스윕이 우연히 뽑아내는 F1"
    rg = random.Random(0); null = []
    for _ in range(200):
        ys = y[:]; rg.shuffle(ys)
        b2 = 0.0
        for lo in lo_grid:
            for hi in hi_grid:
                if hi <= lo: continue
                pr = [lo <= v <= hi for v in x]
                tp = sum(1 for a, b in zip(pr, ys) if a and b)
                fp = sum(1 for a, b in zip(pr, ys) if a and not b)
                fn = sum(1 for a, b in zip(pr, ys) if not a and b)
                P_, R_ = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
                b2 = max(b2, 2 * P_ * R_ / max(P_ + R_, 1e-9))
        null.append(b2)
    ctrl = sum(null) / len(null)
    f1, lo, hi, P_, R_ = best
    r = f1 / max(ctrl, 1e-9)
    print(f'  {target:12s} vs 나머지 · {fname} {lo:.0f}~{hi:.0f}  '
          f'F1 {f1:.3f} (P {P_:.3f} R {R_:.3f}) · 순열 {ctrl:.3f} · 배율 {r:.2f}× '
          f'{"✅" if r >= 2 else "❌ 폐기(<2)"}   n={npos}')
    return r

print('\n■ 픽셀 규칙으로 유형을 가를 수 있나 (규칙 스윕 + 순열 대조)')
G255 = list(range(0, 256, 16))
for t in [k for k, n in Counter(r[0] for r in ROWS).most_common() if n >= 10]:
    sweep(t, 1, G255, G255, '색상H')
    sweep(t, 2, G255, G255, '채도S')

print('\n판정')
print('  · 배율 ≥2가 하나라도 있다 → **정보는 크롭에 있다.** 이름 붙이기(질의 정규화·프롬프트)를 고치면 된다')
print('  · 전부 <2                 → 크롭이 정보를 안 담고 있다. 크롭을 키우거나 원본 해상도로 다시 떠야 한다')

# ── 🔑 1단 질의어 진단 (공짜 — 덤프에 이긴 질의어가 그대로 있다) ────────
# OWLv2 점수는 **질의마다 스케일이 다르다** → 전역 argmax면 잘 반응하는 질의 하나가 다 먹는다.
# 여기서 그 질의를 이름으로 특정한다. 특정되면 고치는 방법은 질의별 정규화 하나뿐이다.
print('\n■ 이긴 질의어 분포 (사람 점을 담은 박스 %d개)' % len(ROWS))
_q = Counter(r[7] for r in ROWS)
for q, n in _q.most_common(10):
    sub = [r for r in ROWS if r[7] == q]
    top = Counter(r[0] for r in sub).most_common(2)
    print(f'  {q:34s} {n:4d} ({n/len(ROWS):5.1%})  실제: ' +
          ' · '.join(f'{k}×{c}' for k, c in top))
_dom = _q.most_common(1)[0]
print(f'  → 최다 질의 {_dom[0]!r}가 {_dom[1]/len(ROWS):.1%} 차지'
      + ('  🔴 독점 = 질의별 점수 정규화 필요(게이트에는 영향 0, 리포트만)'
         if _dom[1] / len(ROWS) > 0.35 else '  (독점 아님 — 다른 원인)'))

# ── 🔑 사람이 녹이라 한 곳을 1단은 어떤 질의로 잡았나 ──────────────────
print('\n■ 사람이 "녹·부식"이라 찍은 박스를 1단은 무슨 질의로 잡았나')
_r = [r for r in ROWS if r[0] == '녹·부식']
for q, n in Counter(x[7] for x in _r).most_common(6):
    print(f'  {q:34s} {n:4d} / {len(_r)}')
print('  → rust 계열 질의가 여기 상위에 있으면 **1단은 녹을 녹으로 보고 있는데 태그만 뺏긴 것**')
print('    = 정규화로 고쳐진다. rust가 아예 없으면 질의어 자체를 바꿔야 한다')


In [ ]:
# == §I 녹 질의어 교체 — 검출만 (VLM 없음, 후보당 약 55초) ==
# 🔴 채택 규칙을 실행 전에 고정한다 (결과 보고 바꾸지 않는다):
#    ① 녹 태그 정확도 **순열 배율 ≥ 2**       ← 못 넘으면 무조건 기각
#    ② 전체 점 recall 하락 ≤ 0.02             ← 검출력을 팔아서 이름을 사지 않는다
#    ③ 게이트 recall 1.000 유지               ← 확정 운영점을 깨지 않는다
#    셋 다 통과한 것 중 녹 태그 정확도 최고를 채택. 하나라도 어기면 baseline 유지.
#
# 선행: unsup §1(detect · QUERY_MAP · NEG_MAP · TAG_OF · QUERIES) · 이 노트북 §C(GT · EVAL_PATHS)
import time, random
from collections import Counter, defaultdict
from PIL import Image

THR_LOC, THR_GATE, N_GATE = 0.12, 0.08, 8
RUST = '녹·부식'

CANDS = {
    'baseline(구)':  ['rust', 'rusty metal', 'brown rust stain', 'corroded metal surface'],
    'A 색변화':      ['orange brown discoloration', 'reddish brown stain',
                      'brown spot on surface', 'yellowish brown patch'],
    'B 부식현상':    ['oxidation on surface', 'tarnished metal', 'discolored patch',
                      'brown corrosion spot'],
    'C 혼합':        ['rust', 'orange brown discoloration', 'brown stain on surface',
                      'oxidation on surface', 'reddish brown spot'],
}

_BK = (dict(QUERY_MAP), list(QUERIES), dict(TAG_OF))   # 어떤 경우에도 되돌린다
_SZ = {p.name: Image.open(p).size for p in EVAL_PATHS}

def _measure(preds):
    hit = defaultdict(int); tot = defaultdict(int)
    pairs = []                                   # (사람유형, 예측태그)
    for name, g in GT.items():
        if not g['points']: continue
        bx = [b for b in preds.get(name, []) if b[4] >= THR_LOC]
        W, H = _SZ[name]
        P = [(p['x'] * W, p['y'] * H, p['t']) for p in g['points']]
        for x, y, t in P:
            tot[t] += 1
            if any(b[0] <= x <= b[2] and b[1] <= y <= b[3] for b in bx): hit[t] += 1
        for b in bx:
            ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
            if ins: pairs.append((Counter(ins).most_common(1)[0][0], b[5]))

    def _acc(idx):
        """idx = 채점할 위치들. 순열은 **전체** 태그를 섞고 같은 위치만 본다"""
        if not idx: return 0.0, 0.0, 0
        a = sum(1 for i in idx if pairs[i][0] == pairs[i][1]) / len(idx)
        tags = [k for _, k in pairs]; r = random.Random(0); s = []
        for _ in range(200):
            sh = tags[:]; r.shuffle(sh)
            s.append(sum(1 for i in idx if pairs[i][0] == sh[i]) / len(idx))
        return a, sum(s) / len(s), len(idx)

    a_all, c_all, n_all = _acc(list(range(len(pairs))))
    a_r,  c_r,  n_r  = _acc([i for i, p in enumerate(pairs) if p[0] == RUST])

    tp = fp = fn = 0
    for name, g in GT.items():
        pred = sum(1 for b in preds.get(name, []) if b[4] >= THR_GATE) >= N_GATE
        truth = bool(g['points'])
        tp += pred and truth; fp += pred and not truth; fn += (not pred) and truth
    P_, R_ = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    T, Hh = sum(tot.values()), sum(hit.values())
    return {'pt_all': Hh / max(T, 1), 'pt_rust': hit[RUST] / max(tot[RUST], 1),
            'acc_rust': a_r, 'ctl_rust': c_r, 'n_rust': n_r,
            'acc_all': a_all, 'ctl_all': c_all, 'n_all': n_all,
            'gP': P_, 'gR': R_, 'gF': 2 * P_ * R_ / max(P_ + R_, 1e-9)}

RES = {}
try:
    for label, qs in CANDS.items():
        QUERY_MAP[RUST] = qs
        QUERIES = [q for v in QUERY_MAP.values() for q in v] + \
                  [q for v in NEG_MAP.values() for q in v]
        TAG_OF = {q: k for k, v in {**QUERY_MAP, **NEG_MAP}.items() for q in v}
        t0 = time.time()
        pr = {p.name: v for p, v in detect(EVAL_PATHS, thr=0.02, topk=50).items()}
        RES[label] = _measure(pr)
        print(f'  {label:16s} {time.time()-t0:5.0f}s', flush=True)
finally:
    QUERY_MAP.clear(); QUERY_MAP.update(_BK[0])
    QUERIES, TAG_OF = _BK[1], _BK[2]
    print('\n원본 QUERY_MAP 복원 완료 — 채택분은 §1에 직접 반영할 것\n')

print(f'{"질의셋":16s} {"녹태그":>7s} {"순열":>6s} {"배율":>6s} {"n":>4s} | '
      f'{"녹 점recall":>11s} {"전체 점recall":>13s} | {"게이트 P":>8s} {"R":>6s} {"F1":>6s}')
for k, r in RES.items():
    print(f'{k:16s} {r["acc_rust"]:7.3f} {r["ctl_rust"]:6.3f} '
          f'{r["acc_rust"]/max(r["ctl_rust"],1e-9):5.2f}× {r["n_rust"]:4d} | '
          f'{r["pt_rust"]:11.3f} {r["pt_all"]:13.3f} | '
          f'{r["gP"]:8.3f} {r["gR"]:6.3f} {r["gF"]:6.3f}')

# 전체 유형 태그 정확도 — 녹만 좋아지고 나머지가 나빠졌는지 본다(질의는 서로 경쟁한다)
print(f'\n{"질의셋":16s} {"전체태그":>8s} {"순열":>6s} {"배율":>6s} {"n":>5s}')
for k, r in RES.items():
    print(f'{k:16s} {r["acc_all"]:8.3f} {r["ctl_all"]:6.3f} '
          f'{r["acc_all"]/max(r["ctl_all"],1e-9):5.2f}× {r["n_all"]:5d}')

base = RES['baseline(구)']
print('\n■ 채택 판정 (규칙은 실행 전에 고정됨)')
win = None
for k, r in RES.items():
    if k.startswith('baseline'): continue
    c1 = r['acc_rust'] / max(r['ctl_rust'], 1e-9) >= 2.0
    c2 = base['pt_all'] - r['pt_all'] <= 0.02
    c3 = r['gR'] >= 1.0
    ok = c1 and c2 and c3
    print(f'  {k:16s} ①배율≥2 {"✅" if c1 else "❌"}  '
          f'②점recall {r["pt_all"]-base["pt_all"]:+.3f} {"✅" if c2 else "❌"}  '
          f'③게이트R {r["gR"]:.3f} {"✅" if c3 else "❌"}  → {"채택 후보" if ok else "기각"}')
    if ok and (win is None or r['acc_rust'] > RES[win]['acc_rust']): win = k
print(f'\n▶ {"채택: " + win if win else "전부 기각 — baseline 유지. 질의어로는 안 고쳐진다는 뜻이다."}')
if win:
    print(f"   unsup §1의 QUERY_MAP['{RUST}'] 를 이걸로 바꿀 것:\n   {CANDS[win]}")
    print('   ⚠️ §E 재실행은 필요 없다 — 게이트·점 recall은 위 표가 이미 확정했고,')
    print('      2단(Qwen)은 검출기 태그를 안 본다(v8 프롬프트에 힌트가 없다). 바뀌는 건 1단 태그뿐이다.')

In [ ]:
# == §J 나머지 유형 질의어 교체 — 검출만 (VLM 없음, 후보당 약 55초 · 총 12회 ≈ 11분) ==
# 🔴 유형끼리 **경쟁한다.** OWLv2는 박스마다 가장 잘 맞는 질의 하나를 고르므로,
#    한 유형의 질의를 바꾸면 다른 유형이 이기던 박스를 뺏거나 뺏긴다.
#    → **한 번에 하나만 바꾼다**(one-factor-at-a-time). 마지막에 승자를 모아 조합 검증한다.
#
# 🔴 채택 규칙을 실행 전에 고정 (결과 보고 바꾸지 않는다):
#    ① 대상 유형 태그 **순열 배율 ≥ 2**
#    ② 전체 점 recall 하락 ≤ 0.02          ← 검출력을 팔아서 이름을 사지 않는다
#    ③ 게이트 recall 1.000 유지            ← 확정 운영점을 깨지 않는다
#    ④ **전체 태그 정확도 하락 ≤ 0.01**     ← §J 신설. 남의 유형에서 뺏어 온 것을 채택하지 않는다
#    넷 다 통과한 것 중 대상 유형 정확도 최고를 채택. 하나라도 어기면 그 유형은 기존 유지.
#
# 선행: unsup §1(detect · QUERY_MAP · NEG_MAP) · 이 노트북 §C(GT · EVAL_PATHS)
import time, random
from collections import Counter, defaultdict
from PIL import Image

THR_LOC, THR_GATE, N_GATE = 0.12, 0.08, 8

CANDS = {
    '벗겨짐·박리': {
        'P1 가장자리': ['peeling film edge', 'wrapper lifted off surface',
                        'label peeling away', 'delaminated coating'],
        'P2 노출':     ['plastic wrap partially removed', 'bare surface where film came off',
                        'flap of loose film'],
    },
    '파손·찢김': {
        'T1 찢김':     ['torn plastic film', 'ripped wrapper', 'split in surface film'],
        'T2 관통':     ['hole through the wrapper', 'gash on surface', 'deep dent with crease'],
    },
    '긁힘·스크래치': {
        # scratch / scratch mark 가 아무 데나 붙는다(36+30개, 대부분 오답) → 선 형태를 명시
        'S1 선형태':   ['thin scratch line', 'linear scratch mark', 'fine abrasion streak'],
        'S2 축소':     ['scuff mark'],
    },
    '들뜸': {
        'L1 기포':     ['bubble trapped under film', 'raised blister on surface'],
        'L2 주름':     ['wrinkle bulge in wrapper', 'film not adhering to surface',
                        'puffed up area'],
    },
    '오염·이물질': {
        'D1 이물':     ['dust particle on surface', 'contamination residue', 'stain patch'],
        'D2 얼룩':     ['dirty smudge', 'grey deposit on surface', 'foreign speck'],
    },
}

_BK = (dict(QUERY_MAP), list(QUERIES), dict(TAG_OF))     # 어떤 경우에도 되돌린다
_SZ = {p.name: Image.open(p).size for p in EVAL_PATHS}
TYPES = list(CANDS)

def _rebuild():
    global QUERIES, TAG_OF
    QUERIES = [q for v in QUERY_MAP.values() for q in v] + \
              [q for v in NEG_MAP.values() for q in v]
    TAG_OF = {q: k for k, v in {**QUERY_MAP, **NEG_MAP}.items() for q in v}

def _run():
    return {p.name: v for p, v in detect(EVAL_PATHS, thr=0.02, topk=50).items()}

def _stats(preds):
    hit = defaultdict(int); tot = defaultdict(int); pairs = []
    for name, g in GT.items():
        if not g['points']: continue
        bx = [b for b in preds.get(name, []) if b[4] >= THR_LOC]
        W, H = _SZ[name]
        P = [(p['x'] * W, p['y'] * H, p['t']) for p in g['points']]
        for x, y, t in P:
            tot[t] += 1
            if any(b[0] <= x <= b[2] and b[1] <= y <= b[3] for b in bx): hit[t] += 1
        for b in bx:
            ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
            if ins: pairs.append((Counter(ins).most_common(1)[0][0], b[5]))

    tags = [k for _, k in pairs]; rng = random.Random(0)
    SHUF = []
    for _ in range(200):
        sh = tags[:]; rng.shuffle(sh); SHUF.append(sh)
    def _acc(idx):
        if not idx: return (0.0, 0.0, 0)
        a = sum(1 for i in idx if pairs[i][0] == pairs[i][1]) / len(idx)
        c = sum(sum(1 for i in idx if pairs[i][0] == sh[i]) / len(idx) for sh in SHUF) / len(SHUF)
        return (a, c, len(idx))

    acc = {t: _acc([i for i, p in enumerate(pairs) if p[0] == t]) for t in tot}
    tp = fp = fn = 0
    for name, g in GT.items():
        pred = sum(1 for b in preds.get(name, []) if b[4] >= THR_GATE) >= N_GATE
        truth = bool(g['points'])
        tp += pred and truth; fp += pred and not truth; fn += (not pred) and truth
    P_, R_ = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    T, Hh = sum(tot.values()), sum(hit.values())
    return {'pt_all': Hh / max(T, 1),
            'pt': {t: hit[t] / max(tot[t], 1) for t in tot},
            'acc': acc, 'acc_all': _acc(list(range(len(pairs)))),
            'gP': P_, 'gR': R_, 'gF': 2 * P_ * R_ / max(P_ + R_, 1e-9)}

def _rat(x): return x[0] / max(x[1], 1e-9)

RES = {}
t_all = time.time()
try:
    _rebuild(); RES['baseline'] = _stats(_run())
    print(f'  baseline {time.time()-t_all:5.0f}s', flush=True)
    for tgt in TYPES:
        for lab, qs in CANDS[tgt].items():
            QUERY_MAP.clear(); QUERY_MAP.update(_BK[0])          # 한 번에 하나만 바꾼다
            QUERY_MAP[tgt] = qs; _rebuild()
            t0 = time.time(); RES[(tgt, lab)] = _stats(_run())
            print(f'  {tgt:8s} {lab:10s} {time.time()-t0:5.0f}s', flush=True)
finally:
    QUERY_MAP.clear(); QUERY_MAP.update(_BK[0]); _rebuild()

B = RES['baseline']
print(f'\n총 {time.time()-t_all:.0f}s\n')
# 🔴 n을 반드시 같이 찍는다 — 실측에서 n을 뺐더니 사람 점이 1~3개뿐인 유형의
#    태그 1.000 / 0.000이 신호처럼 보였다(파손·들뜸). n<10이면 그 줄은 노이즈다.
print(f'{"유형":12s} {"후보":11s} {"n":>4s} {"태그":>6s} {"순열":>6s} {"배율":>6s} | '
      f'{"전체태그":>8s} {"배율":>6s} | {"점recall":>8s} | {"게이트R":>7s}  판정')
print(f'{"(기존)":12s} {"":11s} {"":>6s} {"":>6s} {"":>6s} | '
      f'{B["acc_all"][0]:8.3f} {_rat(B["acc_all"]):5.2f}× | {B["pt_all"]:8.3f} | {B["gR"]:7.3f}')

WIN = {}
for tgt in TYPES:
    b = B['acc'].get(tgt, (0, 0, 0))
    print(f'{tgt:12s} {"기존":11s} {b[2]:4d} {b[0]:6.3f} {b[1]:6.3f} {_rat(b):5.2f}× | '
          f'{"":8s} {"":6s} | {"":8s} | {"":7s}' + ('   ⚠️ n<10 = 노이즈' if b[2] < 10 else ''))
    for lab in CANDS[tgt]:
        r = RES[(tgt, lab)]; a = r['acc'].get(tgt, (0, 0, 0))
        c1 = _rat(a) >= 2.0
        c2 = B['pt_all'] - r['pt_all'] <= 0.02
        c3 = r['gR'] >= 1.0
        c4 = B['acc_all'][0] - r['acc_all'][0] <= 0.01
        ok = c1 and c2 and c3 and c4
        flag = ''.join(('①' if c1 else '❌①', '②' if c2 else '❌②',
                        '③' if c3 else '❌③', '④' if c4 else '❌④'))
        print(f'{"":12s} {lab:11s} {a[2]:4d} {a[0]:6.3f} {a[1]:6.3f} {_rat(a):5.2f}× | '
              f'{r["acc_all"][0]:8.3f} {_rat(r["acc_all"]):5.2f}× | {r["pt_all"]:8.3f} | '
              f'{r["gR"]:7.3f}  {"채택후보" if ok else "기각"} {flag}')
        if ok and (tgt not in WIN or a[0] > RES[(tgt, WIN[tgt])]['acc'][tgt][0]):
            WIN[tgt] = lab

print(f'\n■ 개별 승자: {WIN if WIN else "없음 — 전부 기각, 기존 유지"}')

# ── 조합 검증: 승자를 한꺼번에 적용하면 서로 간섭할 수 있다 ────────────
if WIN:
    try:
        QUERY_MAP.clear(); QUERY_MAP.update(_BK[0])
        for t, lab in WIN.items(): QUERY_MAP[t] = CANDS[t][lab]
        _rebuild()
        t0 = time.time(); COMB = _stats(_run())
        print(f'  조합 {time.time()-t0:5.0f}s')
    finally:
        QUERY_MAP.clear(); QUERY_MAP.update(_BK[0]); _rebuild()
        print('\n원본 QUERY_MAP 복원 완료 — 채택분은 §1에 직접 반영할 것\n')

    print(f'{"":12s} {"전체태그":>8s} {"순열":>6s} {"배율":>6s} | {"점recall":>8s} | '
          f'{"게이트 P":>8s} {"R":>6s} {"F1":>6s}')
    print(f'{"기존":12s} {B["acc_all"][0]:8.3f} {B["acc_all"][1]:6.3f} '
          f'{_rat(B["acc_all"]):5.2f}× | {B["pt_all"]:8.3f} | '
          f'{B["gP"]:8.3f} {B["gR"]:6.3f} {B["gF"]:6.3f}')
    print(f'{"조합":12s} {COMB["acc_all"][0]:8.3f} {COMB["acc_all"][1]:6.3f} '
          f'{_rat(COMB["acc_all"]):5.2f}× | {COMB["pt_all"]:8.3f} | '
          f'{COMB["gP"]:8.3f} {COMB["gR"]:6.3f} {COMB["gF"]:6.3f}')

    print(f'\n{"유형":12s} {"기존":>7s} → {"조합":>7s}   (개별 실험값)')
    for t in TYPES:
        b = B['acc'].get(t, (0, 0, 0)); c = COMB['acc'].get(t, (0, 0, 0))
        solo = RES[(t, WIN[t])]['acc'][t][0] if t in WIN else None
        print(f'{t:12s} {b[0]:7.3f} → {c[0]:7.3f}   ' +
              (f'(단독 {solo:.3f})' + ('  ⚠️ 조합에서 하락 = 유형 간 간섭'
                                       if solo is not None and c[0] < solo - 0.02 else '')
               if solo is not None else '(미교체)'))

    ok = (_rat(COMB['acc_all']) >= 2.0 and B['pt_all'] - COMB['pt_all'] <= 0.02
          and COMB['gR'] >= 1.0 and COMB['acc_all'][0] >= B['acc_all'][0])
    print(f'\n▶ 조합 판정: {"✅ 채택 — §1의 QUERY_MAP을 아래로 교체" if ok else "🔴 조합은 기각"}')
    if ok:
        for t, lab in WIN.items():
            print(f"   QUERY_MAP['{t}'] = {CANDS[t][lab]}")
    else:
        print('   전체 태그 배율이 2 미만이거나 다른 조건을 어겼다. 개별 승자만 하나씩 넣어 재검증할 것.')
        print('   (유형 간 간섭이 있으면 개별로는 통과해도 조합에서 깨진다 — 위 표에서 확인)')

In [ ]:
# == §K 점 recall 후보 검증 — 순열 대조 + 박스 밀도 (VLM 없음, 후보당 약 53초) ==
# 🔴 채택 규칙 사전 고정 (결과 보고 바꾸지 않는다):
#    ① 점 recall 순열 배율 ≥ 2                     ← 집 규칙
#    ② 점 recall이 baseline보다 높을 것
#    ③ 게이트 recall 1.000 유지 (thr0.08 · N≥8)     ← 확정 운영점을 깨지 않는다
#    ④ 결함/무결함 박스 비대칭이 baseline 이상        ← "그냥 많이 뿌린 것"을 배제
#
# 선행: unsup §1(detect · QUERY_MAP · NEG_MAP) · 이 노트북 §C(GT · EVAL_PATHS)
import time, random
from collections import defaultdict
from PIL import Image

THR_LOC, THR_GATE, N_GATE = 0.12, 0.08, 8

# (대상 유형, 질의 리스트). None이면 현재 §1 그대로
CHECK = {
    'baseline':        None,
    'P1 벗겨짐-가장자리': ('벗겨짐·박리', ['peeling film edge', 'wrapper lifted off surface',
                                          'label peeling away', 'delaminated coating']),
    'P2 벗겨짐-노출':    ('벗겨짐·박리', ['plastic wrap partially removed',
                                          'bare surface where film came off', 'flap of loose film']),
    'L2 들뜸-주름':      ('들뜸', ['wrinkle bulge in wrapper', 'film not adhering to surface',
                                   'puffed up area']),
}

_BK = (dict(QUERY_MAP), list(QUERIES), dict(TAG_OF))
_SZ = {p.name: Image.open(p).size for p in EVAL_PATHS}
DEF = [k for k, g in GT.items() if g['points']]
CLN = [k for k, g in GT.items() if not g['points']]

def _rebuild():
    global QUERIES, TAG_OF
    QUERIES = [q for v in QUERY_MAP.values() for q in v] + \
              [q for v in NEG_MAP.values() for q in v]
    TAG_OF = {q: k for k, v in {**QUERY_MAP, **NEG_MAP}.items() for q in v}

def _pt_recall(preds):
    hit = tot = 0
    for name, g in GT.items():
        if not g['points']: continue
        bx = [b for b in preds.get(name, []) if b[4] >= THR_LOC]
        W, H = _SZ[name]
        for p in g['points']:
            x, y = p['x'] * W, p['y'] * H
            tot += 1
            hit += any(b[0] <= x <= b[2] and b[1] <= y <= b[3] for b in bx)
    return hit / max(tot, 1)

def _stats(preds):
    real = _pt_recall(preds)
    # 순열 대조 — 이름을 섞어 남의 이미지 박스로 채점한다
    ks = list(preds); pm = ks[:]; random.Random(0).shuffle(pm)
    ctrl = _pt_recall({a: preds[b] for a, b in zip(ks, pm)})
    d = sum(sum(1 for b in preds.get(k, []) if b[4] >= THR_LOC) for k in DEF) / max(len(DEF), 1)
    c = sum(sum(1 for b in preds.get(k, []) if b[4] >= THR_LOC) for k in CLN) / max(len(CLN), 1)
    ar = [max(b[2] - b[0], 0) * max(b[3] - b[1], 0)
          for v in preds.values() for b in v if b[4] >= THR_LOC]
    ar.sort()
    out = {'pt': real, 'ctl': ctrl, 'rat': real / max(ctrl, 1e-9),
           'bx_def': d, 'bx_cln': c, 'asym': d / max(c, 1e-9),
           'bx_all': sum(len(v) for v in preds.values()) / max(len(preds), 1),
           # 🔴 박스 **면적** 중앙값. 개수가 같아도 면적이 커지면 순열 recall이 뛴다
           'area': (ar[len(ar) // 2] if ar else 0),
           'a90': (ar[int(len(ar) * .9)] if ar else 0)}
    for N in (5, 8, 12):
        tp = fp = fn = 0
        for name, g in GT.items():
            pred = sum(1 for b in preds.get(name, []) if b[4] >= THR_GATE) >= N
            truth = bool(g['points'])
            tp += pred and truth; fp += pred and not truth; fn += (not pred) and truth
        P_, R_ = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        out[f'g{N}'] = (P_, R_, 2 * P_ * R_ / max(P_ + R_, 1e-9))
    return out

RES = {}
t_all = time.time()
try:
    for lab, ch in CHECK.items():
        QUERY_MAP.clear(); QUERY_MAP.update(_BK[0])
        if ch: QUERY_MAP[ch[0]] = ch[1]
        _rebuild()
        t0 = time.time()
        RES[lab] = _stats({p.name: v for p, v in detect(EVAL_PATHS, thr=0.02, topk=50).items()})
        print(f'  {lab:18s} {time.time()-t0:5.0f}s', flush=True)
finally:
    QUERY_MAP.clear(); QUERY_MAP.update(_BK[0]); _rebuild()
    print(f'\n원본 QUERY_MAP 복원 · 총 {time.time()-t_all:.0f}s\n')

print(f'{"후보":18s} {"점recall":>8s} {"순열":>6s} {"배율":>6s} | '
      f'{"결함박스/장":>10s} {"정상박스/장":>10s} {"비대칭":>6s} | {"장당전체":>8s} {"면적중앙":>8s} {"면적p90":>8s}')
for k, r in RES.items():
    print(f'{k:18s} {r["pt"]:8.3f} {r["ctl"]:6.3f} {r["rat"]:5.2f}× | '
          f'{r["bx_def"]:10.1f} {r["bx_cln"]:10.2f} {r["asym"]:5.2f}× | {r["bx_all"]:8.1f} {r["area"]:8.0f} {r["a90"]:8.0f}')

print(f'\n{"후보":18s} ' + ' '.join(f'{"N≥%d P/R/F1" % N:>20s}' for N in (5, 8, 12)))
for k, r in RES.items():
    print(f'{k:18s} ' + ' '.join(
        f'{r[f"g{N}"][0]:6.3f}/{r[f"g{N}"][1]:.3f}/{r[f"g{N}"][2]:.3f}'.rjust(20) for N in (5, 8, 12)))

B = RES['baseline']
print('\n■ 채택 판정 (규칙은 실행 전에 고정됨)')
win = None
for k, r in RES.items():
    if k == 'baseline': continue
    c1 = r['rat'] >= 2.0
    c2 = r['pt'] > B['pt']
    c3 = r['g8'][1] >= 1.0
    c4 = r['asym'] >= B['asym']
    ok = c1 and c2 and c3 and c4
    print(f'  {k:18s} ①배율 {r["rat"]:.2f}× {"✅" if c1 else "❌"}  '
          f'②점recall {r["pt"]-B["pt"]:+.3f} {"✅" if c2 else "❌"}  '
          f'③게이트R {r["g8"][1]:.3f} {"✅" if c3 else "❌"}  '
          f'④비대칭 {r["asym"]:.2f}× vs {B["asym"]:.2f}× {"✅" if c4 else "❌"}  '
          f'→ {"채택 후보" if ok else "기각"}')
    if ok and (win is None or r['pt'] > RES[win]['pt']): win = k

print(f'\n▶ {"채택: " + win if win else "전부 기각 — 기존 유지"}')
if win:
    t, qs = CHECK[win]
    print(f"   unsup §1: QUERY_MAP['{t}'] = {qs}")
    print(f"   점 recall {B['pt']:.3f} → {RES[win]['pt']:.3f}  (발표 헤드라인 갱신 대상)")
    print('   ⚠️ 이름(태그) 정확도는 §J에서 이미 기각된 후보다 — **검출 개선으로만** 채택하는 것이다.')
    print('      발표에는 "질의어 교체로 검출률 상승"이라 쓰고, 유형 분류는 여전히 미해결로 병기할 것.')
else:
    # 🔴 왜 기각인지는 위 ①~④ 표를 보고 말한다. 원인을 단정하지 않는다
    print('   위 ①~④ 중 ❌가 붙은 조건이 이유다.')
    print('   ①이 ❌면 = 박스를 더 뿌린 것(0804 §20.1 예산 모드 recall 0.738 / 배율 1.38×와 같은 함정).')
    print('   ②가 ❌면 = 애초에 안 올랐다. ③④가 ❌면 = 게이트나 정상/결함 분리를 팔았다.')
    print('   어느 경우든 헤드라인 점 recall은 0.818 유지.')

In [ ]:
# == §H2 유형이 색에 있는가 — 중앙값이 아니라 **화소 비율** (VLM·검출 없음, 약 2분) ==
# 🔴 판정 기준 (사전 고정) = **순열 p < 0.01 이면서 F1이 순열 p95보다 클 것.**
#    "배율 ≥2"를 안 쓰는 이유: F1 스윕의 순열 바닥은 0이 아니라 기저율 F1(~0.5)이라
#    배율 상한이 사실상 2.0이다 = 완벽한 분류기만 겨우 통과한다.
#    §F의 크기 검증(rho + 순열 p)에서 이미 쓰던 올바른 방식이다.
#
# 🔑 색상 구간을 내가 정하지 않는다. 12구간 히스토그램을 **먼저 보여주고**,
#    모든 연속 구간 × 임계를 스윕한 뒤 순열 대조로 거른다. "녹은 갈색일 것"이라는 내 가정을 안 넣는다.
# 🔴 스윕은 우연히 맞는다(§H에서 순열이 0.501을 냈다) → 순열도 **같은 스윕**을 돌려야 공정하다.
# 🔴 채도 가중: 무채색(회색) 화소는 색상값이 무의미하다. S로 가중해 표를 오염시키지 않는다.
#
# 선행: 이 노트북 §C(GT · IMGDIR) + §E 산출물(vlm_dump_*.json). §1·§4 불필요.
import json, random
import numpy as np
from collections import Counter
from PIL import Image

NB, SMIN = 12, 40          # 색상 12구간 · 채도 이 미만은 무채색으로 보고 가중 0
_c = sorted(OUT.glob('vlm_dump*.json'), key=lambda p: p.stat().st_mtime)
D = json.loads(_c[-1].read_text(encoding='utf-8'))
print(f'덤프 {_c[-1].name}\n')

LAB, HIST, EXTRA = [], [], []
for name, v in D.items():
    pts = GT[name]['points']
    if not pts or not v['items']: continue
    im = Image.open(IMGDIR / name).convert('RGB')
    hsv = np.asarray(im.convert('HSV'), np.float32); W, H_ = im.size
    P = [(p['x'] * W, p['y'] * H_, p['t']) for p in pts]
    for it in v['items']:
        b = it['bbox']
        ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
        x1, y1 = min(W, int(b[2]) + 1), min(H_, int(b[3]) + 1)
        if x1 <= x0 or y1 <= y0: continue
        p = hsv[y0:y1, x0:x1].reshape(-1, 3)
        w = (p[:, 1] >= SMIN).astype(np.float32)           # 채도 낮은 화소는 투표권 없음
        h = np.histogram(p[:, 0], bins=NB, range=(0, 256), weights=w)[0]
        HIST.append(h / max(w.sum(), 1e-6))                 # 유채색 화소 중 구간별 비율
        LAB.append(Counter(ins).most_common(1)[0][0])
        EXTRA.append((w.mean(), float(p[:, 2].std()), it['area']))
HIST = np.array(HIST); EXTRA = np.array(EXTRA)
LAB = np.array(LAB, dtype=object)
print(f'사람 점을 담은 박스 {len(LAB)}개 · 색상 {NB}구간 (채도<{SMIN} 화소는 제외)\n')

# ── ① 유형별 평균 히스토그램 — 먼저 눈으로 본다 ────────────────────────
BIN = [f'{int(i*256/NB):3d}-{int((i+1)*256/NB):3d}' for i in range(NB)]
DEG = [f'{int(i*360/NB):3d}°' for i in range(NB)]
print('유형별 평균 색상 분포 (채도 가중, 행 합 = 1.0)')
print(f'{"구간(0-255)":14s} ' + ' '.join(f'{b.split("-")[0]:>4s}' for b in BIN))
print(f'{"(각도)":14s} ' + ' '.join(f'{d:>4s}' for d in DEG))
ORDER = [t for t, n in Counter(LAB).most_common()]
for t in ORDER:
    m = HIST[LAB == t].mean(0)
    print(f'{t:12s}{int((LAB==t).sum()):>3d} ' + ' '.join(f'{x:4.2f}' for x in m))
print(f'{"유채색비율":12s}    ' + ' '.join('' for _ in BIN))
for t in ORDER:
    e = EXTRA[LAB == t]
    print(f'  {t:12s} 유채색 화소 {e[:,0].mean():.2f} · 대비 {e[:,1].mean():5.1f} · 면적 {e[:,2].mean():.2f}%')
print('\n  ※ 주황~갈색은 0-255 눈금에서 대략 5~35(=7°~50°). 녹이 거기 몰리면 색으로 잡힌다\n')

# ── ② 모든 연속 구간 × 임계 스윕 + 순열 대조 ───────────────────────────
WINS = [(i, j) for i in range(NB) for j in range(i, NB)]
FRAC = np.stack([HIST[:, i:j + 1].sum(1) for i, j in WINS], 1)     # (박스, 구간조합)
THRS = np.array([0.02, 0.05, 0.10, 0.20, 0.35, 0.50])

def best_f1(y):
    """모든 (구간, 임계) 중 최고 F1과 그 설정"""
    npos = y.sum()
    if npos == 0: return 0.0, None
    bf, bs = 0.0, None
    for t in THRS:
        pr = FRAC >= t                                    # (박스, 조합)
        tp = pr[y].sum(0); fp = pr[~y].sum(0); fn = npos - tp
        f1 = np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
        k = int(f1.argmax())
        if f1[k] > bf: bf, bs = float(f1[k]), (WINS[k], float(t),
                                               float(tp[k] / max(tp[k] + fp[k], 1)),
                                               float(tp[k] / max(npos, 1)))
    return bf, bs

print('■ "그 색 화소 비율"로 유형을 가를 수 있나 (전 구간 스윕 + 순열 대조)')
rng = random.Random(0)
for t in ORDER:
    y = (LAB == t)
    if y.sum() < 10:
        print(f'  {t:12s} n={int(y.sum()):3d}  ⚠️ n<10 = 노이즈, 건너뜀'); continue
    f1, s = best_f1(y)
    null = sorted(best_f1(np.random.RandomState(i).permutation(y))[0] for i in range(200))
    p95 = null[int(.95 * len(null))]
    pv = (sum(1 for x in null if x >= f1) + 1) / (len(null) + 1)
    ok = pv < 0.01 and f1 > p95
    (i, j), th, P_, R_ = s
    print(f'  {t:12s} n={int(y.sum()):3d}  색상 {int(i*256/NB)}~{int((j+1)*256/NB)} 화소 ≥{th:.0%}  '
          f'F1 {f1:.3f} (P {P_:.3f} R {R_:.3f}) · 순열 평균 {np.mean(null):.3f} p95 {p95:.3f} · '
          f'p={pv:.3f} {"✅" if ok else "❌ 폐기"}')

print('\n판정')
print('  · ✅가 하나라도 → **색 정보는 크롭에 있다.** §H의 "중앙값" 판정이 틀렸던 것이고,')
print('    이름 붙이기는 색 규칙으로 보정 가능하다(VLM·검출 재학습 없이)')
print('  · 전부 ❌ → 절대 색 비율로는 안 갈린다. 다음은 §H3(주변 대비)')

# 🔴 one-vs-rest의 함정: 통과했다고 그 유형을 **식별**한다는 뜻이 아니다.
#    '다른 유형을 배제하는 것'만으로도 F1이 오른다(합성 검증에서 신호 0인 유형이 통과했다).
print('\n⚠️ 통과 = 그 유형을 이름 붙일 수 있다는 뜻이 아니다. one-vs-rest라서')
print('   \'다른 유형을 배제하는 것\'만으로도 F1이 오른다.')
print('   통과분은 위 P/R을 볼 것 — P가 낮으면 배제 신호지 식별 신호가 아니다.')


In [ ]:
# == §L P1 노선 재도전 — 어느 질의가 박스를 키우나 (검출만, 후보당 약 53초 · 총 6회 ≈ 6분) ==
# 🔑 박스 면적 중앙값을 같이 찍는다. §K에서 배운 것: 개수가 아니라 **면적**으로 뿌릴 수 있다.
#
# 🔴 채택 규칙 사전 고정 (§K와 동일, 바꾸지 않는다):
#    ① 점 recall 순열 배율 ≥ 2   ② 점 recall > baseline
#    ③ 게이트 recall 1.000 유지   ④ 결함/무결함 비대칭 ≥ baseline
#    통과분을 모아 마지막에 조합 검증. 조합도 같은 4조건을 통과해야 채택.
#
# 선행: unsup §1(detect · QUERY_MAP · NEG_MAP) · 이 노트북 §C(GT · EVAL_PATHS)
import time, random
from collections import defaultdict
from PIL import Image

THR_LOC, THR_GATE, N_GATE = 0.12, 0.08, 8
TGT = '벗겨짐·박리'
P1 = ['peeling film edge', 'wrapper lifted off surface', 'label peeling away', 'delaminated coating']

_BK = (dict(QUERY_MAP), list(QUERIES), dict(TAG_OF))
_BASEQ = list(_BK[0][TGT])
_SZ = {p.name: Image.open(p).size for p in EVAL_PATHS}
DEF = [k for k, g in GT.items() if g['points']]
CLN = [k for k, g in GT.items() if not g['points']]

RUNS = {'baseline': _BASEQ}
for q in P1: RUNS[f'+ {q}'] = _BASEQ + [q]
RUNS['+ P1 전체'] = _BASEQ + P1

def _rebuild():
    global QUERIES, TAG_OF
    QUERIES = [q for v in QUERY_MAP.values() for q in v] + \
              [q for v in NEG_MAP.values() for q in v]
    TAG_OF = {q: k for k, v in {**QUERY_MAP, **NEG_MAP}.items() for q in v}

def _pt(preds):
    hit = tot = 0
    for name, g in GT.items():
        if not g['points']: continue
        bx = [b for b in preds.get(name, []) if b[4] >= THR_LOC]
        W, H = _SZ[name]
        for p in g['points']:
            x, y = p['x'] * W, p['y'] * H
            tot += 1
            hit += any(b[0] <= x <= b[2] and b[1] <= y <= b[3] for b in bx)
    return hit / max(tot, 1)

def _stats(preds):
    real = _pt(preds)
    ks = list(preds); pm = ks[:]; random.Random(0).shuffle(pm)
    ctrl = _pt({a: preds[b] for a, b in zip(ks, pm)})
    d = sum(sum(1 for b in preds.get(k, []) if b[4] >= THR_LOC) for k in DEF) / max(len(DEF), 1)
    c = sum(sum(1 for b in preds.get(k, []) if b[4] >= THR_LOC) for k in CLN) / max(len(CLN), 1)
    ar = sorted(max(b[2] - b[0], 0) * max(b[3] - b[1], 0)
                for v in preds.values() for b in v if b[4] >= THR_LOC)
    tp = fp = fn = 0
    for name, g in GT.items():
        pred = sum(1 for b in preds.get(name, []) if b[4] >= THR_GATE) >= N_GATE
        truth = bool(g['points'])
        tp += pred and truth; fp += pred and not truth; fn += (not pred) and truth
    P_, R_ = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {'pt': real, 'ctl': ctrl, 'rat': real / max(ctrl, 1e-9),
            'bd': d, 'bc': c, 'asym': d / max(c, 1e-9),
            'area': (ar[len(ar) // 2] if ar else 0),
            'a90': (ar[int(len(ar) * .9)] if ar else 0), 'n': len(ar),
            'gP': P_, 'gR': R_, 'gF': 2 * P_ * R_ / max(P_ + R_, 1e-9)}

def _go(qs):
    QUERY_MAP.clear(); QUERY_MAP.update(_BK[0]); QUERY_MAP[TGT] = qs; _rebuild()
    return _stats({p.name: v for p, v in detect(EVAL_PATHS, thr=0.02, topk=50).items()})

RES = {}
t_all = time.time()
try:
    for lab, qs in RUNS.items():
        t0 = time.time(); RES[lab] = _go(qs)
        print(f'  {lab:38s} {time.time()-t0:5.0f}s', flush=True)
finally:
    QUERY_MAP.clear(); QUERY_MAP.update(_BK[0]); _rebuild()

B = RES['baseline']
print(f'\n총 {time.time()-t_all:.0f}s\n')
print(f'{"질의":38s} {"점recall":>8s} {"순열":>6s} {"배율":>6s} | '
      f'{"면적중앙":>8s} {"면적p90":>8s} {"박스수":>6s} | {"비대칭":>6s} | {"게이트R":>7s}')
for k, r in RES.items():
    print(f'{k:38s} {r["pt"]:8.3f} {r["ctl"]:6.3f} {r["rat"]:5.2f}× | '
          f'{r["area"]:8.0f} {r["a90"]:8.0f} {r["n"]:6d} | {r["asym"]:5.2f}× | {r["gR"]:7.3f}')

print('\n■ 채택 판정 (규칙은 §K와 동일, 실행 전에 고정됨)')
WIN = []
for k, r in RES.items():
    if k == 'baseline': continue
    c1 = r['rat'] >= 2.0; c2 = r['pt'] > B['pt']
    c3 = r['gR'] >= 1.0;  c4 = r['asym'] >= B['asym']
    ok = c1 and c2 and c3 and c4
    print(f'  {k:38s} ①{r["rat"]:.2f}× {"✅" if c1 else "❌"} '
          f'②{r["pt"]-B["pt"]:+.3f} {"✅" if c2 else "❌"} '
          f'③{r["gR"]:.3f} {"✅" if c3 else "❌"} '
          f'④{r["asym"]:.2f}vs{B["asym"]:.2f} {"✅" if c4 else "❌"} → '
          f'{"채택 후보" if ok else "기각"}')
    if ok and k != '+ P1 전체': WIN.append(k)

# ── 통과한 단일 질의를 모아 조합 검증 ─────────────────────────────────
if len(WIN) >= 2:
    qs = _BASEQ + [k[2:] for k in WIN]
    try:
        print(f'\n조합 검증: {[k[2:] for k in WIN]}')
        t0 = time.time(); C = _go(qs); print(f'  조합 {time.time()-t0:5.0f}s')
    finally:
        QUERY_MAP.clear(); QUERY_MAP.update(_BK[0]); _rebuild()
    print(f'{"조합":38s} {C["pt"]:8.3f} {C["ctl"]:6.3f} {C["rat"]:5.2f}× | '
          f'{C["area"]:8.0f} {C["a90"]:8.0f} {C["n"]:6d} | {C["asym"]:5.2f}× | {C["gR"]:7.3f}')
    ok = (C['rat'] >= 2.0 and C['pt'] > B['pt'] and C['gR'] >= 1.0 and C['asym'] >= B['asym'])
    print(f'\n▶ 조합 {"✅ 채택" if ok else "🔴 기각 — 단일로는 통과해도 합치면 깨진다(질의 간 간섭)"}')
    if ok: print(f"   unsup §1: QUERY_MAP['{TGT}'] = {qs}")
elif len(WIN) == 1:
    k = WIN[0]
    print(f"\n▶ 채택: {k}\n   unsup §1: QUERY_MAP['{TGT}'] = {_BASEQ + [k[2:]]}")
    print(f"   점 recall {B['pt']:.3f} → {RES[k]['pt']:.3f}  (발표 헤드라인 갱신 대상)")
else:
    print('\n▶ 전부 기각 — baseline 유지. 헤드라인 점 recall 0.818 그대로.')
    print('   위 표의 **면적 중앙/p90**을 볼 것: p90만 커졌으면 "몇 개만 크게" 뿌린 것이다(중앙값은 못 잡는다).')
    print('   면적이 안 커졌는데도 배율이 떨어졌다면 원인은 다른 데 있다(질의 간 경쟁 등).')

In [ ]:
# == §H3 색을 **주변 대비**로 본다 — 박스 안 − 바깥 고리 (VLM·검출 없음, 약 2분) ==
# 🔴 판정 기준 (사전 고정) = **순열 p < 0.01 이면서 F1이 순열 p95보다 클 것.**
#    "배율 ≥2"를 안 쓰는 이유: F1 스윕의 순열 바닥은 0이 아니라 기저율 F1(~0.5)이라
#    배율 상한이 사실상 2.0이다 = 완벽한 분류기만 겨우 통과한다.
#    §F의 크기 검증(rho + 순열 p)에서 이미 쓰던 올바른 방식이다.
#    맞다. §H(중앙값)도 §H2(절대 비율)도 **박스 안쪽만** 봤다. 박스는 전부 결함 위에 놓여 있으니
#    "결함이 무슨 색인가"를 물은 셈이다. 유형을 가르는 건 그게 아니라
#    **주변 대비 어느 방향으로 얼마나 벗어났는가**다.
#    실제로 §H2에서 녹이 주황 구간(0~42)에 0.06뿐이고 자홍(213~234)에 0.56이었다 —
#    녹 박스 면적이 2.54%로 제일 작아 크롭이 **외피 색에 지배**당한 것이다.
#
# 여기서 재는 것 = (박스 안 색상 분포) − (박스를 둘러싼 고리의 색상 분포).
#   고리 = 박스를 RING배로 넓힌 사각형에서 박스를 뺀 영역 = **그 결함의 국소 정상면**.
#   §E가 VLM에 기준 패치를 나란히 보여준 것과 같은 발상을 픽셀로 하는 것이다.
# 🔴 스윕은 우연히 맞는다 → 순열도 **같은 스윕**을 돌려 거른다(§H·§H2와 동일).
#
# 선행: 이 노트북 §C(GT · IMGDIR) + §E 산출물(vlm_dump_*.json). §1·§4 불필요.
import json, random
import numpy as np
from collections import Counter
from PIL import Image

NB, SMIN, RING = 12, 40, 2.2      # 색상 12구간 · 채도 하한 · 고리 배율
_c = sorted(OUT.glob('vlm_dump*.json'), key=lambda p: p.stat().st_mtime)
D = json.loads(_c[-1].read_text(encoding='utf-8'))
print(f'덤프 {_c[-1].name}\n')

def _hist(hsv, x0, y0, x1, y1, mask=None):
    p = hsv[y0:y1, x0:x1].reshape(-1, 3)
    w = (p[:, 1] >= SMIN).astype(np.float32)
    if mask is not None: w = w * mask.reshape(-1)
    s = w.sum()
    if s < 20: return None                       # 표본이 너무 적으면 버린다
    return np.histogram(p[:, 0], bins=NB, range=(0, 256), weights=w)[0] / s

LAB, IN, OUT_, MAG = [], [], [], []
for name, v in D.items():
    pts = GT[name]['points']
    if not pts or not v['items']: continue
    im = Image.open(IMGDIR / name).convert('RGB')
    hsv = np.asarray(im.convert('HSV'), np.float32); W, H_ = im.size
    P = [(p['x'] * W, p['y'] * H_, p['t']) for p in pts]
    for it in v['items']:
        b = it['bbox']
        ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
        x1, y1 = min(W, int(b[2]) + 1), min(H_, int(b[3]) + 1)
        if x1 <= x0 or y1 <= y0: continue
        hin = _hist(hsv, x0, y0, x1, y1)
        if hin is None: continue
        # 고리 = 확대 사각형에서 박스를 뺀 영역
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        hw, hh = (x1 - x0) * RING / 2, (y1 - y0) * RING / 2
        X0, Y0 = max(0, int(cx - hw)), max(0, int(cy - hh))
        X1, Y1 = min(W, int(cx + hw)), min(H_, int(cy + hh))
        m = np.ones((Y1 - Y0, X1 - X0), np.float32)
        m[y0 - Y0:y1 - Y0, x0 - X0:x1 - X0] = 0            # 박스 구멍
        hout = _hist(hsv, X0, Y0, X1, Y1, m)
        if hout is None: continue
        LAB.append(Counter(ins).most_common(1)[0][0]); IN.append(hin); OUT_.append(hout)
        MAG.append(float(np.abs(hin - hout).sum()) / 2)      # 0~1: 분포가 얼마나 달라졌나
IN, OUT_, MAG = np.array(IN), np.array(OUT_), np.array(MAG)
DIF = IN - OUT_
LAB = np.array(LAB, dtype=object)
print(f'고리까지 확보된 박스 {len(LAB)}개 (고리 배율 {RING}×, 채도<{SMIN} 제외)\n')

DEG = [f'{int(i*360/NB):3d}°' for i in range(NB)]
ORDER = [t for t, n in Counter(LAB).most_common()]
print('유형별 **주변 대비** 색상 변화 (박스 안 − 고리, 양수 = 그 색이 늘었다)')
print(f'{"유형":12s}{"n":>4s} ' + ' '.join(f'{d:>5s}' for d in DEG) + '   변화량')
for t in ORDER:
    k = LAB == t
    print(f'{t:12s}{int(k.sum()):>4d} ' + ' '.join(f'{x:+5.2f}' for x in DIF[k].mean(0)) +
          f'   {MAG[k].mean():.3f}')
print('\n  ※ 같은 각도에서 유형끼리 부호·크기가 갈리면 색으로 이름을 붙일 수 있다')
print('    맨 오른쪽 변화량이 유형마다 비슷하면 "얼마나 다른가"로는 못 가른다(방향이 필요)\n')

# ── 스윕 + 순열 대조 ──────────────────────────────────────────────────
WINS = [(i, j) for i in range(NB) for j in range(i, NB)]
FEAT = np.stack([DIF[:, i:j + 1].sum(1) for i, j in WINS] + [MAG], 1)   # +변화량 자체도 후보
NAMES = [f'{int(i*360/NB)}~{int((j+1)*360/NB)}°' for i, j in WINS] + ['변화량']
THRS = np.array([-0.20, -0.10, -0.05, 0.02, 0.05, 0.10, 0.20, 0.35])

def best_f1(y):
    npos = int(y.sum())
    if npos == 0: return 0.0, None
    bf, bs = 0.0, None
    for t in THRS:
        pr = FEAT >= t
        tp = pr[y].sum(0); fp = pr[~y].sum(0); fn = npos - tp
        f1 = np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
        k = int(f1.argmax())
        if f1[k] > bf:
            bf, bs = float(f1[k]), (NAMES[k], float(t),
                                    float(tp[k] / max(tp[k] + fp[k], 1)), float(tp[k] / npos))
    return bf, bs

print('■ "주변 대비 색 변화"로 유형을 가를 수 있나 (전 구간 스윕 + 순열 대조)')
rng = random.Random(0)
best_ratio = 0.0
for t in ORDER:
    y = (LAB == t)
    if y.sum() < 10:
        print(f'  {t:12s} n={int(y.sum()):3d}  ⚠️ n<10 = 노이즈, 건너뜀'); continue
    f1, s = best_f1(y)
    null = sorted(best_f1(np.random.RandomState(i).permutation(y))[0] for i in range(200))
    p95 = null[int(.95 * len(null))]
    pv = (sum(1 for x in null if x >= f1) + 1) / (len(null) + 1)
    ok = pv < 0.01 and f1 > p95
    best_ratio = max(best_ratio, 2.0 if ok else 0.0)
    nm, th, P_, R_ = s
    print(f'  {t:12s} n={int(y.sum()):3d}  {nm:>10s} 변화 ≥{th:+.2f}  '
          f'F1 {f1:.3f} (P {P_:.3f} R {R_:.3f}) · 순열 평균 {np.mean(null):.3f} p95 {p95:.3f} · '
          f'p={pv:.3f} {"✅" if ok else "❌ 폐기"}')

print('\n판정')
if best_ratio >= 2:
    print('  ✅ **주변 대비로 보니 갈린다.** §H(중앙값)·§H2(절대 비율)가 틀린 잣대였던 것이고,')
    print('     사용자 지적("bbox 밖이 맞는 색")이 옳았다. 색 규칙으로 이름을 보정할 수 있다.')
else:
    print('  ❌ 주변 대비로도 안 갈린다 (순열 p ≥ 0.01). 절대·상대 둘 다 실패 = **색 노선 종료.**')
    print('     세 번(§H 중앙값 / §H2 절대 비율 / §H3 주변 대비) 다른 방식으로 재고 다 기각했다.')

# 🔴 one-vs-rest의 함정: 통과했다고 그 유형을 **식별**한다는 뜻이 아니다.
#    '다른 유형을 배제하는 것'만으로도 F1이 오른다(합성 검증에서 신호 0인 유형이 통과했다).
print('\n⚠️ 통과 = 그 유형을 이름 붙일 수 있다는 뜻이 아니다. one-vs-rest라서')
print('   \'다른 유형을 배제하는 것\'만으로도 F1이 오른다.')
print('   통과분은 위 P/R을 볼 것 — P가 낮으면 배제 신호지 식별 신호가 아니다.')


In [ ]:
# == §M 색 기반 태그 보정기 — 홀드아웃 검증 (VLM·검출 없음, 약 1분) ==
# §H2 결과: 색상 화소 비율이 유형을 가른다(녹 F1 0.757 · P 0.707 · 순열 p=0.005).
#   지금까지의 어떤 유형 분류보다 좋다(2단 Qwen 녹 0.000 · 1단 태그 녹 0.225).
#
# 🔴 그런데 그 규칙은 **197박스에서 스윕으로 고른 것**이다.
#    순열 대조는 "무작위 라벨로도 이게 되나"를 물었지 **"새 데이터에서도 되나"**를 안 물었다.
#    → 여기서 **훈련/테스트를 나눈다.** 훈련 절반에서 규칙을 뽑고 **테스트 절반에서만 채점**한다.
#      이걸 안 하면 오늘 하루 잡아낸 함정과 같은 종류를 내가 만드는 것이다.
#
# 🔴 채택 규칙 사전 고정:
#    ① 테스트 정확도 > 1단 OWLv2 태그 정확도 (같은 테스트 표본에서)
#    ② 순열 배율 ≥ 2  ← 정확도 기반이라 순열 바닥이 낮다(≈1/유형수). 배율 기준이 유효하다
#       (F1 스윕이 아니다 — F1에서 배율을 쓰면 안 되는 이유는 §H2 주석 참조)
#    ③ 훈련 정확도 − 테스트 정확도 ≤ 0.15  ← 과적합이면 채택하지 않는다
#
# 선행: 이 노트북 §C(GT · IMGDIR) + §E 산출물(vlm_dump_*.json). §1·§4 불필요.
import json, random
import numpy as np
from collections import Counter
from PIL import Image

NB, SMIN, SEED = 12, 40, 42
_c = sorted(OUT.glob('vlm_dump*.json'), key=lambda p: p.stat().st_mtime)
D = json.loads(_c[-1].read_text(encoding='utf-8'))
print(f'덤프 {_c[-1].name}\n')

LAB, HIST, TAG1, TAG2 = [], [], [], []
for name, v in D.items():
    pts = GT[name]['points']
    if not pts or not v['items']: continue
    im = Image.open(IMGDIR / name).convert('RGB')
    hsv = np.asarray(im.convert('HSV'), np.float32); W, H_ = im.size
    P = [(p['x'] * W, p['y'] * H_, p['t']) for p in pts]
    for it in v['items']:
        b = it['bbox']
        ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
        x1, y1 = min(W, int(b[2]) + 1), min(H_, int(b[3]) + 1)
        if x1 <= x0 or y1 <= y0: continue
        p = hsv[y0:y1, x0:x1].reshape(-1, 3)
        w = (p[:, 1] >= SMIN).astype(np.float32)
        if w.sum() < 20: continue
        HIST.append(np.histogram(p[:, 0], bins=NB, range=(0, 256), weights=w)[0] / w.sum())
        LAB.append(Counter(ins).most_common(1)[0][0])
        TAG1.append(it.get('tag', '?')); TAG2.append(it.get('kind', '?'))
HIST = np.array(HIST); LAB = np.array(LAB, dtype=object)
TAG1 = np.array(TAG1, dtype=object); TAG2 = np.array(TAG2, dtype=object)

WINS = [(i, j) for i in range(NB) for j in range(i, NB)]
FRAC = np.stack([HIST[:, i:j + 1].sum(1) for i, j in WINS], 1)
THRS = np.array([0.02, 0.05, 0.10, 0.20, 0.35, 0.50, 0.65])

# ── 층화 분할: 유형별로 반반 ──────────────────────────────────────────
rng = random.Random(SEED)
tr = np.zeros(len(LAB), bool)
for t in set(LAB):
    idx = [i for i in range(len(LAB)) if LAB[i] == t]; rng.shuffle(idx)
    tr[idx[:len(idx) // 2]] = True
te = ~tr
TYPES = [t for t, n in Counter(LAB[tr]).most_common() if n >= 5]
print(f'박스 {len(LAB)}개 → 훈련 {tr.sum()} / 테스트 {te.sum()} (층화, seed {SEED})')
print(f'규칙을 뽑을 유형: {TYPES}\n')

def fit_rule(y, m):
    """m(훈련 마스크) 안에서 최고 F1을 내는 (구간, 임계)"""
    yy = y[m]; F = FRAC[m]
    npos = int(yy.sum())
    if npos == 0: return None
    best = None
    for t in THRS:
        pr = F >= t
        tp = pr[yy].sum(0); fp = pr[~yy].sum(0); fn = npos - tp
        f1 = np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
        k = int(f1.argmax())
        if best is None or f1[k] > best[0]: best = (float(f1[k]), k, float(t))
    return best

RULES = {}
print(f'{"유형":12s} {"규칙":22s} {"훈련F1":>7s}')
for t in TYPES:
    r = fit_rule(LAB == t, tr)
    if not r: continue
    f1, k, th = r; i, j = WINS[k]
    RULES[t] = (k, th)
    print(f'{t:12s} 색상 {int(i*256/NB):3d}~{int((j+1)*256/NB):3d} ≥{th:4.0%}      {f1:7.3f}')

def predict(m):
    """여유(비율 − 임계)가 가장 큰 유형. 전부 음수면 미분류(None)"""
    out = []
    for r in np.where(m)[0]:
        best, bt = 0.0, None
        for t, (k, th) in RULES.items():
            marg = FRAC[r, k] - th
            if marg > best: best, bt = marg, t
        out.append(bt)
    return np.array(out, dtype=object)

def acc(pred, truth):
    return float(np.mean([a == b for a, b in zip(pred, truth)]))

def ctrl(pred, truth, n=200):
    v = list(pred); r = random.Random(0); s = []
    for _ in range(n):
        sh = v[:]; r.shuffle(sh); s.append(acc(sh, truth))
    return float(np.mean(s))

P_tr, T_tr = predict(tr), LAB[tr]
P_te, T_te = predict(te), LAB[te]
# 미분류는 1단 태그로 대체 — 실제 제품에서 쓸 형태
P_te_fb = np.array([p if p is not None else t for p, t in zip(P_te, TAG1[te])], dtype=object)

print(f'\n{"방법":26s} {"정확도":>7s} {"순열":>6s} {"배율":>6s} {"미분류":>6s}')
rows = [('색규칙 (훈련)', P_tr, T_tr), ('색규칙 (테스트)', P_te, T_te),
        ('색규칙+1단폴백 (테스트)', P_te_fb, T_te),
        ('1단 OWLv2 태그 (테스트)', TAG1[te], T_te),
        ('2단 Qwen 종류 (테스트)', TAG2[te], T_te)]
A = {}
for nm, p, t in rows:
    a = acc(p, t); c = ctrl(p, t); A[nm] = a
    un = int(sum(1 for x in p if x is None))
    print(f'{nm:26s} {a:7.3f} {c:6.3f} {a/max(c,1e-9):5.2f}× {un:6d}')

print(f'\n{"유형별 (테스트)":16s} {"n":>4s} {"색규칙":>7s} {"1단":>7s} {"2단":>7s}')
for t in TYPES:
    k = T_te == t
    if k.sum() == 0: continue
    print(f'{t:16s} {int(k.sum()):4d} {acc(P_te_fb[k], T_te[k]):7.3f} '
          f'{acc(TAG1[te][k], T_te[k]):7.3f} {acc(TAG2[te][k], T_te[k]):7.3f}')

a_te, a_tr = A['색규칙+1단폴백 (테스트)'], A['색규칙 (훈련)']
a_1 = A['1단 OWLv2 태그 (테스트)']
c1 = a_te > a_1
c2 = a_te / max(ctrl(P_te_fb, T_te), 1e-9) >= 2.0
c3 = a_tr - A['색규칙 (테스트)'] <= 0.15
print(f'\n■ 채택 판정  ①1단 초과 {a_te:.3f}>{a_1:.3f} {"✅" if c1 else "❌"}  '
      f'②배율 {"✅" if c2 else "❌"}  ③과적합 {a_tr-A["색규칙 (테스트)"]:+.3f} {"✅" if c3 else "❌"}')
# 🔴 기각 사유를 하드코딩하지 않는다 — 어느 조건이 걸렸는지 그대로 말한다(§K에서 겪은 실수).
if c1 and c2 and c3:
    print('▶ ✅ 채택 — 색 규칙으로 태그를 보정한다(재학습 0, 추론 비용 0)')
else:
    why = []
    if not c1: why.append('①1단을 못 이겼다')
    if not c2: why.append('②순열 배율 <2 = 우연과 구분이 약하다')
    if not c3: why.append('③훈련↔테스트 격차가 크다 = 스윕 과적합')
    print('▶ 🔴 기각 — ' + ' · '.join(why))
    if c1 and c3 and not c2:
        print('   ⚠️ ①③은 통과했다. 과적합이 아니라 **배율만** 미달이다 —')
        print('      테스트 1회의 우연일 수 있으니 §M2(5-폴드)로 다시 잴 것. 기준(≥2)은 그대로 둔다.')
if c1 and c2 and c3:
    print('   규칙:', {t: (f'색상 {int(WINS[k][0]*256/NB)}~{int((WINS[k][1]+1)*256/NB)}', f'≥{th:.0%}')
                       for t, (k, th) in RULES.items()})
    print('   ⚠️ 표본 197박스 · 테스트 절반이다. 발표에는 표본 크기를 같이 적을 것.')

In [ ]:
# == §M2 색 규칙 5-폴드 교차검증 + 유형별 최적 출처 (VLM·검출 없음, 약 1분) ==
# §M 결과: 색규칙 테스트 0.540(1단 0.140의 3.9배) · 과적합 +0.048(없음) · 배율 **1.97×로 0.03 미달**.
#   순수 색규칙은 정확히 2.00×였다 = **분할 하나의 우연**일 수 있다(테스트 100박스 1회).
#
# 🔴 여기서 통계를 바꾸지 않는다. **기준은 배율 ≥2 그대로**다(오늘 이 선으로 14개를 버렸다).
#    바꾸는 것은 **측정 횟수**뿐 — 5-폴드로 재서 1회 분할의 운을 걷어낸다.
# 🔴 사전 고정: **5폴드 배율 중앙값 ≥ 2** 이면서 **5폴드 전부 1단 태그 초과**. 하나라도 어기면 기각.
#
# 🔑 §M에서 본 것: 긁힘은 1단 태그가 0.778로 색규칙(0.111)보다 훨씬 낫다(§G에서도 1단 긁힘 0.722).
#    → 유형마다 출처가 다르다. **누가 이기는지를 훈련 절반에서만 정하고** 테스트에 적용한다.
#      테스트를 보고 고르면 그게 §M이 잡아낸 것과 같은 과적합이다.
#
# 선행: 이 노트북 §C(GT · IMGDIR) + §E 산출물(vlm_dump_*.json). §1·§4 불필요.
import json, random
import numpy as np
from collections import Counter
from PIL import Image

NB, SMIN, KFOLD = 12, 40, 5
_c = sorted(OUT.glob('vlm_dump*.json'), key=lambda p: p.stat().st_mtime)
D = json.loads(_c[-1].read_text(encoding='utf-8'))
print(f'덤프 {_c[-1].name}\n')

LAB, HIST, TAG1, TAG2 = [], [], [], []
for name, v in D.items():
    pts = GT[name]['points']
    if not pts or not v['items']: continue
    im = Image.open(IMGDIR / name).convert('RGB')
    hsv = np.asarray(im.convert('HSV'), np.float32); W, H_ = im.size
    P = [(p['x'] * W, p['y'] * H_, p['t']) for p in pts]
    for it in v['items']:
        b = it['bbox']
        ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
        x1, y1 = min(W, int(b[2]) + 1), min(H_, int(b[3]) + 1)
        if x1 <= x0 or y1 <= y0: continue
        p = hsv[y0:y1, x0:x1].reshape(-1, 3)
        w = (p[:, 1] >= SMIN).astype(np.float32)
        if w.sum() < 20: continue
        HIST.append(np.histogram(p[:, 0], bins=NB, range=(0, 256), weights=w)[0] / w.sum())
        LAB.append(Counter(ins).most_common(1)[0][0])
        TAG1.append(it.get('tag', '?')); TAG2.append(it.get('kind', '?'))
HIST = np.array(HIST); LAB = np.array(LAB, dtype=object)
TAG1 = np.array(TAG1, dtype=object); TAG2 = np.array(TAG2, dtype=object)
N = len(LAB)

WINS = [(i, j) for i in range(NB) for j in range(i, NB)]
FRAC = np.stack([HIST[:, i:j + 1].sum(1) for i, j in WINS], 1)
THRS = np.array([0.02, 0.05, 0.10, 0.20, 0.35, 0.50, 0.65])
BINLBL = [f'{int(i*256/NB)}~{int((j+1)*256/NB)}' for i, j in WINS]

# ── 층화 K-폴드 ───────────────────────────────────────────────────────
fold = np.zeros(N, int)
rng = random.Random(42)
for t in set(LAB):
    idx = [i for i in range(N) if LAB[i] == t]; rng.shuffle(idx)
    for r, i in enumerate(idx): fold[i] = r % KFOLD

def fit_rules(m):
    R = {}
    for t in [t for t, n in Counter(LAB[m]).most_common() if n >= 5]:
        y = (LAB == t)[m]; F = FRAC[m]; npos = int(y.sum())
        best = None
        for th in THRS:
            pr = F >= th
            tp = pr[y].sum(0); fp = pr[~y].sum(0); fn = npos - tp
            f1 = np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
            k = int(f1.argmax())
            if best is None or f1[k] > best[0]: best = (float(f1[k]), k, float(th))
        R[t] = (best[1], best[2])
    return R

def color_pred(R, rows):
    out = []
    for r in rows:
        best, bt = 0.0, None
        for t, (k, th) in R.items():
            mg = FRAC[r, k] - th
            if mg > best: best, bt = mg, t
        out.append(bt)
    return np.array(out, dtype=object)

def acc(p, t): return float(np.mean([a == b for a, b in zip(p, t)])) if len(t) else 0.0
def ctrl(p, t, n=200):
    v = list(p); r = random.Random(0)
    return float(np.mean([acc(r.sample(v, len(v)), t) for _ in range(n)]))

ROWS = []
print(f'{"폴드":6s} {"색규칙":>7s} {"혼합":>7s} {"1단":>7s} {"2단":>7s} {"순열(혼합)":>10s} {"배율":>6s}  훈련에서 색규칙을 믿기로 한 유형')
for f in range(KFOLD):
    te = (fold == f); tr = ~te
    R = fit_rules(tr)
    tr_rows, te_rows = np.where(tr)[0], np.where(te)[0]
    ptr = color_pred(R, tr_rows)
    # 🔑 유형별 출처 결정 — **훈련에서만**. 그 유형에서 색규칙이 1단보다 나을 때만 색규칙을 믿는다
    TRUST = set()
    for t in R:
        k = LAB[tr] == t
        if k.sum() >= 5 and acc(ptr[k], LAB[tr][k]) > acc(TAG1[tr][k], LAB[tr][k]): TRUST.add(t)
    pte = color_pred(R, te_rows)
    mix = np.array([p if (p in TRUST) else t for p, t in zip(pte, TAG1[te])], dtype=object)
    a_c, a_m = acc(pte, LAB[te]), acc(mix, LAB[te])
    a_1, a_2 = acc(TAG1[te], LAB[te]), acc(TAG2[te], LAB[te])
    c_m = ctrl(mix, LAB[te]); rat = a_m / max(c_m, 1e-9)
    ROWS.append((a_c, a_m, a_1, a_2, c_m, rat))
    print(f'{f:^6d} {a_c:7.3f} {a_m:7.3f} {a_1:7.3f} {a_2:7.3f} {c_m:10.3f} {rat:5.2f}×  {sorted(TRUST)}')

A = np.array([r[:6] for r in ROWS])
med = np.median(A, 0); mn = A.mean(0)
print(f'\n{"중앙값":6s} {med[0]:7.3f} {med[1]:7.3f} {med[2]:7.3f} {med[3]:7.3f} {med[4]:10.3f} {med[5]:5.2f}×')
print(f'{"평균":6s} {mn[0]:7.3f} {mn[1]:7.3f} {mn[2]:7.3f} {mn[3]:7.3f} {mn[4]:10.3f} {mn[5]:5.2f}×')

c1 = med[5] >= 2.0
c2 = all(r[1] > r[2] for r in ROWS)
print(f'\n■ 채택 판정 (기준은 §M과 동일 — 측정 횟수만 늘렸다)')
print(f'  ①배율 중앙값 {med[5]:.2f}× ≥2 {"✅" if c1 else "❌"}   '
      f'②5폴드 전부 1단 초과 {"✅" if c2 else "❌"} '
      f'({sum(1 for r in ROWS if r[1] > r[2])}/{KFOLD})')
if c1 and c2:
    print(f'\n▶ ✅ 채택 — 유형 태그 = 색 규칙(믿는 유형) + 1단 태그(나머지)')
    print(f'   테스트 정확도 {med[1]:.3f} vs 1단 단독 {med[2]:.3f} · 2단 단독 {med[3]:.3f}')
    print(f'   재학습 0 · 추론 비용 0 · 표본 {N}박스 5-폴드')
    print('   ⚠️ 발표에는 **표본 크기와 교차검증**을 같이 적을 것. 홀드아웃 없는 스윕 성적이 아니다.')
else:
    print(f'\n▶ 🔴 기각. ' +
          ('배율 중앙값이 2 미만 = 우연과 충분히 구분되지 않는다. ' if not c1 else '') +
          ('일부 폴드에서 1단을 못 이겼다 = 분할에 따라 뒤집힌다. ' if not c2 else '') +
          '\n   기준은 §M과 같다 — 결과를 보고 바꾸지 않았다.')

print(f'\n{"유형별 (5폴드 합산)":18s} {"n":>4s} {"색규칙":>7s} {"1단":>7s} {"2단":>7s}')
ALLP = np.empty(N, dtype=object)
for f in range(KFOLD):
    te = (fold == f)
    ALLP[te] = color_pred(fit_rules(~te), np.where(te)[0])
for t, n in Counter(LAB).most_common():
    if n < 5: continue
    k = LAB == t
    print(f'{t:18s} {n:4d} {acc(ALLP[k], LAB[k]):7.3f} {acc(TAG1[k], LAB[k]):7.3f} '
          f'{acc(TAG2[k], LAB[k]):7.3f}')
print('  ※ 유형마다 이기는 쪽이 다르다 — 이 표가 "어느 유형에 어느 출처를 쓸지"의 근거다')

In [ ]:
# == §A2 2차 표본 — 1차와 겹치지 않는 150장 (2~3분) ==
# 왜 2차가 필요한가: §M2에서 색 규칙이 검출기 태그를 5/5 폴드로 이겼는데 판정 통계가 미달했다.
#   통계를 고쳐야 하는데 **같은 데이터로 다시 재면 그건 판정이 아니다.**
#   → 1차 197박스 = 규칙 탐색용(훈련), 2차 = 최종 채점용(검증). 사전 등록 완료.
#
# 🔴 겹치면 안 된다. 1차에 쓴 파일명을 **명시적으로 제외**한다(같은 셀에서 다른 프레임이 나와도 무방 —
#    프레임 단위로만 안 겹치면 박스는 독립이다. 셀 단위 분리까지 하려면 EXCL_CELL=True).
# 🔴 §B는 그대로 쓴다. `WORKER='me2'` 만 바꾸면 labels_me2.json에 따로 쌓인다.
import json, random, re, os, glob, zipfile
from pathlib import Path
from collections import defaultdict

N_SAMPLE, PER_CELL, SPLIT, SEED2 = 150, 2, 'val', 2026
EXCL_CELL = False        # True면 1차에 쓴 **셀**까지 통째로 제외(더 엄격, 표본 다양성은 줄어듦)

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass
OUT = next((c for c in (Path('/content/drive/MyDrive/kt_out/ext_evalset'),
                        Path('/content/drive/MyDrive/ext_evalset'),
                        Path('/content/ext_evalset')) if (c / 'images').is_dir()), None)
assert OUT, '★평가셋 폴더를 못 찾았다 — §A를 먼저'
IMGDIR = OUT / 'images'
_ID = re.compile(r'cylindrical_(\d+)_')

PREV = set()
for f in sorted(OUT.glob('labels_*.json')):
    PREV |= set(json.loads(f.read_text(encoding='utf-8')))
for f in sorted(OUT.glob('sample_seed*_n*.json')):
    PREV |= set(json.loads(f.read_text(encoding='utf-8')))
PREV_CELL = {(_ID.search(n).group(1) if _ID.search(n) else n) for n in PREV}
print(f'1차 표본 {len(PREV)}장 (셀 {len(PREV_CELL)}개) — 제외 대상')

MAN2 = OUT / f'sample2_seed{SEED2}_n{N_SAMPLE}.json'
if MAN2.exists() and all((IMGDIR / n).exists() for n in json.loads(MAN2.read_text(encoding='utf-8'))):
    names2 = json.loads(MAN2.read_text(encoding='utf-8'))
    print(f'기존 2차 표본 재사용: {MAN2}')
else:
    cand = []
    for pat in ('/content/drive/MyDrive/**/*crop*v42*.zip', '/content/drive/**/*crop*v42*.zip'):
        cand = glob.glob(pat, recursive=True)
        if cand: break
    assert cand, '★ext_crop_v42.zip을 못 찾았다'
    ZP = max(cand, key=os.path.getsize)
    print(f'zip: {ZP} ({os.path.getsize(ZP)/1e9:.1f} GB) — namelist만 읽는다')
    with zipfile.ZipFile(ZP) as z:
        members = [m for m in z.namelist()
                   if m.startswith(f'images/{SPLIT}/') and m.endswith('.jpg')
                   and Path(m).name not in PREV]
        if EXCL_CELL:
            members = [m for m in members
                       if (_ID.search(m).group(1) if _ID.search(m) else '') not in PREV_CELL]
        assert len(members) >= N_SAMPLE, f'★후보 부족 {len(members)}장'
        by = defaultdict(list)
        for m in members:
            g = _ID.search(m); by[int(g.group(1)) if g else -1].append(m)
        rng = random.Random(SEED2)
        pool = []
        for cid in sorted(by):
            v = sorted(by[cid]); rng.shuffle(v); pool += v[:PER_CELL]
        rng.shuffle(pool); pick = pool[:N_SAMPLE]
        names2 = []
        for i, m in enumerate(pick, 1):
            nm = Path(m).name
            (IMGDIR / nm).write_bytes(z.read(m)); names2.append(nm)
            if i % 30 == 0: print(f'  {i}/{len(pick)}', flush=True)
    MAN2.write_text(json.dumps(names2, ensure_ascii=False), encoding='utf-8')
    print(f'2차 표본 확정: {MAN2}')

assert not (set(names2) & PREV), '★1차와 겹친다 — 제외 로직 확인'
_nc = len({(_ID.search(n).group(1) if _ID.search(n) else n) for n in names2})
print(f'\n2차 {len(names2)}장 (셀 {_nc}개) · 1차와 겹침 0장')
print(f'겹치는 셀 {len(PREV_CELL & {(_ID.search(n).group(1) if _ID.search(n) else n) for n in names2})}개'
      + ('  (EXCL_CELL=True로 하면 0이 된다 — 더 엄격한 분리가 필요하면)' if not EXCL_CELL else ''))

# ── §B가 바로 이어받도록 변수를 맞춰 둔다 ────────────────────────────
WORKER = 'me2'                                   # ★labels_me2.json 으로 따로 쌓인다
SAMPLE = [IMGDIR / n for n in names2]
SHARD  = (0, 1)
MINE   = SAMPLE
print(f"\n▶ 이제 **§B를 그대로 실행**하면 된다 (WORKER='{WORKER}', {len(MINE)}장).")
print('   §B는 매 장 저장하므로 중간에 끊겨도 이어서 하면 된다.')
print('   ⚠️ §C는 labels_*.json을 **전부 합친다** — 1차/2차 분리 채점은 §M3가 매니페스트로 가른다.')

In [ ]:
# == §H3b 주변 대비, 고리를 **셀 안으로** 자른다 (VLM·검출 없음, 약 2분) ==
# §H3가 §H2보다 나빴던 이유: 오염 박스가 셀 면적의 19.2%·벗겨짐 16.2%라
#   2.2배 고리를 뜨면 **셀을 벗어나 흰 배경**을 포함한다 = "주변"이 정상 표면이 아니었다.
#   → 고리를 셀 몸통(cell_box) 안으로 자르고, 셀 밖 화소는 가중 0으로 뺀다.
# 이걸로 사용자 가설("bbox 밖이 맞는 색")을 **공정하게** 다시 판정한다.
#
# 🔴 판정 기준: 순열 **p < 0.01** 이면서 F1 > 순열 p95 (§H2와 동일. 배율은 F1 스윕에 못 쓴다 — §11.9.1)
# 선행: 이 노트북 §C(GT · IMGDIR) + §E 산출물(vlm_dump_*.json). §1·§4 불필요.
import json, random
import numpy as np
from collections import Counter
from PIL import Image

NB, SMIN, RING = 12, 40, 2.2
_c = sorted(OUT.glob('vlm_dump*.json'), key=lambda p: p.stat().st_mtime)
D = json.loads(_c[-1].read_text(encoding='utf-8'))
print(f'덤프 {_c[-1].name}\n')

def cell_box(im, tol=55, frac=0.12):
    a = np.asarray(im, np.float32); H, W = a.shape[:2]; e = max(1, W // 20)
    bg = np.concatenate([a[:, :e].reshape(-1, 3), a[:, -e:].reshape(-1, 3)])
    m = np.abs(a - np.median(bg, 0)).sum(2) > tol
    xs = np.where(m.mean(0) > frac)[0]; ys = np.where(m.mean(1) > frac)[0]
    return (0, 0, W, H) if len(xs) < 5 or len(ys) < 5 else \
           (int(xs[0]), int(ys[0]), int(xs[-1]) + 1, int(ys[-1]) + 1)

LAB, DIF, MAG, LOST = [], [], [], []
for name, v in D.items():
    pts = GT[name]['points']
    if not pts or not v['items']: continue
    im = Image.open(IMGDIR / name).convert('RGB')
    hsv = np.asarray(im.convert('HSV'), np.float32); W, H_ = im.size
    cb = cell_box(im)
    P = [(p['x'] * W, p['y'] * H_, p['t']) for p in pts]
    for it in v['items']:
        b = it['bbox']
        ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
        x1, y1 = min(W, int(b[2]) + 1), min(H_, int(b[3]) + 1)
        if x1 <= x0 or y1 <= y0: continue
        pin = hsv[y0:y1, x0:x1].reshape(-1, 3)
        win = (pin[:, 1] >= SMIN).astype(np.float32)
        if win.sum() < 20: continue
        hin = np.histogram(pin[:, 0], bins=NB, range=(0, 256), weights=win)[0] / win.sum()

        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        hw, hh = (x1 - x0) * RING / 2, (y1 - y0) * RING / 2
        # 🔑 여기가 §H3와 다른 전부: 고리를 **셀 몸통 안으로** 자른다
        X0, Y0 = max(cb[0], int(cx - hw)), max(cb[1], int(cy - hh))
        X1, Y1 = min(cb[2], int(cx + hw)), min(cb[3], int(cy + hh))
        if X1 - X0 < 4 or Y1 - Y0 < 4: LOST.append(name); continue
        m = np.ones((Y1 - Y0, X1 - X0), np.float32)
        iy0, iy1 = max(0, y0 - Y0), max(0, min(Y1, y1) - Y0)
        ix0, ix1 = max(0, x0 - X0), max(0, min(X1, x1) - X0)
        m[iy0:iy1, ix0:ix1] = 0
        pout = hsv[Y0:Y1, X0:X1].reshape(-1, 3)
        wout = (pout[:, 1] >= SMIN).astype(np.float32) * m.reshape(-1)
        if wout.sum() < 20: LOST.append(name); continue
        hout = np.histogram(pout[:, 0], bins=NB, range=(0, 256), weights=wout)[0] / wout.sum()
        LAB.append(Counter(ins).most_common(1)[0][0])
        DIF.append(hin - hout); MAG.append(float(np.abs(hin - hout).sum()) / 2)
DIF = np.array(DIF); MAG = np.array(MAG); LAB = np.array(LAB, dtype=object)
print(f'고리를 셀 안에서 확보한 박스 {len(LAB)}개 (실패 {len(LOST)}개 — 박스가 셀을 거의 채운 경우)\n')

DEG = [f'{int(i*360/NB):3d}°' for i in range(NB)]
ORDER = [t for t, n in Counter(LAB).most_common()]
print('유형별 주변 대비 색상 변화 (셀 안 고리 기준)')
print(f'{"유형":12s}{"n":>4s} ' + ' '.join(f'{d:>5s}' for d in DEG) + '   변화량')
for t in ORDER:
    k = LAB == t
    print(f'{t:12s}{int(k.sum()):>4d} ' + ' '.join(f'{x:+5.2f}' for x in DIF[k].mean(0)) +
          f'   {MAG[k].mean():.3f}')

WINS = [(i, j) for i in range(NB) for j in range(i, NB)]
FEAT = np.stack([DIF[:, i:j + 1].sum(1) for i, j in WINS] + [MAG], 1)
NAMES = [f'{int(i*360/NB)}~{int((j+1)*360/NB)}°' for i, j in WINS] + ['변화량']
THRS = np.array([-0.20, -0.10, -0.05, 0.02, 0.05, 0.10, 0.20, 0.35])

def best_f1(y):
    npos = int(y.sum())
    if npos == 0: return 0.0, None
    bf, bs = 0.0, None
    for th in THRS:
        pr = FEAT >= th
        tp = pr[y].sum(0); fp = pr[~y].sum(0); fn = npos - tp
        f1 = np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
        k = int(f1.argmax())
        if f1[k] > bf: bf, bs = float(f1[k]), (NAMES[k], float(th),
                                               float(tp[k] / max(tp[k] + fp[k], 1)),
                                               float(tp[k] / npos))
    return bf, bs

print('\n■ 셀 안 고리 기준 — 유형을 가를 수 있나')
print(f'{"유형":12s} {"n":>4s} {"규칙":>16s} {"F1":>6s} {"P":>6s} {"R":>6s} {"순열p95":>8s} {"p":>7s}')
for t in ORDER:
    y = (LAB == t)
    if y.sum() < 10:
        print(f'{t:12s} {int(y.sum()):4d}  ⚠️ n<10 = 노이즈, 건너뜀'); continue
    f1, s = best_f1(y)
    null = sorted(best_f1(np.random.RandomState(i).permutation(y))[0] for i in range(200))
    p95 = null[int(.95 * len(null))]
    pv = (sum(1 for x in null if x >= f1) + 1) / (len(null) + 1)
    nm, th, P_, R_ = s
    print(f'{t:12s} {int(y.sum()):4d} {nm+f" ≥{th:+.2f}":>16s} {f1:6.3f} {P_:6.3f} {R_:6.3f} '
          f'{p95:8.3f} {pv:7.3f} {"✅" if (pv < 0.01 and f1 > p95) else "❌"}')

print('\n판정 — §H2(절대 색)와 비교할 것')
print('  · §H2보다 F1·P가 높다  → 사용자 가설이 옳았고 §H3의 고리 오염이 원인이었다')
print('  · §H2보다 낮거나 비슷 → 이 도메인에서는 **절대 색이 더 강한 신호**다. 색 노선은 §H2로 확정')
print('  ※ 참고 §H2 실측: 녹 F1 0.757 P 0.707 / 오염 0.715 0.621 / 벗겨짐 0.512 0.413')

In [ ]:
# == §M3 색 규칙 최종 판정 — 1차에서 규칙, 2차에서 채점 (검출 1분 · VLM 없음) ==
# 🔒 판정 규칙은 **2차 라벨을 한 장도 보기 전에** 못 박았다. 여기서 그대로 집행한다.
#    ① 순열 p < 0.01 (1000회)
#    ② 2차 정확도가 검출기 태그보다 **0.10 이상** 높을 것   ← 효과 크기 하한(§L 교훈)
#    ③ 1차(훈련) − 2차(검증) ≤ 0.15                      ← 과적합이면 기각
#    ④ 규칙은 1차에서만 뽑는다. 2차를 보고 구간·임계를 바꾸지 않는다
#    🔴🔴 사후 확인: ②의 기준을 **검출기 태그**로 잡은 것이 잘못이었다.
#       다중 클래스에서 넘어야 할 선은 **가장 흔한 유형을 찍기**(2차 = 무조건 녹 0.376)다.
#       색규칙 0.457은 그 대비 **+0.081**로 내 문턱(+0.10)에도 미달이고,
#       검출기 0.267은 아예 **베이스라인보다 낮다**. → 이 셀의 '채택'은 §V·§W로 **폐기**됐다.
#       다음 사전등록에서는 ②의 기준선을 반드시 **최빈 유형 정확도**로 쓸 것.
#    하나라도 어기면 **색 노선 종료. 세 번째 재도전 없다.**
# 🔴 배율은 참고로만 찍는다. 판정에 쓰지 않는다(다중클래스에서 순열 바닥이 오른다 — §11.10).
#
# 선행: unsup §1(detect) · 이 노트북 §C(GT · IMGDIR · OUT) · §A2+§B로 2차 라벨 완료
import json, random
import numpy as np
from collections import Counter
from PIL import Image

NB, SMIN = 12, 40
S2 = sorted(OUT.glob('sample2_seed*_n*.json'))
assert S2, '★2차 표본 매니페스트가 없다 — §A2를 먼저'
SET2 = set(json.loads(S2[-1].read_text(encoding='utf-8')))
A = {k: v for k, v in GT.items() if k not in SET2}      # 1차 = 규칙 탐색
B = {k: v for k, v in GT.items() if k in SET2}          # 2차 = 최종 채점
assert B, '★2차 라벨이 아직 없다 — §B를 WORKER="me2"로 돌릴 것'
print(f'1차(훈련) {len(A)}장 · 2차(검증) {len(B)}장 · 겹침 {len(set(A)&set(B))}장\n')

def build(frames):
    """→ (hist, 사람유형, 검출기태그).  박스는 지금 다시 검출한다(덤프에 2차가 없다)"""
    paths = [IMGDIR / n for n in frames if frames[n]['points']]
    DET = {p.name: v for p, v in detect(paths, thr=0.12, topk=6).items()}
    H, L, T = [], [], []
    for n in frames:
        if not frames[n]['points']: continue
        im = Image.open(IMGDIR / n).convert('RGB')
        hsv = np.asarray(im.convert('HSV'), np.float32); W, H_ = im.size
        P = [(p['x'] * W, p['y'] * H_, p['t']) for p in frames[n]['points']]
        for b in DET.get(n, []):
            ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
            if not ins: continue
            x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
            x1, y1 = min(W, int(b[2]) + 1), min(H_, int(b[3]) + 1)
            if x1 <= x0 or y1 <= y0: continue
            p = hsv[y0:y1, x0:x1].reshape(-1, 3)
            w = (p[:, 1] >= SMIN).astype(np.float32)
            if w.sum() < 20: continue
            H.append(np.histogram(p[:, 0], bins=NB, range=(0, 256), weights=w)[0] / w.sum())
            L.append(Counter(ins).most_common(1)[0][0]); T.append(b[5])
    return (np.array(H), np.array(L, dtype=object), np.array(T, dtype=object))

HA, LA, TA = build(A); HB, LB, TB = build(B)
print(f'박스: 1차 {len(LA)}개 · 2차 {len(LB)}개')
print(f'2차 유형 분포: {Counter(LB).most_common()}\n')

WINS = [(i, j) for i in range(NB) for j in range(i, NB)]
FA = np.stack([HA[:, i:j + 1].sum(1) for i, j in WINS], 1)
FB = np.stack([HB[:, i:j + 1].sum(1) for i, j in WINS], 1)
THRS = np.array([0.02, 0.05, 0.10, 0.20, 0.35, 0.50, 0.65])

# ── ④ 규칙은 1차에서만 ─────────────────────────────────────────────
RULES, TRUST = {}, set()
print(f'{"유형":12s} {"규칙(1차에서만)":22s} {"1차F1":>6s}  색규칙을 믿을 유형인가(1차 기준)')
for t, n in Counter(LA).most_common():
    if n < 5: continue
    y = (LA == t); npos = int(y.sum()); best = None
    for th in THRS:
        pr = FA >= th
        tp = pr[y].sum(0); fp = pr[~y].sum(0); fn = npos - tp
        f1 = np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
        k = int(f1.argmax())
        if best is None or f1[k] > best[0]: best = (float(f1[k]), k, float(th))
    RULES[t] = (best[1], best[2])
    i, j = WINS[best[1]]
    print(f'{t:12s} 색상 {int(i*256/NB):3d}~{int((j+1)*256/NB):3d} ≥{best[2]:4.0%}      {best[0]:6.3f}')

def predict(F):
    out = []
    for r in range(F.shape[0]):
        best, bt = 0.0, None
        for t, (k, th) in RULES.items():
            mg = F[r, k] - th
            if mg > best: best, bt = mg, t
        out.append(bt)
    return np.array(out, dtype=object)

PA = predict(FA)
for t in RULES:                       # 유형별 출처도 1차에서만 결정
    k = LA == t
    if k.sum() >= 5 and np.mean(PA[k] == LA[k]) > np.mean(TA[k] == LA[k]): TRUST.add(t)
print(f'\n색규칙을 믿기로 한 유형(1차 기준): {sorted(TRUST)}')

def mix(P, T): return np.array([p if p in TRUST else t for p, t in zip(P, T)], dtype=object)
acc = lambda p, t: float(np.mean(p == t)) if len(t) else 0.0

MA, MB = mix(PA, TA), mix(predict(FB), TB)
a_tr, a_te, a_1, a_2 = acc(MA, LA), acc(MB, LB), acc(TB, LB), None

# ── ① 순열 p값 (1000회) ────────────────────────────────────────────
rng = random.Random(0); v = list(MB); null = []
for _ in range(1000):
    sh = v[:]; rng.shuffle(sh); null.append(acc(np.array(sh, dtype=object), LB))
null.sort()
pv = (sum(1 for x in null if x >= a_te) + 1) / 1001
print(f'\n{"":22s} {"정확도":>7s}')
print(f'{"색규칙+폴백 (1차 훈련)":22s} {a_tr:7.3f}')
print(f'{"색규칙+폴백 (2차 검증)":22s} {a_te:7.3f}   ← 판정 대상')
print(f'{"검출기 태그 (2차)":22s} {a_1:7.3f}')
print(f'\n순열 1000회: 평균 {np.mean(null):.3f} · p95 {null[949]:.3f} · **p = {pv:.4f}** '
      f'(배율 {a_te/max(np.mean(null),1e-9):.2f}× — 참고용, 판정에 안 씀)')

c1 = pv < 0.01
c2 = a_te - a_1 >= 0.10
c3 = a_tr - a_te <= 0.15
print(f'\n■ 사전 등록 규칙 집행 (0805.md §11.11)')
print(f'  ① p={pv:.4f} < 0.01           {"✅" if c1 else "❌"}')
print(f'  ② 검출기 대비 {a_te-a_1:+.3f} ≥ +0.10  {"✅" if c2 else "❌"}')
print(f'  ③ 훈련−검증 {a_tr-a_te:+.3f} ≤ 0.15   {"✅" if c3 else "❌"}')
if c1 and c2 and c3:
    print(f'\n▶ ✅ **채택** — 유형 태그 = 색 규칙(믿는 유형) + 검출기 태그(나머지)')
    print(f'   2차 검증 {a_te:.3f} vs 검출기 {a_1:.3f} · 재학습 0 · 추론 비용 0')
    print(f'   규칙: ' + str({t: f'색상 {int(WINS[k][0]*256/NB)}~{int((WINS[k][1]+1)*256/NB)} ≥{th:.0%}'
                              for t, (k, th) in RULES.items() if t in TRUST}))
    print('   ⚠️ 발표에는 **1차에서 규칙·2차에서 채점**과 표본 크기를 반드시 같이 적을 것')
else:
    why = [n for n, c in (('①p값', c1), ('②효과크기', c2), ('③과적합', c3)) if not c]
    print(f'\n▶ 🔴 **기각** ({" · ".join(why)}) — 사전 등록대로 **색 노선 종료**. 세 번째 재도전 없다.')

print(f'\n{"유형별 (2차)":14s} {"n":>4s} {"색규칙+폴백":>10s} {"검출기":>7s}')
for t, n in Counter(LB).most_common():
    k = LB == t
    print(f'{t:14s} {n:4d} {acc(MB[k], LB[k]):10.3f} {acc(TB[k], LB[k]):7.3f}')

In [ ]:
# == §M4 혼합 규칙의 구조적 결함 수정 — 검출기가 강한 유형은 덮지 않는다 (검출 1분) ==
# §M3 2차 실측에서 드러난 것:
#   파손·찢김  혼합 0.000  vs  검출기 0.629   ← 폴백이면 0.629여야 하는데 0.000이다
#   긁힘       혼합 0.000  vs  검출기 0.615
# 원인: 혼합이 `색규칙 예측이 TRUST면 그걸 쓴다`라서, **색 규칙에 없는 유형**(파손 — 1차 표본
#   부족으로 규칙 자체를 못 뽑았다)의 박스가 "녹" 같은 다른 TRUST 유형으로 예측되면
#   **검출기의 맞는 답을 덮어쓴다.** 튜닝 문제가 아니라 구성상 틀린 것이다.
#
# 수정: **검출기가 "색 규칙이 없거나 못 믿는 유형"이라고 하면 검출기를 그대로 둔다.**
#   이 판단에 쓰는 정보(어느 유형에 규칙이 있나 / 1차에서 누가 이기나)는 **전부 1차 것**이다.
#
# 🔴🔴 정직하게 적을 것: 이 결함을 **2차 표를 보고** 알아챘다. 규칙은 1차 정보만 쓰지만
#    2차 점수는 그만큼 **낙관적**이다. → 발표 숫자는 §M3의 **0.457**을 쓴다.
#    아래 B는 "다음 라운드에서 검증할 개선안"이지 오늘 주장할 성적이 아니다.
#
# 선행: §M3를 먼저 실행(RULES · TRUST · FA · FB · LA · LB · TA · TB · WINS · predict를 그대로 쓴다)
import random
import numpy as np
from collections import Counter

assert 'RULES' in dir() and 'TRUST' in dir(), '★§M3를 먼저 실행할 것'
NO_RULE = ({t for t in set(LA) if t not in RULES} | (set(RULES) - TRUST))
print(f'색 규칙이 없거나 못 믿는 유형(1차 기준): {sorted(NO_RULE)}')
print('  → 검출기가 이 중 하나라고 하면 **덮지 않는다**\n')

def mixA(P, T):   # §M3에서 채택된 것
    return np.array([p if p in TRUST else t for p, t in zip(P, T)], dtype=object)
def mixB(P, T):   # 수정안
    return np.array([t if t in NO_RULE else (p if p in TRUST else t)
                     for p, t in zip(P, T)], dtype=object)

acc = lambda p, t: float(np.mean(p == t)) if len(t) else 0.0
def pval(pred, truth, n=1000):
    v = list(pred); r = random.Random(0)
    null = sorted(acc(np.array(r.sample(v, len(v)), dtype=object), truth) for _ in range(n))
    a = acc(pred, truth)
    return a, float(np.mean(null)), (sum(1 for x in null if x >= a) + 1) / (n + 1)

PB = predict(FB)
ROWS = [('A 색규칙 우선 (§M3 채택)', mixA(PB, TB)),
        ('B 검출기 강한 유형 보존', mixB(PB, TB)),
        ('C 검출기 태그만', TB)]
print(f'{"혼합 규칙":26s} {"2차 정확도":>9s} {"순열평균":>8s} {"p":>7s}')
A = {}
for nm, p in ROWS:
    a, c, pv = pval(p, LB); A[nm] = a
    print(f'{nm:26s} {a:9.3f} {c:8.3f} {pv:7.4f}')

print(f'\n{"유형별 (2차)":14s} {"n":>4s} {"A":>7s} {"B":>7s} {"검출기":>7s}')
mA, mB = mixA(PB, TB), mixB(PB, TB)
for t, n in Counter(LB).most_common():
    k = LB == t
    print(f'{t:14s} {n:4d} {acc(mA[k], LB[k]):7.3f} {acc(mB[k], LB[k]):7.3f} {acc(TB[k], LB[k]):7.3f}')

print(f'\n■ 읽는 법')
print(f'  · B > A 이면 구조적 결함이 실재했다는 뜻이다 (파손·긁힘을 안 덮으니 회복)')
print(f'  · 🔴 그래도 **발표 숫자는 A의 {A["A 색규칙 우선 (§M3 채택)"]:.3f}**를 쓴다 —')
print(f'    B는 2차를 본 뒤에 고친 것이라 이 점수가 낙관적이다. 3차 라벨로 사전 등록 후 재검증할 것.')
print(f'  · 다음 라운드 사전 등록안: "혼합 B가 검출기 태그보다 +0.10 이상, p<0.01, 과적합 ≤0.15"')

In [ ]:
# == §V 육안 확인 — 색 규칙 판정을 셀별 이미지로 저장 (검출 1분) ==
# §M3 결과(2차 0.457 vs 검출기 0.267)를 **눈으로** 본다. 지금까지 전부 숫자로만 봤다.
# 각 크롭에 세 줄을 적는다: 사람 라벨 / 색 규칙 판정 / 검출기 태그.
# 저장: {OUT}/check/cell_XXXX.jpg  — **셀 하나당 파일 하나**.
#
# 선행: §M3 실행(RULES · TRUST · WINS · NB · SMIN) · unsup §1(detect) · §C(GT · IMGDIR · OUT)
import json, random, re, subprocess, glob as _g
import numpy as np
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

N_SHOW, TILE, PAD = 100, 220, 0.30    # 뽑을 박스 수 · 타일 폭 · 크롭 여백(박스 최대변 대비)
CAP_H, COLS = 66, 4                    # 캡션 높이 · 한 줄에 몇 칸
assert 'RULES' in dir() and 'TRUST' in dir(), '★§M3를 먼저 실행할 것'

# ── 한글 폰트 (없으면 설치, 그래도 없으면 영문 약어) ──────────────────
def _kfont(sz):
    c = ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
         '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf',
         '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
         'C:/Windows/Fonts/malgun.ttf']
    c += _g.glob('/usr/share/fonts/**/*Nanum*.ttf', recursive=True)
    c += _g.glob('/usr/share/fonts/**/*CJK*.tt?', recursive=True)
    for p in c:
        try: return ImageFont.truetype(p, sz), True
        except Exception: pass
    return ImageFont.load_default(), False

F, KO = _kfont(17)
if not KO:
    print('한글 폰트 설치 중 (~15초)...')
    subprocess.run('apt-get -qq install -y fonts-nanum', shell=True)
    F, KO = _kfont(17)
FB = _kfont(15)[0]
AB = {'녹·부식': 'RUST', '벗겨짐·박리': 'PEEL', '파손·찢김': 'TEAR', '긁힘·스크래치': 'SCRATCH',
      '들뜸': 'LIFT', '오염·이물질': 'DIRT', '기타': 'ETC', None: '-'}
def L(x): return (x if KO else AB.get(x, str(x)))
if not KO: print('⚠️ 한글 폰트 없음 — 영문 약어로 표기 (RUST/PEEL/TEAR/SCRATCH/LIFT/DIRT)')

# ── 2차 프레임의 박스를 다시 검출하고 색 규칙을 적용 ─────────────────
S2 = sorted(OUT.glob('sample2_seed*_n*.json'))
SET2 = set(json.loads(S2[-1].read_text(encoding='utf-8'))) if S2 else set()
FR = {k: v for k, v in GT.items() if k in SET2 and v['points']} or \
     {k: v for k, v in GT.items() if v['points']}
print(f'대상 프레임 {len(FR)}장 ({"2차" if SET2 else "전체"})')

DET = {p.name: v for p, v in detect([IMGDIR / n for n in FR], thr=0.12, topk=6).items()}
ITEM = []
for n in FR:
    im = Image.open(IMGDIR / n).convert('RGB')
    hsv = np.asarray(im.convert('HSV'), np.float32); W, H = im.size
    P = [(p['x'] * W, p['y'] * H, p['t']) for p in FR[n]['points']]
    for b in DET.get(n, []):
        ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
        x1, y1 = min(W, int(b[2]) + 1), min(H, int(b[3]) + 1)
        if x1 <= x0 or y1 <= y0: continue
        q = hsv[y0:y1, x0:x1].reshape(-1, 3)
        w = (q[:, 1] >= SMIN).astype(np.float32)
        if w.sum() < 20: continue
        h = np.histogram(q[:, 0], bins=NB, range=(0, 256), weights=w)[0] / w.sum()
        best, pred = 0.0, None
        for t, (k, th) in RULES.items():
            i, j = WINS[k]; mg = h[i:j + 1].sum() - th
            if mg > best: best, pred = mg, t
        ITEM.append({'frame': n, 'box': (x0, y0, x1, y1),
                     'truth': Counter(ins).most_common(1)[0][0],
                     'color': pred, 'tag': b[5],
                     'final': pred if pred in TRUST else b[5]})
print(f'사람 점을 담은 박스 {len(ITEM)}개\n')

# ── 유형별 층화로 100개 ───────────────────────────────────────────────
by = defaultdict(list)
for it in ITEM: by[it['truth']].append(it)
rng = random.Random(0)
per = max(1, N_SHOW // max(len(by), 1)); SEL = []
for t in by:
    v = by[t][:]; rng.shuffle(v); SEL += v[:per]
rest = [x for x in ITEM if x not in SEL]; rng.shuffle(rest)
SEL += rest[:max(0, N_SHOW - len(SEL))]
print('뽑은 표본 유형 분포:', Counter(x['truth'] for x in SEL).most_common())

# ── 셀별로 묶어 파일 하나씩 ───────────────────────────────────────────
_ID = re.compile(r'cylindrical_(\d+)_')
CK = OUT / 'check'; CK.mkdir(exist_ok=True)
for f in CK.glob('cell_*.jpg'): f.unlink()
cells = defaultdict(list)
for it in SEL:
    m = _ID.search(it['frame']); cells[m.group(1) if m else 'unknown'].append(it)

def tile(it):
    im = Image.open(IMGDIR / it['frame']).convert('RGB')
    x0, y0, x1, y1 = it['box']; mg = max(x1 - x0, y1 - y0) * PAD
    c = im.crop((max(0, x0 - mg), max(0, y0 - mg),
                 min(im.width, x1 + mg), min(im.height, y1 + mg)))
    k = TILE / max(c.size)
    c = c.resize((max(1, round(c.width * k)), max(1, round(c.height * k))), Image.LANCZOS)
    sh = Image.new('RGB', (TILE, TILE + CAP_H), (255, 255, 255))
    sh.paste(c, ((TILE - c.width) // 2, (TILE - c.height) // 2))
    d = ImageDraw.Draw(sh)
    d.rectangle([0, 0, TILE - 1, TILE - 1], outline=(190, 190, 190))
    ok = it['final'] == it['truth']
    d.rectangle([0, TILE, TILE - 1, TILE + CAP_H - 1],
                fill=(226, 245, 228) if ok else (253, 228, 228))
    d.text((6, TILE + 3),  f"사람   {L(it['truth'])}" if KO else f"HUMAN  {L(it['truth'])}",
           fill=(0, 0, 0), font=F)
    d.text((6, TILE + 23), (f"색규칙 {L(it['color'])} {'O' if ok else 'X'}" if KO
                            else f"COLOR  {L(it['color'])} {'O' if ok else 'X'}"),
           fill=(20, 110, 40) if ok else (185, 30, 30), font=F)
    d.text((6, TILE + 44), f"검출기 {L(it['tag'])}" if KO else f"DETECT {L(it['tag'])}",
           fill=(110, 110, 110), font=FB)
    return sh

n_ok = 0
for cid, its in sorted(cells.items()):
    rows = -(-len(its) // COLS)
    sh = Image.new('RGB', (COLS * TILE, rows * (TILE + CAP_H) + 28), (255, 255, 255))
    d = ImageDraw.Draw(sh)
    a = sum(1 for x in its if x['final'] == x['truth']); n_ok += a
    d.text((6, 6), f'cell {cid}  ·  {len(its)} boxes  ·  color-rule correct {a}/{len(its)}',
           fill=(0, 0, 0), font=F)
    for i, it in enumerate(its):
        sh.paste(tile(it), (i % COLS * TILE, 28 + i // COLS * (TILE + CAP_H)))
    sh.save(CK / f'cell_{cid}.jpg', quality=92)

print(f'\n저장: {CK}  · 파일 {len(cells)}개 (셀당 1개)')
print(f'표본 {len(SEL)}개 중 색 규칙 적중 {n_ok} = {n_ok/max(len(SEL),1):.3f}')
print(f'초록 = 맞음 · 빨강 = 틀림. 캡션 3줄 = 사람 / 색규칙 / 검출기\n')
for cid, its in sorted(cells.items())[:3]:
    print(f'── cell {cid} ──'); display(Image.open(CK / f'cell_{cid}.jpg'))
print('나머지는 Drive의 check/ 폴더에서 볼 것.')
print('한 번에 받으려면:  import shutil; shutil.make_archive("/content/check","zip",str(CK))')

In [ ]:
# == §W 표본 편향·채점 정의 재검 + 불일치 전용 시트 (검출 1분) ==
# 사용자 육안 판정: "검출기가 잘 잡는다. 색규칙은 참고만."
# 숫자(색 0.457 vs 검출기 0.267)와 갈린다 → **눈을 의심하기 전에 표본과 채점 정의를 의심한다.**
#
# 의심 ①: §V가 **유형별 층화 추출**이라 색규칙이 압도하는 녹을 38%→16%로 줄이고
#         검출기가 강한 파손·긁힘을 부풀렸다. = 검출기가 실제보다 좋아 보이는 표본이었다(내 실수).
# 의심 ②: 박스 하나에 **여러 유형의 점**이 들어 있으면 우리는 다수결로 하나만 정답 처리한다.
#         녹 옆에 긁힘이 있는 크롭이면 검출기의 "긁힘"은 **틀린 게 아닌데** 오답이 된다.
#         → `any-match`(박스 안 점 유형 중 하나와 맞으면 정답)로도 재채점한다.
#
# 산출: ①실제 비율 100개 시트 `check_natural/` ②불일치만 모은 시트 `check_diff/`
#       ③다수결 vs any-match 정확도 표
# 선행: §M3(RULES·TRUST·WINS·NB·SMIN) · §V(폰트 F·FB·KO·L·tile) · unsup §1(detect) · §C
import json, random, re
import numpy as np
from collections import Counter, defaultdict
from PIL import Image, ImageDraw
from IPython.display import display

assert 'RULES' in dir() and 'TRUST' in dir(), '★§M3를 먼저'
assert 'tile' in dir() and 'F' in dir(), '★§V를 먼저 (폰트·타일 함수를 쓴다)'
N_NAT, N_DIFF, COLS2 = 100, 60, 4

S2 = sorted(OUT.glob('sample2_seed*_n*.json'))
SET2 = set(json.loads(S2[-1].read_text(encoding='utf-8'))) if S2 else set()
FR = {k: v for k, v in GT.items() if k in SET2 and v['points']} or \
     {k: v for k, v in GT.items() if v['points']}
DET = {p.name: v for p, v in detect([IMGDIR / n for n in FR], thr=0.12, topk=6).items()}

ALL = []
for n in FR:
    im = Image.open(IMGDIR / n).convert('RGB')
    hsv = np.asarray(im.convert('HSV'), np.float32); W, H = im.size
    P = [(p['x'] * W, p['y'] * H, p['t']) for p in FR[n]['points']]
    for b in DET.get(n, []):
        ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
        x1, y1 = min(W, int(b[2]) + 1), min(H, int(b[3]) + 1)
        if x1 <= x0 or y1 <= y0: continue
        q = hsv[y0:y1, x0:x1].reshape(-1, 3)
        w = (q[:, 1] >= SMIN).astype(np.float32)
        if w.sum() < 20: continue
        h = np.histogram(q[:, 0], bins=NB, range=(0, 256), weights=w)[0] / w.sum()
        best, pred = 0.0, None
        for t, (k, th) in RULES.items():
            i, j = WINS[k]; mg = h[i:j + 1].sum() - th
            if mg > best: best, pred = mg, t
        ALL.append({'frame': n, 'box': (x0, y0, x1, y1), 'truth': Counter(ins).most_common(1)[0][0],
                    'all_truth': set(ins), 'color': pred, 'tag': b[5],
                    'final': pred if pred in TRUST else b[5]})
print(f'박스 {len(ALL)}개\n')

# ── ② 채점 정의 재검: 다수결 vs any-match ─────────────────────────────
multi = [x for x in ALL if len(x['all_truth']) > 1]
print(f'■ 박스 안에 **여러 유형**의 점이 있는 경우: {len(multi)}/{len(ALL)} = {len(multi)/len(ALL):.1%}')
if multi:
    print('   예:', Counter(tuple(sorted(x['all_truth'])) for x in multi).most_common(4))
print(f'\n{"방법":16s} {"다수결 정확도":>12s} {"any-match":>10s} {"차이":>7s}')
for nm, key in (('색규칙+폴백', 'final'), ('색 규칙 단독', 'color'), ('검출기 태그', 'tag')):
    a1 = np.mean([x[key] == x['truth'] for x in ALL])
    a2 = np.mean([x[key] in x['all_truth'] for x in ALL])
    print(f'{nm:16s} {a1:12.3f} {a2:10.3f} {a2-a1:+7.3f}')
print('   → any-match에서 검출기가 크게 오르면, 검출기는 "다른 맞는 답"을 말하고 있었던 것이다')

# 🔴 사소한 베이스라인 — 다중 클래스에서 **진짜 넘어야 할 선**
#    순열 대조는 '라벨을 섞어도 되나'를 묻지 '가장 흔한 유형만 찍는 것보다 나은가'를 안 묻는다.
# 실측 검출기 태그 0.267 < 무조건-녹 0.376 = **사소한 규칙보다 못한다.**
_cnt = Counter(x['truth'] for x in ALL)
_maj, _n = _cnt.most_common(1)[0]
_base = _n / len(ALL)
print(f'\n■ 사소한 베이스라인: 무조건 \'{_maj}\' → {_base:.3f}  (n={_n}/{len(ALL)})')
print(f'{"방법":16s} {"정확도":>8s} {"베이스라인 대비":>14s}')
for _nm, _k in (('색규칙+폴백', 'final'), ('색 규칙 단독', 'color'), ('검출기 태그', 'tag')):
    _a = float(np.mean([x[_k] == x['truth'] for x in ALL]))
    print(f'{_nm:16s} {_a:8.3f} {_a-_base:+14.3f}' + ('   🔴 사소한 규칙보다 못하다' if _a < _base else ''))
print('   → 이 열이 +0.10 미만이면 유형 분류로서 의미가 없다. 비교 모델이 아니라 여기를 넘어야 한다.')


print(f'\n{"유형별":14s} {"n":>4s} {"실제비율":>8s} | {"색규칙":>7s} {"검출기":>7s}')
for t, n in Counter(x['truth'] for x in ALL).most_common():
    k = [x for x in ALL if x['truth'] == t]
    print(f'{t:14s} {n:4d} {n/len(ALL):8.1%} | '
          f'{np.mean([x["final"]==t for x in k]):7.3f} {np.mean([x["tag"]==t for x in k]):7.3f}')
print('   ※ §V는 층화 추출이라 위 실제비율과 다르게 보였다 — 아래 check_natural이 진짜 비율이다')

# ── ① 실제 비율 표본 · ② 불일치 전용 ─────────────────────────────────
_ID = re.compile(r'cylindrical_(\d+)_')
rng = random.Random(0)

def dump(items, sub, title, neutral=False):
    d0 = OUT / sub; d0.mkdir(exist_ok=True)
    for f in d0.glob('*.jpg'): f.unlink()
    cells = defaultdict(list)
    for it in items:
        m = _ID.search(it['frame']); cells[m.group(1) if m else 'unknown'].append(it)
    for cid, its in sorted(cells.items()):
        rows = -(-len(its) // COLS2)
        sh = Image.new('RGB', (COLS2 * TILE, rows * (TILE + CAP_H) + 28), (255, 255, 255))
        dr = ImageDraw.Draw(sh)
        a = sum(1 for x in its if x['final'] == x['truth'])
        dr.text((6, 6), f'{title}  cell {cid}  ·  {len(its)} boxes' +
                ('' if neutral else f'  ·  color-rule {a}/{len(its)}'), fill=(0, 0, 0), font=F)
        for i, it in enumerate(its):
            t = tile(it)
            if neutral:      # 판정색을 지운다 — 사람이 편견 없이 보게
                dr2 = ImageDraw.Draw(t)
                dr2.rectangle([0, TILE, TILE - 1, TILE + CAP_H - 1], fill=(245, 245, 245))
                dr2.text((6, TILE + 3),  f"사람   {L(it['truth'])}" if KO else f"HUMAN  {L(it['truth'])}",
                         fill=(0, 0, 0), font=F)
                dr2.text((6, TILE + 23), f"색규칙 {L(it['color'])}" if KO else f"COLOR  {L(it['color'])}",
                         fill=(0, 0, 0), font=F)
                dr2.text((6, TILE + 44), f"검출기 {L(it['tag'])}" if KO else f"DETECT {L(it['tag'])}",
                         fill=(0, 0, 0), font=FB)
            sh.paste(t, (i % COLS2 * TILE, 28 + i // COLS2 * (TILE + CAP_H)))
        sh.save(d0 / f'cell_{cid}.jpg', quality=92)
    print(f'  저장 {d0} · 파일 {len(cells)}개 · 박스 {len(items)}개')
    return d0

nat = ALL[:]; rng.shuffle(nat); nat = nat[:N_NAT]
print(f'\n■ ① 실제 비율 표본 (층화 없음)')
print('   유형 분포:', Counter(x['truth'] for x in nat).most_common())
dump(nat, 'check_natural', 'NATURAL')

dif = [x for x in ALL if x['color'] is not None and x['color'] != x['tag']]
rng.shuffle(dif); dif = dif[:N_DIFF]
print(f'\n■ ② 색규칙 ≠ 검출기 인 박스만 (판정색 없음 — 눈으로 직접 고르라고)')
if dif:
    c_ok = np.mean([x['color'] == x['truth'] for x in dif])
    t_ok = np.mean([x['tag'] == x['truth'] for x in dif])
    print(f'   이 불일치 구간에서: 색규칙 {c_ok:.3f} · 검출기 {t_ok:.3f} (사람 라벨 기준)')
    dump(dif, 'check_diff', 'DISAGREE', neutral=True)

print('\n▶ 볼 순서')
print('  1) check_natural — 실제 비율. §V(층화)보다 녹이 훨씬 많이 보인다')
print('  2) check_diff    — 둘이 갈리는 박스만. 사람 라벨과 무관하게 **어느 쪽이 맞는지 직접** 볼 것')
print('  3) 위 any-match 표 — 검출기가 여기서 크게 오르면 우리 채점이 검출기를 부당하게 깎고 있던 것')

In [ ]:
# == §X top-2 후보 + 점수 — 결함이 애매하니 1등만 강요하지 않는다 (검출 1분) ==
# 사용자 제안: "결함별로 점수를 줘서 top2까지 보여주자. top1 긁힘 63% / top2 27%"
#   근거가 있다 — §H에서 **녹과 긁힘의 픽셀 프로필이 사실상 같다**(색상 217 vs 214.5).
#   애매한 것을 억지로 하나로 찍게 하는 대신, 후보 2개와 점수를 리포트 VLM에 넘긴다.
#
# 🔴 함정 ①: **top-2의 사소한 베이스라인은 0.581**이다(최빈 2개 = 녹 37.6% + 벗겨짐 20.5%).
#    top-1에서 0.376을 놓쳐 두 번 잘못 읽었다. 여기서는 처음부터 같이 찍는다.
# 🔴 함정 ②: "63%"라 적어놓고 63%일 때 실제 63%가 아니면 **리포트 VLM을 오도한다.**
#    → **캘리브레이션 표**(점수 구간별 실제 적중률)를 같이 낸다. 이게 맞아야 점수를 쓸 수 있다.
# 🔴 함정 ③: top-2는 **2차를 다 본 뒤에 정한 지표**다. 규칙 자체는 1차에서 뽑았고 새로 맞춘
#    하이퍼파라미터도 없지만(순위를 2까지 읽을 뿐), **주장하려면 3차가 필요하다.**
#    오늘은 "쓸 만한가"를 보는 것까지.
#
# 선행: §M3(RULES·TRUST·WINS·NB·SMIN) · §V(F·FB·KO·L·TILE·CAP_H) · unsup §1(detect) · §C
import json, random, re
import numpy as np
from collections import Counter, defaultdict
from PIL import Image, ImageDraw
from IPython.display import display

assert 'RULES' in dir(), '★§M3를 먼저'
assert 'F' in dir() and 'L' in dir(), '★§V를 먼저 (폰트를 쓴다)'
TEMP, N_IMG, COLS3 = 0.12, 60, 4      # 점수 소프트맥스 온도 · 이미지 장수 · 열 수

S2 = sorted(OUT.glob('sample2_seed*_n*.json'))
SET2 = set(json.loads(S2[-1].read_text(encoding='utf-8'))) if S2 else set()
FR = {k: v for k, v in GT.items() if k in SET2 and v['points']} or \
     {k: v for k, v in GT.items() if v['points']}
DET = {p.name: v for p, v in detect([IMGDIR / n for n in FR], thr=0.12, topk=6).items()}

R = []
for n in FR:
    im = Image.open(IMGDIR / n).convert('RGB')
    hsv = np.asarray(im.convert('HSV'), np.float32); W, H = im.size
    P = [(p['x'] * W, p['y'] * H, p['t']) for p in FR[n]['points']]
    for b in DET.get(n, []):
        ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
        x1, y1 = min(W, int(b[2]) + 1), min(H, int(b[3]) + 1)
        if x1 <= x0 or y1 <= y0: continue
        q = hsv[y0:y1, x0:x1].reshape(-1, 3)
        w = (q[:, 1] >= SMIN).astype(np.float32)
        if w.sum() < 20: continue
        h = np.histogram(q[:, 0], bins=NB, range=(0, 256), weights=w)[0] / w.sum()
        mg = {}
        for t, (k, th) in RULES.items():
            i, j = WINS[k]; mg[t] = float(h[i:j + 1].sum() - th)
        ks = list(mg); z = np.array([mg[t] for t in ks]) / TEMP
        z = np.exp(z - z.max()); sc = dict(zip(ks, z / z.sum()))
        rank = sorted(sc, key=lambda t: -sc[t])
        R.append({'frame': n, 'box': (x0, y0, x1, y1), 'truth': Counter(ins).most_common(1)[0][0],
                  'rank': rank, 'sc': sc, 'tag': b[5]})
print(f'박스 {len(R)}개\n')

# ── 사소한 베이스라인 (top-1 · top-2) ─────────────────────────────────
cnt = Counter(x['truth'] for x in R); tot = len(R)
mc = cnt.most_common()
# 🔴 top-k **모든 k**에 베이스라인을 찍는다 — top-3를 '-'로 비워뒀더니 0.748이 좋아 보였는데
#    실제 베이스라인이 0.762라 **이하**였다. 빈칸이 곧 함정이다.
BK = {}
_c = 0
for _k, (_t, _n) in enumerate(mc, 1):
    _c += _n; BK[_k] = _c / tot
b1, b2 = BK[1], BK[2]
print(f'■ 사소한 베이스라인 — 흔한 유형을 k개 나열하기만 해도 나오는 값')
for _k in sorted(BK):
    if _k > 4: break
    print(f'   top-{_k}  {" + ".join(t for t, _ in mc[:_k]):<34s} → {BK[_k]:.3f}')
print('   ← 이 선을 넘어야 의미가 있다\n')

def acc(f): return float(np.mean([f(x) for x in R]))
def perm_p(f, n=500):
    obs = acc(f); v = [f(x) for x in R]; rg = random.Random(0)
    null = [float(np.mean(rg.sample(v, len(v)))) for _ in range(n)]   # 형태 유지용(참고)
    return obs, (sum(1 for z in null if z >= obs) + 1) / (n + 1)

ROWS = [
    ('색규칙 top-1',            lambda x: x['rank'][0] == x['truth'],                       b1),
    ('색규칙 top-2',            lambda x: x['truth'] in x['rank'][:2],                      b2),
    ('색규칙 top-3',            lambda x: x['truth'] in x['rank'][:3],                      'k3'),
    ('검출기 태그 (top-1)',      lambda x: x['tag'] == x['truth'],                           b1),
    ('색규칙1 + 검출기태그 (2개)', lambda x: x['truth'] in {x['rank'][0], x['tag']},           b2),
    ('색규칙2 + 검출기태그 (3개)', lambda x: x['truth'] in set(x['rank'][:2]) | {x['tag']},    'k3'),
]
print(f'{"방법":26s} {"정확도":>8s} {"베이스라인":>9s} {"대비":>8s}')
for nm, f, bl in ROWS:
    a = acc(f)
    if bl == 'k3': bl = BK.get(3)
    if bl is None: print(f'{nm:26s} {a:8.3f} {"-":>9s} {"-":>8s}')
    else:
        d = a - bl
        print(f'{nm:26s} {a:8.3f} {bl:9.3f} {d:+8.3f}' +
              ('   🔴 사소한 규칙 이하' if d <= 0 else ('   ⚠️ +0.10 미만' if d < 0.10 else '   ✅')))

# ── 캘리브레이션: 점수가 진짜 신뢰도인가 ──────────────────────────────
print(f'\n■ 캘리브레이션 — top-1 점수 구간별 **실제** 적중률')
print(f'{"점수 구간":12s} {"n":>5s} {"평균점수":>8s} {"실제적중":>8s} {"차이":>7s}')
EDGES = [0, .35, .45, .55, .70, .85, 1.01]
for lo, hi in zip(EDGES, EDGES[1:]):
    k = [x for x in R if lo <= x['sc'][x['rank'][0]] < hi]
    if len(k) < 5: continue
    ms = float(np.mean([x['sc'][x['rank'][0]] for x in k]))
    ma = float(np.mean([x['rank'][0] == x['truth'] for x in k]))
    print(f'{f"{lo:.2f}~{hi:.2f}":12s} {len(k):5d} {ms:8.3f} {ma:8.3f} {ma-ms:+7.3f}')
print('   → 평균점수 ≈ 실제적중이면 점수를 신뢰도로 쓸 수 있다(리포트 VLM이 "애매함"을 표현 가능).')
print('     크게 어긋나면 그 숫자를 화면에 적으면 안 된다 — 없는 확신을 만든다.')

# ── 유형별 top-1 vs top-2 ─────────────────────────────────────────────
print(f'\n{"유형":14s} {"n":>4s} {"top-1":>7s} {"top-2":>7s} {"이득":>7s}')
for t, n in cnt.most_common():
    k = [x for x in R if x['truth'] == t]
    a1 = float(np.mean([x['rank'][0] == t for x in k]))
    a2 = float(np.mean([t in x['rank'][:2] for x in k]))
    print(f'{t:14s} {n:4d} {a1:7.3f} {a2:7.3f} {a2-a1:+7.3f}')

# ── 이미지: 사용자가 요청한 표기 형식 ────────────────────────────────
_ID = re.compile(r'cylindrical_(\d+)_')
d0 = OUT / 'check_top2'; d0.mkdir(exist_ok=True)
for f in d0.glob('*.jpg'): f.unlink()
sel = R[:]; random.Random(0).shuffle(sel); sel = sel[:N_IMG]
cells = defaultdict(list)
for it in sel:
    m = _ID.search(it['frame']); cells[m.group(1) if m else 'unknown'].append(it)
CH = 84
for cidn, its in sorted(cells.items()):
    rows = -(-len(its) // COLS3)
    sh = Image.new('RGB', (COLS3 * TILE, rows * (TILE + CH) + 28), (255, 255, 255))
    dr = ImageDraw.Draw(sh)
    dr.text((6, 6), f'TOP-2  cell {cidn}  ·  {len(its)} boxes', fill=(0, 0, 0), font=F)
    for i, it in enumerate(its):
        im = Image.open(IMGDIR / it['frame']).convert('RGB')
        x0, y0, x1, y1 = it['box']; mgp = max(x1 - x0, y1 - y0) * 0.30
        c = im.crop((max(0, x0 - mgp), max(0, y0 - mgp),
                     min(im.width, x1 + mgp), min(im.height, y1 + mgp)))
        kk = TILE / max(c.size)
        c = c.resize((max(1, round(c.width * kk)), max(1, round(c.height * kk))), Image.LANCZOS)
        tl = Image.new('RGB', (TILE, TILE + CH), (255, 255, 255))
        tl.paste(c, ((TILE - c.width) // 2, (TILE - c.height) // 2))
        d2 = ImageDraw.Draw(tl)
        r1, r2 = it['rank'][0], it['rank'][1]
        hit = it['truth'] in (r1, r2)
        d2.rectangle([0, TILE, TILE - 1, TILE + CH - 1],
                     fill=(226, 245, 228) if hit else (253, 228, 228))
        d2.text((6, TILE + 2),  f"top1 {L(r1)} {it['sc'][r1]:.0%}", fill=(0, 0, 0), font=F)
        d2.text((6, TILE + 23), f"top2 {L(r2)} {it['sc'][r2]:.0%}", fill=(70, 70, 70), font=F)
        d2.text((6, TILE + 44), f"검출기 {L(it['tag'])}" if KO else f"DETECT {L(it['tag'])}",
                fill=(120, 120, 120), font=FB)
        d2.text((6, TILE + 63), f"사람 {L(it['truth'])}" if KO else f"HUMAN {L(it['truth'])}",
                fill=(20, 110, 40) if hit else (185, 30, 30), font=FB)
        sh.paste(tl, (i % COLS3 * TILE, 28 + i // COLS3 * (TILE + CH)))
    sh.save(d0 / f'cell_{cidn}.jpg', quality=92)
print(f'\n저장 {d0} · 파일 {len(cells)}개 · 박스 {len(sel)}개 (초록 = 정답이 top2 안에 있음)')
for cidn in sorted(cells)[:2]:
    print(f'── cell {cidn} ──'); display(Image.open(d0 / f'cell_{cidn}.jpg'))
print('\n🔴 top-2는 2차를 다 본 뒤에 정한 지표다. 주장하려면 3차 라벨이 필요하다 —')
print('   오늘 답할 것은 "쓸 만한가"까지다. 위 **베이스라인 대비**와 **캘리브레이션**으로 판단할 것.')

In [ ]:
# == §CO 크롭 → 원본(1920×1080) 좌표 매핑 복원 (약 3~5분) ==
# 🔴 우리 bbox는 **크롭 좌표계**(예: 1464×895)인데 분류기·백엔드는 **원본 1920×1080** 기준이다.
#    그대로 넘기면 박스가 통째로 어긋난 채 "정상 동작"으로 보인다 — 에러가 안 난다.
#
# 🔑 복원이 가능한 이유: 크롭이 **결정론적 함수**로 만들어졌다(`battery_ext_crop_retrain` §1).
#    `crop_box(원본)` → 정규화 (x1,y1,x2,y2) · `apply_crop`이 `int(x1*W), int(y1*H)`에서 자른다.
#    → 원본을 다시 열어 같은 함수를 돌리면 오프셋이 **정확히** 나온다.
#
# 🔑 그리고 자체 검증이 된다: 계산한 크롭 크기가 실제 평가셋 이미지 크기와 **정확히 같아야** 한다.
#    하나라도 어긋나면 크롭 버전이 다르다는 뜻이므로 **거기서 멈춘다.**
#
# 선행: 이 노트북 §C(GT · IMGDIR · OUT). 원본 v4.2 zip이 Drive에 있어야 한다.
import os, io, glob, json, zipfile, time
import numpy as np
from pathlib import Path
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

# ── 크롭 함수 원본 그대로 (retrain §1 복제 — 값 하나라도 바뀌면 오프셋이 틀어진다) ──
PAD, SAT_TH, DENS_TH = 0.10, 40, 0.10
MIN_AREA, MAX_AREA = 0.02, 0.90

def cell_bbox(im, small=400):
    t = im.copy(); t.thumbnail((small, small))
    hsv = np.asarray(t.convert('HSV'))
    S = hsv[:, :, 1].astype(np.int16); m = S > SAT_TH
    if m.mean() < 0.01 or m.mean() > 0.98:
        V = hsv[:, :, 2].astype(np.int16)
        m = np.abs(V - np.median(np.concatenate([V[0], V[-1]]))) > 40
    if m.sum() < 50: return None
    h, w = m.shape; rs, cs = m.sum(1), m.sum(0)
    if rs.max() == 0 or cs.max() == 0: return None
    ry = np.where(rs > rs.max() * DENS_TH)[0]; cx = np.where(cs > cs.max() * DENS_TH)[0]
    if len(ry) == 0 or len(cx) == 0: return None
    bb = (cx.min() / w, ry.min() / h, (cx.max() + 1) / w, (ry.max() + 1) / h)
    a = (bb[2] - bb[0]) * (bb[3] - bb[1])
    return None if not (MIN_AREA <= a <= MAX_AREA) else bb

def crop_box(im, pad=PAD):
    bb = cell_bbox(im)
    if bb is None: return (0.0, 0.0, 1.0, 1.0), False
    x1, y1, x2, y2 = bb; bw, bh = x2 - x1, y2 - y1
    return (max(0.0, x1 - bw * pad / 2), max(0.0, y1 - bh * pad / 2),
            min(1.0, x2 + bw * pad / 2), min(1.0, y2 + bh * pad / 2)), True

# ── 원본 zip 찾기 (크롭본 zip은 제외) ─────────────────────────────────
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass
CAND = []
for pat in ('/content/drive/MyDrive/**/*v4*2*.zip', '/content/drive/MyDrive/**/*v42*.zip',
            '/content/drive/**/*v4*2*.zip'):
    CAND += glob.glob(pat, recursive=True)
CAND = sorted({c for c in CAND if 'crop' not in Path(c).name.lower()}, key=os.path.getsize,
              reverse=True)
assert CAND, '★원본 v4.2 zip을 못 찾았다 — 경로를 직접 지정할 것'
print('원본 zip 후보:')
for c in CAND[:6]: print(f'   {os.path.getsize(c)/1e9:6.1f} GB  {c}')

NAMES = sorted(GT)
NEED = set(NAMES)
MEMBER = {}
t0 = time.time()
for z in CAND:
    if not NEED - set(MEMBER): break
    try: nl = zipfile.ZipFile(z).namelist()
    except Exception as e: print(f'   스킵 {Path(z).name}: {str(e)[:50]}'); continue
    hit = 0
    for n in nl:
        b = Path(n).name
        if b in NEED and b not in MEMBER: MEMBER[b] = (z, n); hit += 1
    print(f'   {Path(z).name}: {hit}장 매칭 (누적 {len(MEMBER)}/{len(NEED)})')
print(f'인덱싱 {time.time()-t0:.0f}초\n')
_miss = sorted(NEED - set(MEMBER))
assert not _miss, (f'★원본을 못 찾은 평가셋 이미지 {len(_miss)}장 (예: {_miss[:3]}) — '
                   f'원본 zip 경로를 확인할 것')

# ── 오프셋 복원 + 자체 검증 ───────────────────────────────────────────
_ZF = {}
def zread(z, n):
    if z not in _ZF: _ZF[z] = zipfile.ZipFile(z)
    return _ZF[z].read(n)

OFF, bad, fallback = {}, [], 0
t0 = time.time()
for i, name in enumerate(NAMES, 1):
    z, mem = MEMBER[name]
    orig = Image.open(io.BytesIO(zread(z, mem))).convert('RGB')
    W, H = orig.size
    box, ok = crop_box(orig)
    if not ok: fallback += 1
    x0, y0 = int(box[0] * W), int(box[1] * H)
    x1, y1 = int(box[2] * W), int(box[3] * H)
    cw, ch = x1 - x0, y1 - y0
    aw, ah = Image.open(IMGDIR / name).size          # ★실제 평가셋 이미지 크기
    if (cw, ch) != (aw, ah):
        bad.append((name, f'계산 {cw}x{ch} ≠ 실제 {aw}x{ah}'))
    OFF[name] = {'orig_size': [W, H], 'offset': [x0, y0], 'crop_size': [cw, ch],
                 'crop_box_norm': [round(v, 6) for v in box], 'estimated': bool(ok)}
    if i % 50 == 0: print(f'  {i}/{len(NAMES)} · {(time.time()-t0)/i:.2f}s/장', flush=True)

print(f'\n원본 크기 분포: {dict(__import__("collections").Counter(tuple(v["orig_size"]) for v in OFF.values()))}')
print(f'크롭 추정 실패(원본 전체 사용): {fallback}장')
print(f'크기 검증 불일치: {len(bad)}장 ' + ('✅' if not bad else '🔴'))
for n, w in bad[:10]: print(f'   {n[-46:]:46s} {w}')

assert not bad, ('★크롭 크기가 안 맞는다 = 이 오프셋으로 변환하면 박스가 어긋난다.\n'
                 '   원인 후보: 평가셋이 다른 버전 크롭본에서 나왔다 / crop_box 파라미터가 바뀌었다.\n'
                 '   **여기서 멈추고 원인을 찾을 것. 틀린 좌표를 백엔드에 넘기면 조용히 망가진다.**')

(OUT / 'crop_offsets.json').write_text(json.dumps(OFF, ensure_ascii=False, indent=1),
                                       encoding='utf-8')
print(f'\n✅ 전 {len(OFF)}장 크기 검증 통과 → 오프셋 신뢰 가능')
print(f'저장 {OUT/"crop_offsets.json"}')
_k = NAMES[0]
print(f'\n예시  {_k}')
print(f'   원본 {OFF[_k]["orig_size"]} · 오프셋 {OFF[_k]["offset"]} · 크롭 {OFF[_k]["crop_size"]}')
print(f'   변환식:  x_원본 = x_크롭 + {OFF[_k]["offset"][0]} · y_원본 = y_크롭 + {OFF[_k]["offset"][1]}')
print('\n▶ 이제 §Y를 다시 돌리면 bbox가 원본 1920×1080 기준으로 나간다.')

In [ ]:
# == §Y 리포트 LLM 넘김 규격 — 항목마다 **검증 상태를 같이** 넘긴다 (검출 1분) ==
# 🔴 리포트를 쓰는 것은 **LLM**이다(VLM 아님). 즉 **이미지를 못 본다.**
#    → 우리가 틀린 값을 넘기면 LLM은 그걸 **사실로 받아 적는다.** 스스로 눈치챌 방법이 없다.
#    그래서 값만 넘기면 안 되고 **그 값이 어디까지 검증됐는지**를 같이 넘겨야 한다.
#
# 검증 상태 (오늘까지의 실측 그대로)
#   판정  게이트 thr0.08·N≥8 → P0.873 / R0.995 / F1 0.930 (300장)  → 사실로 써도 된다
#   위치  점 recall 0.818 (순열 대조 2.51×)                      → 사실로 써도 된다
#   크기  VLM 크기 ↔ 실측 면적 rho +0.297 / p 0.001, 유효 2단계    → 쓰되 "큼/작음" 수준
#   유형  🔴 **최빈 베이스라인 0.376을 못 넘음**(색규칙 +0.081)     → **후보+점수, 단정 금지**
#
# 🔴🔴 좌표계: bbox는 **원본 1920x1080 기준**으로 내보낸다(분류기·백엔드가 그 구조로 돈다).
#    우리 검출은 크롭 이미지 위에서 도므로 §CO가 복원한 오프셋을 더해 원본 좌표로 바꾼다.
#    오프셋이 없으면 **여기서 멈춘다** — 크롭 좌표를 넘기면 박스가 통째로 어긋난 채
#    **에러 없이** 흘러간다. 가장 조용한 사고다.
#
# 선행: §M3(RULES·WINS·NB·SMIN) · unsup §1(detect) · §C(GT·IMGDIR·OUT) · **§CO(crop_offsets.json)**
import json, re
import numpy as np
from collections import Counter, defaultdict
from PIL import Image

assert 'RULES' in dir(), '★§M3를 먼저'
TEMP, THR_GATE, N_GATE, THR_LOC, CAP_ZONE = 0.12, 0.08, 8, 0.12, 0.22

def cell_box(im, tol=55, frac=0.12):
    a = np.asarray(im, np.float32); H, W = a.shape[:2]; e = max(1, W // 20)
    bg = np.concatenate([a[:, :e].reshape(-1, 3), a[:, -e:].reshape(-1, 3)])
    m = np.abs(a - np.median(bg, 0)).sum(2) > tol
    xs = np.where(m.mean(0) > frac)[0]; ys = np.where(m.mean(1) > frac)[0]
    return (0, 0, W, H) if len(xs) < 5 or len(ys) < 5 else \
           (int(xs[0]), int(ys[0]), int(xs[-1]) + 1, int(ys[-1]) + 1)

FR = sorted(GT)
DET = {p.name: v for p, v in detect([IMGDIR / n for n in FR], thr=0.02, topk=50).items()}
_ID = re.compile(r'cylindrical_(\d+)_')

# ── 우리가 넘기는 것의 계약서. LLM 프롬프트 맨 앞에 그대로 붙인다 ────────
CONTRACT = {
    "설명": "배터리 셀 외관 검사 결과. 각 필드에 신뢰 등급이 붙어 있다.",
    "지켜야 할 것": [
        "신뢰='높음'인 값만 단정해서 쓴다.",
        "신뢰='낮음'인 값은 반드시 '~로 보이나 확정되지 않음' 형태로 쓴다. 단정하면 안 된다.",
        "유형후보는 순서만 있고 **박스별 확률이 아니다**. 개별 후보에 퍼센트를 붙이지 않는다.",
        "확률을 언급해야 한다면 '필드 신뢰 등급.유형후보.실측_적중률'의 값만 쓴다. 그 값은 목록 전체에 대한 것이지 개별 후보에 대한 것이 아니다.",
        "유형후보가 2개 이상이면 모두 언급한다. 1등만 쓰고 나머지를 버리지 않는다.",
        "여기 없는 값을 지어내지 않는다. 특히 원인·조치는 우리가 판단하지 않았다.",
        "이미지를 보지 않고 쓰는 리포트임을 전제한다.",
    ],
    "필드 신뢰 등급": {
        "판정": {"신뢰": "높음", "근거": "P 0.873 / R 0.995 / F1 0.930 (사람 라벨 300장)"},
        "좌표계": {"신뢰": "높음",
                     "설명": "bbox는 **원본 이미지 1920x1080 기준**이다. 크롭 좌표가 아니다.",
                     "근거": "크롭 오프셋을 원본에서 재계산하고 크롭 크기 일치로 전수 검증(§CO)"},
        "위치": {"신뢰": "높음", "근거": "점 recall 0.818, 순열 대조 2.51배 (사람 라벨 300장)"},
        "크기": {"신뢰": "중간", "근거": "실측 면적과 순위상관 0.297 (p=0.001). 큼/작음 2단계만 유효"},
        "유형후보": {"신뢰": "낮음",
                     "근거": "최빈 유형 베이스라인을 못 넘음 — top-1 +0.081 · top-2 +0.038 · top-3 -0.014",
                     "주의": "유형은 참고용이다. 리포트에서 확정 표현을 쓰지 말 것",
                     "점수_없는_이유": "박스별 신뢰도 점수를 낼 수 있었으나 캘리브레이션이 어긋나(평균 0.954 vs 실제 적중 0.477, 전체의 84%가 그 구간) 내보내지 않는다. 후보 순서 외의 확신을 부여하지 말 것",
                     # 🔑 박스별 확률은 못 주지만 **목록 전체의 적중률은 실측값**이라 줄 수 있다.
                     #    전역 통계라 캘리브레이션 문제가 없다 — 측정한 그대로다(2차 210박스).
                     "실측_적중률": {
                         "후보목록_전체": "0.79",
                         "후보_1개만_봤을_때": "0.46",
                         "사소한_베이스라인": "0.76 (흔한 유형 3개를 그냥 나열해도 이만큼 맞는다)",
                         "해석": "목록 안에 실제 유형이 있을 확률이 약 79%지만, 아무 정보 없이 흔한 유형만 나열해도 76%다. 즉 이 후보 목록은 거의 정보가 없다. 확정 표현을 쓰지 말고 '가능성 있는 후보'로만 언급할 것",
                         "표본": "사람 라벨 210박스"}},
    },
}

# ── 크롭 → 원본 좌표 오프셋 (§CO 산출) ──────────────────────────────
_off_p = OUT / 'crop_offsets.json'
assert _off_p.exists(), '★crop_offsets.json이 없다 — **§CO를 먼저 실행**할 것 (좌표계가 틀어진다)'
OFFS = json.loads(_off_p.read_text(encoding='utf-8'))
_no = [n for n in FR if n not in OFFS]
assert not _no, f'★오프셋 없는 프레임 {len(_no)}장 (예: {_no[:2]}) — §CO를 다시 돌릴 것'
print(f'좌표 변환: 크롭 → 원본 {OFFS[FR[0]]["orig_size"]} (오프셋 {len(OFFS)}장 확보)')

OUTD = OUT / 'report_input'; OUTD.mkdir(exist_ok=True)
for f in OUTD.glob('*.json'): f.unlink()
(OUTD / '_contract.json').write_text(json.dumps(CONTRACT, ensure_ascii=False, indent=1),
                                     encoding='utf-8')

cells = defaultdict(list)
for n in FR:
    m = _ID.search(n); cells[m.group(1) if m else 'unknown'].append(n)

n_def = 0
for cid, frames in sorted(cells.items()):
    rec = {"셀": cid, "검사_프레임수": len(frames),
           "주의": "프레임은 같은 셀을 360도 회전하며 찍은 것이다. 같은 결함이 여러 프레임에 나올 수 있다.",
           "프레임": []}
    for n in frames:
        raw = DET.get(n, [])
        n_gate = sum(1 for b in raw if b[4] >= THR_GATE)
        flag = n_gate >= N_GATE
        fr = {"파일": n, "판정": "결함" if flag else "정상",
              "판정_신뢰": "높음",
              "판정_근거": {"임계": THR_GATE, "박스수": n_gate, "기준": f"{N_GATE}개 이상"},
              "결함": []}
        if flag:
            _o = OFFS[n]; OX, OY = _o['offset']; OW, OH = _o['orig_size']
            fr["원본_크기"] = [OW, OH]
            im = Image.open(IMGDIR / n).convert('RGB')
            hsv = np.asarray(im.convert('HSV'), np.float32)
            cb = cell_box(im); CH = max(cb[3] - cb[1], 1); CA = max((cb[2] - cb[0]) * CH, 1)
            for b in sorted([x for x in raw if x[4] >= THR_LOC], key=lambda x: -x[4])[:6]:
                x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
                x1, y1 = min(im.width, int(b[2]) + 1), min(im.height, int(b[3]) + 1)
                if x1 <= x0 or y1 <= y0: continue
                q = hsv[y0:y1, x0:x1].reshape(-1, 3)
                w = (q[:, 1] >= SMIN).astype(np.float32)
                cand = []
                if w.sum() >= 20:
                    h = np.histogram(q[:, 0], bins=NB, range=(0, 256), weights=w)[0] / w.sum()
                    mg = {}
                    for t, (k, th) in RULES.items():
                        i, j = WINS[k]; mg[t] = float(h[i:j + 1].sum() - th)
                    ks = list(mg); z = np.array([mg[t] for t in ks]) / TEMP
                    z = np.exp(z - z.max()); z = z / z.sum()
                    # 🔴 **점수를 넘기지 않는다.** 캘리브레이션 실측:
                    #    점수 0.85~1.01 구간에 176/210이 몰리는데 평균 0.954 vs 실제 적중 0.477
                    #    = **−0.477**. 거의 모든 박스에 "95% 확신"이라 적고 절반이 틀린다.
                    #    리포트를 쓰는 것은 **LLM**이라 그림을 못 본다 → 그 숫자를 그대로 확신으로 옮긴다.
                    #    → 순위만 넘긴다. 숫자는 우리가 못 믿는 것이므로 밖으로 내보내지 않는다.
                    cand = [t for t, _ in sorted(zip(ks, z), key=lambda p: -p[1])[:2]]
                if b[5] not in cand:
                    cand.append(b[5])                  # 검출기 태그도 후보로 병기(순위 없음)
                rel = ((y0 + y1) / 2 - cb[1]) / CH
                area = (x1 - x0) * (y1 - y0) / CA * 100
                fr["결함"].append({
                    # 🔴 bbox는 **원본 좌표**다. 크롭 좌표는 디버그용으로만 병기한다.
                    "위치": {"구역": "금속캡" if rel < CAP_ZONE else "몸통",
                             "셀_상단에서": round(float(min(max(rel, 0), 1)), 2),
                             "bbox": [x0 + OX, y0 + OY, x1 + OX, y1 + OY],
                             "bbox_좌표계": f"원본 {OW}x{OH}",
                             "bbox_크롭": [x0, y0, x1, y1],
                             "크롭_오프셋": [OX, OY]},
                    "위치_신뢰": "높음",
                    "크기": {"셀면적비_퍼센트": round(float(area), 2),
                             "구분": "큼" if area >= 1.0 else "작음"},
                    "크기_신뢰": "중간",
                    "유형후보": cand,          # 순위 있는 이름 목록. **점수는 일부러 안 넣는다**(위 주석)
                    "유형_신뢰": "낮음",
                    "유형_주의": "검증 미통과. 확정 표현 금지",
                })
            n_def += 1
        rec["프레임"].append(fr)
    (OUTD / f'cell_{cid}.json').write_text(json.dumps(rec, ensure_ascii=False, indent=1),
                                           encoding='utf-8')

print(f'저장 {OUTD}')
print(f'  _contract.json  ← LLM 프롬프트 맨 앞에 붙일 계약서')
print(f'  cell_*.json     ← 셀당 1개, 총 {len(cells)}개 (결함 프레임 {n_def}개)\n')
_ex = sorted(OUTD.glob('cell_*.json'))[0]
_d = json.loads(_ex.read_text(encoding='utf-8'))
_d['프레임'] = [f for f in _d['프레임'] if f['판정'] == '결함'][:1] or _d['프레임'][:1]
if _d['프레임'] and _d['프레임'][0].get('결함'):
    _d['프레임'][0]['결함'] = _d['프레임'][0]['결함'][:2]
print(f'── 예시 ({_ex.name}) ──')
print(json.dumps(_d, ensure_ascii=False, indent=1)[:1800])
print('\n▶ LLM에는 _contract.json + cell_XXXX.json 을 같이 준다.')
print('  계약서가 없으면 LLM은 유형후보 1등을 **확정으로 써버린다** — 그림을 못 보니 의심할 수가 없다.')

In [ ]:
# == §E2 v9 — VLM에 **셀 전체 + 확대 크롭**을 같이 보여준다 (약 13분) ==
# 🔑 사용자 지적: "VLM이 결함만 crop된 부분을 보고 유형을 정하나?
#    전체적으로 보고 결함만 crop해서 보여주는 게 맞는 것 같은데."  → 맞다. 지금은 크롭만 본다.
#
# v8이 넘기던 것: [기준 패치(같은 셀 다른 부위) | 확대 크롭]  ← **셀 전체를 한 번도 안 보여준다**
#   그래서 VLM이 모르는 것: 이 크롭이 캡인지 몸통인지 · 이 색이 이 셀에서 이상한지 원래 그런지 ·
#   결함이 셀에서 얼마나 큰 부분인지. 손톱만 한 조각만 보고 유형을 맞히라는 셈이었다.
#   (§G 실측: 2단 유형 정확도 0.157, 우연 0.120)
#
# v9가 넘기는 것: [셀 전체 + 빨간 박스로 위치 표시 | 그 부분 확대]  = overview + detail
#   왼쪽이 문맥과 기준을 동시에 준다(셀 전체가 곧 이 셀의 정상 표면) · 박스가 둘을 연결한다.
#   → 기준 패치는 뺀다. 셀 전체가 그 역할을 대신한다.
#
# 🔴 비용: 사람 점이 든 박스만 돈다(유형 정확도는 거기서만 정의된다) → 638 → 약 197개 = 13분.
# 🔴 v8과 **같은 프레임(1차)**에 돌린다 — 패널 구조만 바꾼 A/B여야 원인을 특정할 수 있다.
# 🔴 프롬프트 형식은 v8 그대로(번호·알파벳). 한 번에 하나만 바꾼다.
#
# 선행: unsup §1(detect) · unsup §4(_gen) · 이 노트북 §C(GT · IMGDIR · OUT)
import json, re, time
import numpy as np
from collections import Counter
from PIL import Image, ImageDraw
from IPython.display import display

assert callable(detect) and callable(_gen), '★unsup §1·§4를 먼저'
PROMPT_VER = 'v9'
DUMP = OUT / f'vlm_dump_{PROMPT_VER}.json'
THR_LOC, CAP_N = 0.12, 6
PANEL_H, CROP_PX, MARGIN = 420, 336, 0.35

S2 = sorted(OUT.glob('sample2_seed*_n*.json'))
SET2 = set(json.loads(S2[-1].read_text(encoding='utf-8'))) if S2 else set()
FR = sorted(k for k, v in GT.items() if v['points'] and k not in SET2)   # ★1차만 = v8과 동일
print(f'대상 1차 결함 프레임 {len(FR)}장 (v8과 같은 표본)')

KIND_ID = {0: '정상', 1: '녹·부식', 2: '벗겨짐·박리', 3: '파손·찢김',
           4: '긁힘·스크래치', 5: '들뜸', 6: '오염·이물질'}
P9 = ('왼쪽은 배터리 셀 **전체** 사진이고, 빨간 네모가 검사할 위치다.\n'
      '오른쪽은 그 부분을 확대한 것이다.\n'
      '셀 전체와 비교해서 그 부분에 무엇이 있는지 고르라. 설명하지 말고 아래 세 줄만 답하라.\n\n'
      '종류  0 없음  1 녹·부식  2 벗겨짐  3 찢김·파손  4 긁힘  5 들뜸  6 오염·이물질\n'
      '크기  A 사진 절반 이상   B 사진의 1/4쯤   C 손톱만 함   D 점 하나\n\n'
      '종류: (숫자)\n크기: (알파벳)\n본것: (한 문장)')

def fit(im, px, by_h=False):
    k = px / (im.height if by_h else max(im.size))
    return im.resize((max(1, round(im.width * k)), max(1, round(im.height * k))), Image.LANCZOS)

def panel(im, b):
    """왼쪽 = 셀 전체(박스 표시) · 오른쪽 = 확대 크롭"""
    full = im.copy(); d = ImageDraw.Draw(full)
    d.rectangle(list(b), outline=(255, 0, 0), width=max(3, im.width // 60))
    full = fit(full, PANEL_H, by_h=True)
    w, h = b[2] - b[0], b[3] - b[1]; m = max(w, h) * MARGIN
    crop = fit(im.crop((max(0, b[0] - m), max(0, b[1] - m),
                        min(im.width, b[2] + m), min(im.height, b[3] + m))), CROP_PX)
    W = full.width + crop.width + 14; H = max(full.height, crop.height)
    p = Image.new('RGB', (W, H), (255, 255, 255))
    p.paste(full, (0, (H - full.height) // 2))
    p.paste(crop, (full.width + 14, (H - crop.height) // 2))
    return p

def parse9(r):
    t = str(r.get('셀요약') or json.dumps(r, ensure_ascii=False)) if isinstance(r, dict) else str(r)
    k = re.search(r'종류\D{0,4}([0-6])', t); z = re.search(r'크기\W{0,4}([A-Da-d])', t)
    o = re.search(r'본것\s*[:：]\s*([^\n]+)', t)
    return (int(k.group(1)) if k else 0, (z.group(1).upper() if z else 'D'),
            (o.group(1).strip() if o else t.strip())[:140])

DET = {p.name: v for p, v in detect([IMGDIR / n for n in FR], thr=THR_LOC, topk=CAP_N).items()}
dump = json.loads(DUMP.read_text(encoding='utf-8')) if DUMP.exists() else {}
JOBS = []
for n in FR:
    im = Image.open(IMGDIR / n).convert('RGB'); W, H = im.size
    P = [(p['x'] * W, p['y'] * H, p['t']) for p in GT[n]['points']]
    for b in DET.get(n, []):
        ins = [t for x, y, t in P if b[0] <= x <= b[2] and b[1] <= y <= b[3]]
        if not ins: continue
        key = f'{n}|{int(b[0])},{int(b[1])},{int(b[2])},{int(b[3])}'
        if key in dump: continue
        JOBS.append((key, n, tuple(b[:4]), Counter(ins).most_common(1)[0][0], b[5]))
print(f'사람 점을 담은 박스 {len(JOBS) + len(dump)}개 · 남은 {len(JOBS)}개\n')
if JOBS:
    display(panel(Image.open(IMGDIR / JOBS[0][1]).convert('RGB'), JOBS[0][2]))
    print('↑ VLM이 보게 될 화면 (왼쪽 셀 전체 + 빨간 박스, 오른쪽 확대)\n')

t0 = time.time()
for i, (key, n, b, truth, tag) in enumerate(JOBS, 1):
    im = Image.open(IMGDIR / n).convert('RGB')
    kid, sz, obs = parse9(_gen(panel(im, b), P9, 120))
    dump[key] = {'frame': n, 'bbox': [round(v, 1) for v in b], 'truth': truth,
                 'tag': tag, 'kind_id': kid, 'kind': KIND_ID[kid], 'size': sz, 'obs': obs}
    if i % 10 == 0 or i == len(JOBS):
        DUMP.write_text(json.dumps(dump, ensure_ascii=False), encoding='utf-8')
        el = time.time() - t0
        print(f'  {i}/{len(JOBS)} · {el/i:.1f}s/개 · 남은 {(len(JOBS)-i)*el/i/60:.0f}분', flush=True)
DUMP.write_text(json.dumps(dump, ensure_ascii=False), encoding='utf-8')

# ── 채점: v9 vs v8 vs 검출기, **사소한 베이스라인** 대비 ────────────────
V = list(dump.values())
cnt = Counter(x['truth'] for x in V); base = cnt.most_common(1)[0][1] / len(V)
a9 = float(np.mean([x['kind'] == x['truth'] for x in V]))
at = float(np.mean([x['tag'] == x['truth'] for x in V]))
print(f'\n■ 유형 정확도 (박스 {len(V)}개)')
print(f'   사소한 베이스라인 (무조건 {cnt.most_common(1)[0][0]})  {base:.3f}   ← 넘어야 할 선')
print(f'   v9  셀 전체 + 크롭                        {a9:.3f}  ({a9-base:+.3f})')
print(f'   v8  기준 패치 + 크롭 (§G 실측)             0.157  (-{base-0.157:.3f})')
print(f'   검출기 태그                               {at:.3f}  ({at-base:+.3f})')
print(f'\n{"유형":14s} {"n":>4s} {"v9":>7s} {"검출기":>7s}')
for t, n in cnt.most_common():
    k = [x for x in V if x['truth'] == t]
    print(f'{t:14s} {n:4d} {np.mean([x["kind"]==t for x in k]):7.3f} '
          f'{np.mean([x["tag"]==t for x in k]):7.3f}')
_ob = [x['obs'] for x in V]
print(f'\n관찰 문장 서로 다른 것 {len(set(_ob))}/{len(_ob)} = {len(set(_ob))/max(len(_ob),1):.0%}'
      + ('  🔴 복사 중' if len(set(_ob)) < len(_ob) * 0.3 else ''))
print(f'종류 분포: {Counter(x["kind"] for x in V).most_common()}')
print('\n▶ 판정: v9가 **베이스라인 + 0.10**을 넘으면 패널 구조가 원인이었던 것이다.')
print('   못 넘으면 문맥을 줘도 안 된다 = 7B VLM의 유형 판별 한계. 유형 미해결 확정.')

In [ ]:
# == §Z 넘김 JSON 검사 — LLM에 잘못된 확신이 새어 나가지 않는지 (즉시) ==
# 🔴 리포트를 쓰는 것은 **LLM**이라 이미지를 못 본다. 우리가 넘긴 값을 그대로 사실로 옮긴다.
#    그래서 **실제 저장된 파일**을 열어 다음을 검사한다(코드가 아니라 산출물을 본다):
#      ① 유형후보에 **숫자(확률)**가 섞여 있는가        ← 캘리브레이션 붕괴로 빼기로 한 것
#      ② 계약서의 신뢰 근거가 **현재 실측치**와 맞는가
#      ③ 스키마가 스펙(EXT_최종스펙.md §5.5)과 맞는가
#      ④ 신뢰 '낮음'인 필드에 주의 문구가 빠진 곳이 없는가
# 선행: §Y 실행(= report_input/ 생성)
import json
from collections import Counter

RI = OUT / 'report_input'
assert RI.is_dir(), '★report_input이 없다 — §Y를 먼저'
CT = RI / '_contract.json'
CELLS = sorted(RI.glob('cell_*.json'))
print(f'검사 대상: 계약서 1 + 셀 {len(CELLS)}개\n')

bad = []

# ── ① 유형후보에 숫자가 섞였나 ────────────────────────────────────────
n_item = n_num = 0
lens = Counter()
for f in CELLS:
    d = json.loads(f.read_text(encoding='utf-8'))
    for fr in d.get('프레임', []):
        for it in fr.get('결함', []):
            n_item += 1
            c = it.get('유형후보')
            lens[len(c) if isinstance(c, list) else -1] += 1
            if not isinstance(c, list):
                bad.append((f.name, '유형후보가 리스트가 아니다')); continue
            for x in c:
                if not isinstance(x, str):
                    n_num += 1
                    bad.append((f.name, f'유형후보에 숫자/구조가 섞였다: {x!r}'))
print(f'① 유형후보  박스 {n_item}개 · 후보 개수 분포 {dict(sorted(lens.items()))}')
print(f'   숫자 섞인 항목 {n_num}개 ' + ('✅ 없음' if n_num == 0 else '🔴 있다 — §Y를 다시 돌릴 것'))

# ── ② 계약서 신뢰 근거가 현재 실측치와 맞나 ──────────────────────────
NOW = {'판정': ['0.873', '0.995', '0.930', '300장'],
       '좌표계': ['원본'],
       '위치': ['0.818', '2.51'],
       '크기': ['0.297'],
       '유형후보': ['베이스라인']}
ct = json.loads(CT.read_text(encoding='utf-8')) if CT.exists() else {}
grades = ct.get('필드 신뢰 등급', {})
print(f'\n② 계약서 신뢰 등급 {len(grades)}개')
for k, need in NOW.items():
    g = grades.get(k)
    if not g:
        bad.append(('_contract.json', f'{k} 항목 없음')); print(f'   {k:8s} 🔴 없음'); continue
    txt = json.dumps(g, ensure_ascii=False)
    miss = [x for x in need if x not in txt]
    print(f'   {k:8s} 신뢰={g.get("신뢰")}' + ('  ✅' if not miss else f'  🔴 옛 수치 — 빠진 것 {miss}'))
    if miss: bad.append(('_contract.json', f'{k} 근거가 현재 실측과 다르다 (빠짐: {miss})'))

# ── ③ 스키마 ─────────────────────────────────────────────────────────
NEED_FR = {'파일', '판정', '판정_신뢰', '판정_근거', '결함'}
NEED_DF = {'위치', '위치_신뢰', '크기', '크기_신뢰', '유형후보', '유형_신뢰', '유형_주의'}
miss_fr = miss_df = 0
for f in CELLS:
    d = json.loads(f.read_text(encoding='utf-8'))
    for fr in d.get('프레임', []):
        if not NEED_FR <= set(fr): miss_fr += 1; bad.append((f.name, f'프레임 키 누락 {NEED_FR-set(fr)}'))
        for it in fr.get('결함', []):
            if not NEED_DF <= set(it): miss_df += 1; bad.append((f.name, f'결함 키 누락 {NEED_DF-set(it)}'))
print(f'\n③ 스키마  프레임 키 누락 {miss_fr} · 결함 키 누락 {miss_df} '
      + ('✅' if miss_fr + miss_df == 0 else '🔴'))

# ── ④ 신뢰 '낮음'인데 주의 문구가 없는 곳 ────────────────────────────
no_warn = 0
for f in CELLS:
    d = json.loads(f.read_text(encoding='utf-8'))
    for fr in d.get('프레임', []):
        for it in fr.get('결함', []):
            if it.get('유형_신뢰') == '낮음' and not it.get('유형_주의'):
                no_warn += 1; bad.append((f.name, '유형_신뢰=낮음인데 주의 문구 없음'))
print(f'④ 낮음인데 주의 문구 없는 항목 {no_warn} ' + ('✅' if no_warn == 0 else '🔴'))

# ── ⑤ 좌표계: bbox가 원본 기준인가 ───────────────────────────────────
# 🔴 크롭 좌표를 넘기면 박스가 통째로 어긋난 채 **에러 없이** 흘러간다. 가장 조용한 사고다.
n_bb = n_out = n_nocs = 0
for f in CELLS:
    d = json.loads(f.read_text(encoding='utf-8'))
    for fr in d.get('프레임', []):
        OW, OH = fr.get('원본_크기', [None, None])
        for it in fr.get('결함', []):
            loc = it.get('위치', {}); bb = loc.get('bbox')
            n_bb += 1
            if not loc.get('bbox_좌표계'):
                n_nocs += 1; bad.append((f.name, 'bbox 좌표계 표기 없음 — §CO 후 §Y 재실행 필요'))
                continue
            if OW and bb and (bb[2] > OW or bb[3] > OH or min(bb) < 0):
                n_out += 1; bad.append((f.name, f'bbox가 원본 밖: {bb} vs {OW}x{OH}'))
            ck, off = loc.get('bbox_크롭'), loc.get('크롭_오프셋')
            if ck and off and [ck[0] + off[0], ck[1] + off[1]] != bb[:2]:
                bad.append((f.name, f'bbox ≠ 크롭+오프셋 {ck}+{off} vs {bb}'))
print(f'\n⑤ 좌표계  bbox {n_bb}개 · 좌표계 표기 없음 {n_nocs} · 원본 범위 밖 {n_out} '
      + ('✅' if n_nocs + n_out == 0 else '🔴'))

# ── 요약 + 실제 예시 ─────────────────────────────────────────────────
print('\n' + '=' * 62)
if bad:
    print(f'🔴 문제 {len(bad)}건')
    for f, w in bad[:15]: print(f'   {f:22s} {w}')
    if len(bad) > 15: print(f'   ... 외 {len(bad)-15}건')
else:
    print('✅ 넘김 JSON 이상 없음 — 확률 숫자 유출 없음, 계약서 최신, 스키마 일치')

_ex = next((it for f in CELLS
            for fr in json.loads(f.read_text(encoding='utf-8')).get('프레임', [])
            for it in fr.get('결함', [])), None)
if _ex:
    print('\n── 실제 결함 항목 하나 ──')
    print(json.dumps(_ex, ensure_ascii=False, indent=1))
if grades.get('유형후보', {}).get('실측_적중률'):
    print('\n── LLM이 쓸 수 있는 문장 예시 ──')
    r = grades['유형후보']['실측_적중률']
    print(f'  "{_ex["유형후보"][0]} 또는 {_ex["유형후보"][1] if len(_ex["유형후보"])>1 else "-"}로 보이나 확정되지 않음"')
    print(f'  (후보 목록 안에 실제 유형이 포함될 확률 {r.get("후보목록_전체")} — '
          f'흔한 유형을 그냥 나열해도 {r.get("사소한_베이스라인")}이므로 참고용)')